# DA3METRIC-LARGE — **Self-Supervised** Fine-Tuning Ablation Study on KITTI Eigen-Split

**Companion to the fully-supervised ablation** (`DA3_supervised_FT_Ablation_HPT_v6.ipynb`). This notebook studies *self-supervised* and *hybrid* fine-tuning where the bulk of the learning signal comes from **geometric / photometric consistency between consecutive frames** rather than from dense LiDAR labels.

---

## Why a separate self-supervised study?

For a metric depth foundation model already trained on huge amounts of data (DA3METRIC-LARGE), the supervised fine-tuning question is "which loss best uses my sparse LiDAR?". The self-supervised question is fundamentally different: **"which geometric prior best uses my video stream when labels are absent, noisy, or scarce?"**. The two regimes have different failure modes, different hyper-parameter sensitivities, and different best-in-class losses — so we treat them as independent ablations and compare their winners at the end.

The hierarchy of self-supervised signals for monocular metric depth on driving data is well-established (see Monodepth2, PackNet-SfM, ManyDepth, FeatDepth, VideoDA, UniDepthV2):

| Tier | Signal | Geometric correctness | Metric-scale capacity | Cost |
|------|--------|----------------------|----------------------|------|
| **1** | Pose + differentiable warping (photometric reprojection) | Excellent | Best (with vel. or sparse anchor) | Moderate |
| **2** | Optical-flow-consistency / flow–depth coupling | Moderate | Weak / medium | Higher |
| **3** | Temporal depth consistency via *warped* depth | Good (proper) | Medium | Low |
| **3a (heuristic)** | Naive "mean depth-delta" temporal regulariser | **Wrong (globally constant Δd assumption)** | None | Low |
| **4 (anchor)** | Sparse LiDAR SILog as scale anchor | Direct | Excellent | Low |

> A central question of this notebook: does adding self-supervised signal to a foundation MDE model **help, hurt, or do nothing** when only sparse LiDAR is available? And which signal type yields the best transfer to the Eigen test split?

---

## What this notebook does

| Step | Description |
|------|-------------|
| 1 | Implements a **6-DoF pose network** (ResNet-18 backbone, axis-angle + translation) and **differentiable inverse-warping** in PyTorch |
| 2 | Defines **13 self-supervised / hybrid loss variants** spanning 2014 → 2025 SOTA papers |
| 3 | Defines **3 architecture variants** (decoder-only / last-2-blocks / full fine-tune) |
| 4 | Defines **4 augmentation variants** (none / flip / base / paired-safe full) |
| 5 | Defines **5 optimizer/scheduler variants** (AdamW × cosine_warmup / Adam / SGD / OneCycle / StepLR) |
| 6 | **NEW Group 5 — Loss-combination ablation**: 7 cumulative recipes from "photo-only" up to the document's full recommended pipeline |
| 7 | **NEW Group 6 — Loss-weight hyper-parameter sweep**: 8 different $(\lambda_\text{photo}, \lambda_\text{smooth}, \lambda_\text{temporal}, \lambda_\text{anchor})$ settings |
| 8 | Runs each configuration with **HPT** (learning-rate sweep) + **early stopping** + strict TEST-set isolation |
| 9 | Evaluates on KITTI Eigen-Split test set with **all 4 scale-alignment strategies** (unscaled / median / mean / affine) |
| 10 | Reports pixel-weighted true RMSE, MAE, AbsRel, SqRel, RMSE_log, log10, δ<1.25, Bias — identical schema to the supervised study so the two CSVs are directly comparable |
| 11 | Produces **15 analysis plots** + per-config epoch curves + trajectory-quality tables + winner declarations |
| 12 | Emits an elaborate written analysis after every individual test group |

---

## Self-supervised losses covered (13 — full coverage 2015 → 2025)

| ID | Name | Paper / Origin | Year |
|----|------|----------------|------|
| `photo_l1` | Pure L1 photometric reprojection | Garg et al., ECCV 2016 | 2016 |
| `photo_ssim` | SSIM-only photometric | Wang et al. (SSIM), 2004 / adapted in MDE | adapted |
| `photo_ssim_l1` | α·SSIM + (1-α)·L1 photometric (Monodepth) | Godard et al., CVPR 2017 | 2017 |
| `edge_aware_smooth` | Edge-aware disparity smoothness only | Heise et al. 2013 / Godard 2017 | 2017 |
| `min_reproj_photo` | **Min-reprojection + auto-mask** (Monodepth2) | Godard et al., ICCV 2019 ★ | 2019 |
| `multi_scale_photo` | 4-scale photometric reprojection | Monodepth2 multi-scale | 2019 |
| `geometric_consistency` | Forward-backward depth consistency (SC-SfMLearner) | Bian et al., NeurIPS 2019 | 2019 |
| `feature_metric` | Feature-metric photometric loss (FeatDepth) | Shu et al., ECCV 2020 | 2020 |
| `flow_depth_consistency` | Flow ↔ depth+pose coupling (GeoNet style) | Yin & Shi, CVPR 2018 | 2018 |
| `temporal_warp_consistency` | **Proper** temporal depth consistency via *warped depth* | Bian 2019 + this work | 2019/this |
| `naive_temporal_delta` | **Heuristic** "constant Δd" temporal regulariser | User's prior attempt (critiqued) | this |
| `combined_monodepth2` | Full Monodepth2 recipe (min_photo + smooth + auto-mask + multi-scale) | Godard 2019 | 2019 |
| `combined_recommended` | **Document's recipe**: photo + smooth + temporal_warp + edge_smooth + sparse LiDAR anchor | This work | 2025 |

Two **anchor baselines** are included for context:
- `lidar_anchor_only` (pure supervised SILog, no SSL signal) → upper-bound for what GT alone can do at the current data scale.
- The naive `naive_temporal_delta` is included **specifically to demonstrate** that the geometrically incorrect "mean-Δd" assumption underperforms the geometrically correct `temporal_warp_consistency`.

---

## Architecture / Optimizer / Augmentation / Combination / Weight ablations also included

All four supervised-study axes are replicated (decoder-only / last-2-blocks / full FT × 4 aug modes × 5 optim/sched × 4 LR candidates) plus two new self-supervision-specific axes:

- **Group 5 (loss combination)**: 7 cumulative recipes — what does each added term buy?
- **Group 6 (loss weights)**: 8 weight settings around the recommended recipe — how sensitive is the result to the mixing coefficients?

---

## Total HPT trial budget

| Group | Configs | LR candidates | HPT trials |
|------|---------|---------------|------------|
| 1. Loss function | 13 | 4 | 39 |
| 2. Architecture | 3 | 4 | 9 |
| 3. Augmentation | 4 | 4 | 12 |
| 4. Optim/Sched | 5 | 4 | 15 |
| 5. Loss combination | 7 | 4 | 21 |
| 6. Loss weights | 8 | 4 | 24 |
| **TOTAL** | **40** | — | **120** |

Each trial = up to `ABLATION_EPOCHS` (default 5) of training + held-out validation; the best-val checkpoint is then evaluated on the TEST split **exactly once** per ablation label.

---

> Run order: ① set paths in the *Configuration* cell, ② run cells top-to-bottom. The notebook is **resumable**: any ablation already present in `SUMMARY_CSV` is skipped. Per-trial state is checkpointed for crash recovery.


## 0 · Imports


In [ ]:
import os
# Set the visible device to index 1 (GPU 1)
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
from __future__ import annotations
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import copy
import csv
import glob
import json
import logging
import math
import shutil
import sys
import time
import warnings
from itertools import combinations
from dataclasses import dataclass, field
from scipy.optimize import linear_sum_assignment
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.models as tvm
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, Subset
import torch.backends.cudnn as cudnn
cudnn.benchmark = True

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("ssl_ablation")
logging.getLogger("transformers").setLevel(logging.ERROR)

print("Imports OK  |  CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"  device: {torch.cuda.get_device_name(0)}")


Imports OK  |  CUDA: True
  device: Tesla V100-PCIE-32GB


## 1 · Configuration


In [ ]:
# ─── PATHS ───────────────────────────────────────────────────────────────────
KITTI_ROOT      = "/home/arka/Downloads/MMDE/datasets/KITTI/tiny"
TRAIN_ROOT      = os.path.join(KITTI_ROOT, "train")
TEST_ROOT       = os.path.join(KITTI_ROOT, "test")
CACHE_DIR       = "/home/arka/Downloads/MMDE/cache"

OUTPUT_BASE     = "/home/arka/Downloads/MMDE/outputs/self_supervised_FT_output/abalation_study"
CHECKPOINT_DIR  = os.path.join(OUTPUT_BASE, "checkpoints")
RESULTS_DIR     = os.path.join(OUTPUT_BASE, "results")
PLOT_DIR        = os.path.join(OUTPUT_BASE, "plots")
ANALYSIS_DIR    = os.path.join(OUTPUT_BASE, "analysis")
SUMMARY_CSV     = os.path.join(RESULTS_DIR, "ablation_summary.csv")
TEMP_CKPT_DIR   = os.path.join(OUTPUT_BASE, "temp_ckpts")
for d in [OUTPUT_BASE, CHECKPOINT_DIR, RESULTS_DIR, PLOT_DIR, ANALYSIS_DIR, TEMP_CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# ─── MODEL ────────────────────────────────────────────────────────────────────
MODEL_ID        = "depth-anything/DA3METRIC-LARGE"
ORTHO_NORM      = 300.0
FOCAL_LENGTH_X  = 721.5377         # KITTI default: avg(fx, fy) ≈ 721.5 px
USE_MODEL_FOCAL = False            # DA3METRIC-LARGE doesn't have intrinsic prediction

# ─── TRAINING BUDGET ──────────────────────────────────────────────────────────
ABLATION_EPOCHS = 10                # epochs per HPT trial (use 10-20 for full study)
BATCH_SIZE      = 8
ACCUM_STEPS     = 4                # effective batch = 16
LEARNING_RATE   = 1e-6              # default LR
HPT_LR_CANDIDATES  = [1e-4, 1e-5, 1e-6, 1e-7]
WEIGHT_DECAY    = 1e-4
WARMUP_EPOCHS   = 1
MAX_GRAD_NORM   = 1.0
MIXED_PREC      = True
NUM_WORKERS     = 8
MAX_TRAIN       = 50                # 0 = all; set e.g. 500 for fast sweep
MAX_TEST        = 0
gpu_duty_cycle = 1.0
"""
    gpu_duty_cycle: fraction of wall-time the GPU is actively computing.
                    0.5 means equal compute and sleep → ~50% utilisation.
                    Set to 0.2 for 20% utilisation.
                    Set to 0.8 for 80% utilisation.
                    Set to 1.0 to disable throttling entirely.
"""

# ─── DATASET ──────────────────────────────────────────────────────────────────
TARGET_H        = 378              # multiple of 14
TARGET_W        = 1246             # multiple of 14
MIN_DEPTH       = 1e-3
MAX_DEPTH       = 80.0
SSL_MIN_DEPTH   = 1e-3
SSL_MAX_DEPTH   = 80.0

# ─── ABLATION DIMENSIONS TO RUN ──────────────────────────────────────────────
RUN_LOSS_ABLATION         = True
RUN_ARCH_ABLATION         = True
RUN_AUG_ABLATION          = True
RUN_OPTIM_ABLATION        = True
RUN_COMBINATION_ABLATION  = True   # NEW Group 5 — cumulative recipes
RUN_WEIGHT_ABLATION       = True   # NEW Group 6 — weight sweep

# ─── SSL-SPECIFIC HYPERPARAMETERS ────────────────────────────────────────────
EPS                             = 1e-6
SSIM_ALPHA                      = 0.85
SMOOTHNESS_DECAY                = 1.0
MIN_REPROJ_AUTO_MASK_THRESHOLD  = 0.0       # 0 → strict auto-mask (Monodepth2 default)
TEMPORAL_WARP_THRESHOLD         = 0.05      # m — static-pixel threshold
POSE_NET_FREEZE_EPOCHS          = 0         # epochs to keep pose net frozen at init

# Default loss weights for combined_recommended
DEFAULT_W_PHOTO    = 1.0
DEFAULT_W_SMOOTH   = 1e-3
DEFAULT_W_TEMPORAL = 0.1
DEFAULT_W_ANCHOR   = 0.5
DEFAULT_W_FEAT     = 0.01
DEFAULT_W_GEOM     = 0.1

# ─── HPT & EARLY-STOPPING ────────────────────────────────────────────────────
EARLY_STOP_PATIENCE  = 8
VAL_SPLIT_FRAC       = 0.10     # from 0.05
VAL_SPLIT_SEED       = 42
SAVE_EVERY_N_EPOCHS  = 1        # save a periodic checkpoint every N epochs (0 = disabled)


LOSS_HPT_GRID  = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
ARCH_HPT_GRID  = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
AUG_HPT_GRID   = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
OPTIM_HPT_GRID = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
COMBO_HPT_GRID = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
WEIGHT_HPT_GRID = [{"lr": lr} for lr in HPT_LR_CANDIDATES]

print("Configuration loaded.")
print(f"  patience={EARLY_STOP_PATIENCE}  val_frac={VAL_SPLIT_FRAC}  "
      f"lr_candidates={HPT_LR_CANDIDATES}")
print(f"  ssim_alpha={SSIM_ALPHA}  smooth_weight={DEFAULT_W_SMOOTH}  "
      f"temporal_weight={DEFAULT_W_TEMPORAL}  anchor_weight={DEFAULT_W_ANCHOR}")
# ─── TRACKING & DYNAMIC-OBJECT FILTER (Part 3 / 4 / 7) ────────────────────────
ENABLE_OXTS            = True            # gate for GT-pose dataset/losses (Part 2b, already implemented)
ENABLE_TRACKING        = True            # gate for Part 3 mask tracking
ENABLE_DYNAMIC_FILTER  = True            # gate for Part 4 stationary-object filter
ENABLE_GT_POSE         = True            # use OXTS gt_pose when available, else PoseNet
ENABLE_CHAMFER         = False           # Part 5 Loss 4 (wired in the next build stage)
ENABLE_UNIFORMITY      = True            # Part 5 Loss 2 (wired in the next build stage)

TRACKER_TYPE           = "hungarian"     # "iou" | "hungarian" | "bytetrack"
SAM3_MASK_DIR          = os.path.join(KITTI_ROOT, "sam3_masks")
DYNAMIC_FILTER_MODE    = "both"          # "class" | "flow" | "both"
FLOW_RESIDUAL_PX       = 2.0             # px, residual object flow after ego-flow subtraction
TRACK_IOU_MATCH_THRESH = 0.15            # Tracker 1 minimum combined score to accept a match
TRACK_MAX_AGE          = 5               # frames a track survives with no matched detection
TRACK_MIN_HITS         = 1               # matched frames required before a track is "confirmed"
BYTETRACK_HIGH_THRESH  = 0.5             # SAM3 score split point: high vs low confidence detections
BYTETRACK_LOW_THRESH   = 0.1             # detections below this score are discarded entirely

print(f"  output -> {OUTPUT_BASE}")


Configuration loaded.
  patience=5  val_frac=0.05  lr_candidates=[0.0001, 1e-05, 1e-06]
  ssim_alpha=0.85  smooth_weight=0.001  temporal_weight=0.1  anchor_weight=0.5
  output -> /STORAGE/Arka/a_MDE_Code/4_Self_supervised_FT/ssl_ft_ablation_outputs


## 2 · Dataset (sequential frame pairs)


In [4]:
_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]
_train_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
])


def _read_kitti_depth(path: str) -> np.ndarray:
    raw = np.array(Image.open(path), dtype=np.uint16)
    return raw.astype(np.float32) / 256.0


def _parse_kitti_calib(date_dir: Path, drive_dir: Path) -> Optional[np.ndarray]:
    calib: Dict[str, np.ndarray] = {}
    for fname in ["calib_cam_to_cam.txt"]:
        for search in [date_dir, drive_dir]:
            p = search / fname
            if p.exists():
                for line in p.read_text().splitlines():
                    if ":" in line:
                        k, v = line.split(":", 1)
                        try:
                            calib[k.strip()] = np.array(v.split(), dtype=np.float64)
                        except Exception:
                            pass
    K = None
    try:
        if "P_rect_02" in calib:
            P = calib["P_rect_02"].reshape(3, 4); K = P[:3, :3].copy()
        elif "P2" in calib:
            P = calib["P2"].reshape(3, 4); K = P[:3, :3].copy()
    except Exception:
        pass
    return K


# ─── Augmentations (paired-safe variants) ───────────────────────────────────

def _aug_flip(rgb, depth, rng):
    if rng.rand() < 0.5:
        rgb   = np.fliplr(rgb).copy()
        depth = np.fliplr(depth).copy()
    return rgb, depth

def _aug_base(rgb, depth, rng):
    rgb, depth = _aug_flip(rgb, depth, rng)
    f  = rgb.astype(np.float32) / 255.0
    br = rng.uniform(0.8, 1.2)
    co = rng.uniform(0.8, 1.2)
    f  = np.clip((f - f.mean()) * co + f.mean() * br, 0.0, 1.0)
    return (f * 255.0).astype(np.uint8), depth

def _aug_full(rgb, depth, rng):
    # 'Full' for SSL = flip + colour jitter only. Random crops break the
    # principal-point assumption used by the warping operator, so they are
    # deliberately excluded.
    return _aug_base(rgb, depth, rng)

AUG_FNS = {"none": None, "flip": _aug_flip, "base": _aug_base, "full": _aug_full}


# ─── Sequential frame-pair dataset (required for ALL SSL losses) ────────────

class KITTISequentialDepthDataset(Dataset):
    # Emits (frame_i, frame_{i+1}) pairs from the same KITTI drive.
    # Identical augmentation seed across the pair preserves pixel-correspondence
    # needed for photometric / temporal warping losses.
    
    def __init__(self, root, split="train",
                 target_hw=(TARGET_H, TARGET_W),
                 min_depth=MIN_DEPTH, max_depth=MAX_DEPTH,
                 aug_mode="base", max_samples=0, seed=42):
        self.root      = Path(root)
        self.split     = split
        self.target_hw = target_hw
        self.min_depth = min_depth
        self.max_depth = max_depth
        self.aug_fn    = AUG_FNS.get(aug_mode)
        self.aug_mode  = aug_mode
        self.augment   = (split == "train" and self.aug_fn is not None)
        self.rng       = np.random.RandomState(seed)
        self._pairs: List[Tuple[Dict, Dict]] = []
        self._build_pairs()
        if max_samples > 0 and len(self._pairs) > max_samples:
            idx = self.rng.choice(len(self._pairs), max_samples, replace=False)
            self._pairs = [self._pairs[i] for i in sorted(idx)]
        log.info("[KITTI-Seq-%s] %d pairs (aug=%s)",
                 split, len(self._pairs), aug_mode)

    def _build_drive_index(self) -> Dict:
        depth_glob = str(self.root / "*" / "*_sync" /
                         "proj_depth" / "groundtruth" / "image_02" / "*.png")
        depth_paths = sorted(glob.glob(depth_glob))
        if not depth_paths:
            depth_glob = str(self.root / "*" / "*" /
                             "proj_depth" / "groundtruth" / "image_02" / "*.png")
            depth_paths = sorted(glob.glob(depth_glob))

        by_drive: Dict = {}
        for dp in depth_paths:
            parts = Path(dp).parts
            try:
                gi        = list(parts).index("proj_depth")
                drive_dir = Path(*parts[:gi])
                date_dir  = drive_dir.parent
                fname     = parts[-1]
            except (ValueError, IndexError):
                continue
            rgb_path = drive_dir / "image_02" / "data" / fname
            if not rgb_path.exists():
                continue
            K = _parse_kitti_calib(date_dir, drive_dir)
            key = (date_dir.name, drive_dir.name)
            rec = dict(rgb_path=str(rgb_path), depth_path=dp, K=K,
                       frame_id=int(Path(fname).stem))
            by_drive.setdefault(key, []).append(rec)

        for key in by_drive:
            by_drive[key].sort(key=lambda r: r["frame_id"])
        return by_drive

    def _build_pairs(self):
        for frames in self._build_drive_index().values():
            for i in range(len(frames) - 1):
                # Only keep truly consecutive frames (frame_id differs by 1).
                if frames[i + 1]["frame_id"] - frames[i]["frame_id"] == 1:
                    self._pairs.append((frames[i], frames[i + 1]))

    def _load_frame(self, rec: Dict) -> Dict:
        H, W    = self.target_hw
        rgb     = cv2.cvtColor(cv2.imread(rec["rgb_path"]), cv2.COLOR_BGR2RGB)
        depth   = _read_kitti_depth(rec["depth_path"])
        orig_hw = rgb.shape[:2]
        valid_o = (depth > self.min_depth) & (depth < self.max_depth)

        rgb   = cv2.resize(rgb,   (W, H), interpolation=cv2.INTER_LINEAR)
        depth = cv2.resize(depth, (W, H), interpolation=cv2.INTER_NEAREST)
        valid = cv2.resize(valid_o.astype(np.uint8), (W, H),
                           interpolation=cv2.INTER_NEAREST).astype(bool)
        depth = np.clip(depth, 0.0, self.max_depth)

        # Eigen crop (rows 44 : 44+352)
        eigen_crop = np.zeros((H, W), dtype=bool)
        eigen_crop[44:min(44 + 352, H), :] = True
        valid = valid & eigen_crop

        K = rec["K"]
        if K is not None:
            K = K.copy()
            K[0] *= W / orig_hw[1]
            K[1] *= H / orig_hw[0]
        return dict(rgb_np=rgb, depth=depth, valid=valid, K=K,
                    rgb_path=rec["rgb_path"], depth_path=rec["depth_path"],
                    frame_id=rec["frame_id"])

    def _to_tensors(self, frame: Dict) -> Dict:
        rgb_t   = _train_transform(Image.fromarray(frame["rgb_np"]))
        depth_t = torch.from_numpy(frame["depth"]).unsqueeze(0).float()
        valid_t = torch.from_numpy(frame["valid"].astype(np.float32)).unsqueeze(0)
        K       = frame["K"]
        _DEFAULT_K = np.array([
        [721.5377,   0.0,   609.5593],
        [  0.0,   721.5377, 172.8540],
        [  0.0,     0.0,     1.0  ],
        ], dtype=np.float32)
        K_t = torch.from_numpy(K).float() if K is not None else torch.from_numpy(_DEFAULT_K)
        # K_t     = torch.from_numpy(K).float() if K is not None else torch.zeros(3, 3)
        has_K   = torch.tensor(K is not None, dtype=torch.bool)
        # Raw uint8 RGB scaled to [0,1] — for SSIM / feature-metric loss
        rgb_raw = torch.from_numpy(frame["rgb_np"]).permute(2, 0, 1).float() / 255.0
        return {
            "rgb": rgb_t, "rgb_raw": rgb_raw,
            "depth_gt": depth_t, "valid_mask": valid_t,
            "K": K_t, "has_K": has_K,
            "rgb_path": frame["rgb_path"], "depth_path": frame["depth_path"],
        }

    def __len__(self):
        return len(self._pairs)

    def __getitem__(self, idx):
        rec_curr, rec_next = self._pairs[idx]
        curr = self._load_frame(rec_curr)
        nxt  = self._load_frame(rec_next)

        # Apply IDENTICAL augmentation to both frames (preserves pixel motion).
        worker_info = torch.utils.data.get_worker_info()
        if self.augment and self.aug_fn is not None:
            base_seed = (worker_info.seed if worker_info is not None
                         else self.rng.randint(0, 2**31))
            aug_seed = (base_seed + idx) % (2**31)
            for frame in [curr, nxt]:
                rng_local = np.random.RandomState(aug_seed)
                frame["rgb_np"], frame["depth"] = self.aug_fn(
                    frame["rgb_np"], frame["depth"], rng_local)
                frame["valid"] = ((frame["depth"] > self.min_depth) &
                                  (frame["depth"] < self.max_depth) &
                                  frame["valid"])

        curr_t = self._to_tensors(curr)
        nxt_t  = self._to_tensors(nxt)
        result = dict(curr_t)
        result.update({f"next_{k}": v for k, v in nxt_t.items()})
        return result


def collate_fn(batch):
    from torch.utils.data.dataloader import default_collate
    tensor_keys = [k for k in batch[0] if isinstance(batch[0][k], torch.Tensor)]
    str_keys    = [k for k in batch[0] if isinstance(batch[0][k], str)]
    out = {k: default_collate([b[k] for b in batch]) for k in tensor_keys}
    out.update({k: [b[k] for b in batch] for k in str_keys})
    return out


print("Dataset helpers ready (sequential frame pairs).")
print("  KITTISequentialDepthDataset -> consecutive (frame_i, frame_{i+1}) pairs")
print("  All SSL losses operate on these pairs.")


Dataset helpers ready (sequential frame pairs).
  KITTISequentialDepthDataset -> consecutive (frame_i, frame_{i+1}) pairs
  All SSL losses operate on these pairs.


## 2b · GPS / IMU Ground-Truth Pose from OXTS

KITTI records raw GPS/IMU data from an **Oxford Technical Solutions RT3003** unit at 100 Hz, down-sampled to match the camera frame rate. Each `oxts/data/NNNNNNNNNN.txt` file contains 30 space-separated fields per the `dataformat.txt` spec:

| Fields | Meaning |
|--------|---------|
| `lat, lon, alt` | WGS-84 position — converted to a flat-earth Mercator XY frame |
| `roll, pitch, yaw` | RPY orientation in radians — converted to SO(3) via Z-Y-X Euler angles |
| `vn, ve, vf, vl, vu` | Velocities (not used here) |
| `ax…wz…` | Accelerations + angular rates (not used here) |
| `navstat, numsats, …` | GPS status fields |

### Why use GT pose instead of PoseNet?

The PoseNet predicts relative pose from **appearance** alone, which is noisy in textureless / overexposed regions. The OXTS provides **direct metric measurements** of translation and rotation between consecutive frames with centimetre / 0.02° accuracy. When this ground-truth relative pose is fed into the differentiable warping operator, the photometric and temporal-warp losses are computing the *correct* reprojection error — rather than an error corrupted by pose estimation noise.

This lets us measure:
- How much of the SSL loss signal is lost to pose-estimation error (predicted vs GT-pose photometric)
- Whether the temporal-warp consistency benefit is principally from the temporal term or the pose accuracy
- Whether GT-pose flow-depth coupling adds cleaner supervision than flow estimated from appearance

### Calibration chain

OXTS data is in the **IMU/vehicle body frame**. To warp correctly in the camera frame we apply:

```
T_cam02_from_imu = T_02_offset @ R_rect_00 @ T_velo→cam0 @ T_imu→velo
```

sourced from `calib_imu_to_velo.txt`, `calib_velo_to_cam.txt`, `calib_cam_to_cam.txt` at the date directory level. If calibration files are missing, poses are returned in the IMU body frame (a reasonable approximation for a nearly-level, forward-moving vehicle).


In [5]:
import math

# =============================================================================
# OXTS / GPS-IMU utilities
# =============================================================================
# Reference: KITTI raw data devkit (A. Geiger, P. Lenz, C. Stiller, R. Urtasun)
# Each oxts/data/NNNNNNNNNN.txt has 30 space-separated floats in the order
# defined by dataformat.txt (lat, lon, alt, roll, pitch, yaw, ...).
# =============================================================================

_EARTH_RADIUS_M = 6378137.0    # WGS-84 semi-major axis


def _latlon_to_mercator(lat_deg: float, lon_deg: float, scale: float) -> Tuple[float, float]:
    """Flat-earth Mercator projection: (lat, lon) in degrees -> (mx, my) in metres."""
    mx = scale * math.radians(lon_deg) * _EARTH_RADIUS_M
    my = scale * _EARTH_RADIUS_M * math.log(
        math.tan(math.pi / 4.0 + math.radians(lat_deg) / 2.0))
    return mx, my


def _rpy_to_rotmat_np(roll: float, pitch: float, yaw: float) -> np.ndarray:
    """Z-Y-X Euler angles -> rotation matrix R = Rz(yaw) @ Ry(pitch) @ Rx(roll)."""
    cr, sr = math.cos(roll),  math.sin(roll)
    cp, sp = math.cos(pitch), math.sin(pitch)
    cy, sy = math.cos(yaw),   math.sin(yaw)
    return np.array([
        [ cy*cp,  cy*sp*sr - sy*cr,  cy*sp*cr + sy*sr],
        [ sy*cp,  sy*sp*sr + cy*cr,  sy*sp*cr - cy*sr],
        [-sp,     cp*sr,             cp*cr            ],
    ], dtype=np.float64)


def _parse_oxts_file(path: str) -> Optional[List[float]]:
    """Parse one OXTS data file -> list of 30 floats, or None on error."""
    try:
        vals = [float(v) for v in Path(path).read_text().strip().split()]
        return vals if len(vals) >= 6 else None
    except Exception:
        return None


def _oxts_packet_to_world_se3(pkt: List[float], scale: float) -> np.ndarray:
    """Convert one OXTS packet to a 4x4 SE(3) in flat-earth Mercator world frame."""
    lat, lon, alt, roll, pitch, yaw = pkt[:6]
    mx, my = _latlon_to_mercator(lat, lon, scale)
    R = _rpy_to_rotmat_np(roll, pitch, yaw)
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = R
    T[:3,  3] = [mx, my, alt]
    return T


def _load_imu_to_cam02(date_dir: Path) -> Optional[np.ndarray]:
    """
    Compose T_cam02_from_imu from KITTI calibration files found at <date_dir>/:
      calib_imu_to_velo.txt  -> Tr_imu_to_velo  [4x4]
      calib_velo_to_cam.txt  -> Tr_velo_to_cam0 [4x4]
      calib_cam_to_cam.txt   -> R_rect_00 + P_rect_02

    Full chain:
      T_cam02_from_imu = T_02_offset @ R_rect_00 @ Tr_velo_to_cam0 @ Tr_imu_to_velo

    Returns None if any file is absent or cannot be parsed.
    """
    def _kv(path: Path) -> Dict[str, np.ndarray]:
        d: Dict[str, np.ndarray] = {}
        for line in path.read_text().splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                try:
                    d[k.strip()] = np.array(v.split(), dtype=np.float64)
                except Exception:
                    pass
        return d

    try:
        f_iv = date_dir / "calib_imu_to_velo.txt"
        f_vc = date_dir / "calib_velo_to_cam.txt"
        f_cc = date_dir / "calib_cam_to_cam.txt"
        if not all(f.exists() for f in [f_iv, f_vc, f_cc]):
            return None

        kv_iv = _kv(f_iv); kv_vc = _kv(f_vc); kv_cc = _kv(f_cc)

        # Tr_imu->velo [4x4]
        Tr_iv = np.eye(4, dtype=np.float64)
        Tr_iv[:3, :3] = kv_iv["R"].reshape(3, 3)
        Tr_iv[:3,  3] = kv_iv["T"].ravel()

        # Tr_velo->cam0 [4x4]
        Tr_vc = np.eye(4, dtype=np.float64)
        Tr_vc[:3, :3] = kv_vc["R"].reshape(3, 3)
        Tr_vc[:3,  3] = kv_vc["T"].ravel()

        # R_rect_00 (rectification of cam0)
        R_rect = np.eye(4, dtype=np.float64)
        R_rect[:3, :3] = kv_cc["R_rect_00"].reshape(3, 3)

        # T offset from cam0 to cam02 (from P_rect_02 baseline)
        P02 = kv_cc["P_rect_02"].reshape(3, 4)
        tx_offset = P02[0, 3] / (P02[0, 0] + 1e-12)
        T_02_offset = np.eye(4, dtype=np.float64)
        T_02_offset[0, 3] = tx_offset

        return T_02_offset @ R_rect @ Tr_vc @ Tr_iv

    except Exception as e:
        log.debug("[OXTS calib] %s: %s", date_dir, e)
        return None


def _load_oxts_indexed(
        oxts_data_dir: Path,
        T_imu_to_cam: Optional[np.ndarray] = None,
) -> Dict[int, np.ndarray]:
    """
    Load all OXTS frames from <drive>/oxts/data/ and return a dict mapping
    frame_id (int) -> 4x4 SE(3) pose in local camera frame (or IMU frame if
    calibration is unavailable).

    Poses are local: first frame in this dict = identity.
    Relative pose from frame i to j:
        T_rel = inv(poses[i]) @ poses[j]
    """
    txt_files = sorted(glob.glob(str(oxts_data_dir / "*.txt")))
    if not txt_files:
        return {}

    frame_ids, packets = [], []
    for fpath in txt_files:
        pkt = _parse_oxts_file(fpath)
        if pkt is not None:
            frame_ids.append(int(Path(fpath).stem))
            packets.append(pkt)
    if not packets:
        return {}

    # Mercator scale from first packet's latitude
    scale = math.cos(math.radians(packets[0][0]))

    # World-frame poses
    world_poses = [_oxts_packet_to_world_se3(p, scale) for p in packets]

    # Make local: all relative to first frame
    T0_inv     = np.linalg.inv(world_poses[0])
    local_poses = [T0_inv @ T for T in world_poses]

    # Optionally transform from IMU frame to cam02 frame
    if T_imu_to_cam is not None:
        T_imu_from_cam = np.linalg.inv(T_imu_to_cam)
        # Absolute pose in cam frame:  T_cam_i = T_imu_to_cam @ T_local_i
        # Relative:                    T_rel   = inv(T_cam_i) @ T_cam_j
        #                                      = T_imu_from_cam @ inv(T_local_i)
        #                                        @ T_local_j @ T_imu_to_cam
        # Store absolute poses in cam frame so caller can compute T_rel easily.
        cam_poses = [T_imu_to_cam @ T for T in local_poses]
    else:
        cam_poses = local_poses

    return {fid: pose for fid, pose in zip(frame_ids, cam_poses)}


# =============================================================================
# Dataset subclass: KITTISequentialOXTSDataset
# =============================================================================

class KITTISequentialOXTSDataset(KITTISequentialDepthDataset):
    """
    Extends KITTISequentialDepthDataset by loading GPS/IMU-derived relative pose
    from the KITTI OXTS unit for each consecutive frame pair.

    Each batch item gains two extra keys:
        gt_pose     : torch.FloatTensor [4, 4]   relative SE(3) from frame_t to frame_{t+1}
                      in the camera frame (or IMU frame if calib unavailable).
        has_gt_pose : torch.BoolTensor  [1]       True iff OXTS data was found for this pair.

    All existing keys (rgb, depth_gt, valid_mask, K, ...) are preserved unchanged.

    When has_gt_pose is False the gt_pose tensor is the 4x4 identity matrix.
    GT-pose loss functions check has_gt_pose and fall back to the predicted PoseNet
    pose when GT is unavailable, so the dataset is backward-compatible with regular
    training runs that lack OXTS files.
    """

    def __init__(self, *args, oxts_subdir: str = "oxts", **kwargs):
        self._oxts_subdir = oxts_subdir
        self._drive_poses: Dict[Tuple, Dict[int, np.ndarray]] = {}
        # super().__init__ calls _build_pairs() via polymorphism, so
        # _drive_poses must be initialised first.
        super().__init__(*args, **kwargs)
        loaded = sum(len(v) for v in self._drive_poses.values())
        log.info("[OXTSDataset] %d drives, %d GT poses loaded",
                 len(self._drive_poses), loaded)

    # ── Override: _build_drive_index ─────────────────────────────────────────
    def _build_drive_index(self) -> Dict:
        by_drive = super()._build_drive_index()

        for (date, drive_name), frames in by_drive.items():
            if not frames:
                continue
            # Derive <drive_dir> and <date_dir> from the first frame's rgb_path
            rgb_0 = Path(frames[0]["rgb_path"])
            try:
                parts = list(rgb_0.parts)
                img02_idx = parts.index("image_02")
                drive_dir = Path(*parts[:img02_idx])
                date_dir  = drive_dir.parent
            except (ValueError, IndexError):
                continue

            oxts_data_dir = drive_dir / self._oxts_subdir / "data"
            if not oxts_data_dir.exists():
                log.debug("[OXTS] data dir not found: %s", oxts_data_dir)
                continue

            T_imu_to_cam = _load_imu_to_cam02(date_dir)
            calib_status = "calibrated" if T_imu_to_cam is not None else "IMU-frame (approx)"
            try:
                poses = _load_oxts_indexed(oxts_data_dir, T_imu_to_cam)
                self._drive_poses[(date, drive_name)] = poses
                log.info("[OXTS] %s/%s: %d poses (%s)",
                         date, drive_name, len(poses), calib_status)
            except Exception as e:
                log.warning("[OXTS] Failed %s/%s: %s", date, drive_name, e)

        return by_drive

    # ── Override: _build_pairs ────────────────────────────────────────────────
    def _build_pairs(self):
        """
        Identical logic to parent except each rec dict gets a '_gt_pose_np' key
        holding the numpy [4,4] relative SE(3) for that pair (or None).
        """
        for (date, drive_name), frames in self._build_drive_index().items():
            drive_poses = self._drive_poses.get((date, drive_name), {})
            for i in range(len(frames) - 1):
                if frames[i + 1]["frame_id"] - frames[i]["frame_id"] != 1:
                    continue
                fid_i = frames[i]["frame_id"]
                fid_j = frames[i + 1]["frame_id"]
                if fid_i in drive_poses and fid_j in drive_poses:
                    T_rel = np.linalg.inv(drive_poses[fid_i]) @ drive_poses[fid_j]
                else:
                    T_rel = None
                rc = dict(frames[i]);     rc["_gt_pose_np"] = T_rel
                rn = dict(frames[i + 1]); rn["_gt_pose_np"] = T_rel
                self._pairs.append((rc, rn))

    # ── Override: __getitem__ ─────────────────────────────────────────────────
    def __getitem__(self, idx: int) -> Dict:
        result = super().__getitem__(idx)
        rec_curr, _ = self._pairs[idx]
        gt_pose_np = rec_curr.get("_gt_pose_np")
        if gt_pose_np is not None:
            result["gt_pose"]     = torch.from_numpy(
                gt_pose_np.astype(np.float32))          # [4, 4]
            result["has_gt_pose"] = torch.tensor(True,  dtype=torch.bool)
        else:
            result["gt_pose"]     = torch.eye(4, dtype=torch.float32)
            result["has_gt_pose"] = torch.tensor(False, dtype=torch.bool)
        return result


def make_oxts_loaders(
        train_root: str,
        tr_idx:     List[int],
        val_idx:    List[int],
        aug_mode:   str = "base",
) -> Tuple[DataLoader, DataLoader]:
    """
    Convenience factory: build train + val DataLoaders backed by
    KITTISequentialOXTSDataset (same split indices as the base study).
    """
    ds_train = KITTISequentialOXTSDataset(
        train_root, split="train", aug_mode=aug_mode,
        max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED)
    ds_val   = KITTISequentialOXTSDataset(
        train_root, split="val",   aug_mode="none",
        max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED)
    loader_train = DataLoader(
        Subset(ds_train, tr_idx), batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
        persistent_workers=True, prefetch_factor=2, collate_fn=collate_fn,
        timeout=120)
    loader_val   = DataLoader(
        Subset(ds_val,   val_idx), batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True, prefetch_factor=2, collate_fn=collate_fn,
        timeout=120)
    return loader_train, loader_val


print("OXTS utilities ready.")
print("  _load_imu_to_cam02          -> calibration chain T_cam02_from_imu")
print("  _load_oxts_indexed          -> {frame_id -> SE(3)} per drive")
print("  KITTISequentialOXTSDataset  -> adds gt_pose + has_gt_pose to every batch")
print("  make_oxts_loaders           -> convenience factory for train/val loaders")


OXTS utilities ready.
  _load_imu_to_cam02          -> calibration chain T_cam02_from_imu
  _load_oxts_indexed          -> {frame_id -> SE(3)} per drive
  KITTISequentialOXTSDataset  -> adds gt_pose + has_gt_pose to every batch
  make_oxts_loaders           -> convenience factory for train/val loaders


## 2c · SAM3 Instance Masks & Multi-Object Tracking (Part 3)

### Purpose

The new stationary-object losses (Part 5) need to know **which pixels belong to the same physical object across consecutive frames**. SAM3 gives per-frame instance masks with no correspondence between frames — frame $t$'s "car #3" and frame $t{+}1$'s "car #3" are unrelated indices. This section adds three tracking algorithms of increasing sophistication that solve that correspondence problem, so downstream losses can say "this mask in frame $t$ and that mask in frame $t{+}1$ are the same physical object."

### Mathematical formulation shared by all three trackers

Each tracker solves a **bipartite assignment** between the set of tracks alive from frame $t{-}1$, $\mathcal{T} = \{\tau_1, ..., \tau_N\}$, and the set of new detections in frame $t$, $\mathcal{D} = \{d_1, ..., d_M\}$. Mask IoU is the primary geometric cue:

$$\text{IoU}(\tau_i, d_j) = \frac{|\text{mask}(\tau_i) \cap \text{mask}(d_j)|}{|\text{mask}(\tau_i) \cup \text{mask}(d_j)|}$$

Trackers 1–2 differ only in **how** the assignment is solved (greedy vs. optimal); Tracker 3 additionally changes **which detections are allowed to spawn new tracks**.

### Implementation details

- A cheap bbox-overlap pre-check (`_bbox_iou`) gates the expensive full-resolution `logical_and`/`logical_or` mask IoU computation, since KITTI masks can be ~375×1242 and pairwise IoU across all track×detection pairs would otherwise dominate the frame budget.
- An optional appearance descriptor (`_color_histogram_embedding`, a 48-D HSV histogram inside the mask) is used by Trackers 2–3 when available — SAM3 does not expose a learned Re-ID embedding by default, so this is the practical substitute referenced in the spec ("optional appearance embedding").
- Tracker 3 additionally carries a lightweight constant-velocity **Kalman filter** per track (`_ConstantVelocityKF`) over the 2-D centroid, so the assignment cost is computed against a *predicted* position rather than the last-observed one — this is what lets ByteTrack-style tracking survive a frame or two of partial occlusion.

### Computational complexity

- Tracker 1 (greedy): $O(NM)$ cost construction + $O(NM \log NM)$ sort.
- Tracker 2 (Hungarian): $O(NM)$ cost construction + $O(\min(N,M)^2 \max(N,M))$ Kuhn–Munkres solve (`scipy.optimize.linear_sum_assignment`).
- Tracker 3 (ByteTrack-style): two Hungarian solves per frame (high-score then low-score detections), same asymptotic cost as Tracker 2.

### Limitations

All three are online, appearance-light, nearest-neighbour trackers — none maintains a long-term Re-ID gallery, so identity can still be lost across occlusions longer than `max_age` frames. For the intended use (short-baseline consecutive-frame KITTI pairs, not long video), this is an acceptable trade-off against the cost of a full Re-ID network.

### References
- Bewley et al., *"Simple Online and Realtime Tracking" (SORT)*, ICIP 2016.
- Wojke et al., *"Simple Online and Realtime Tracking with a Deep Association Metric" (DeepSORT)*, ICIP 2017.
- Zhang et al., *"ByteTrack: Multi-Object Tracking by Associating Every Detection Box"*, ECCV 2022.


In [ ]:
# =============================================================================
# SAM3 mask loading + Detection/Track data structures (Part 3)
# =============================================================================

@dataclass
class MaskDetection:
    """One SAM3 instance mask, resolved to the geometric quantities every
    tracker needs (bbox / centroid / area are cached at construction time so
    trackers never recompute them from the raw mask on every comparison)."""
    mask: np.ndarray                              # bool [H, W]
    bbox: Tuple[float, float, float, float]        # (x1, y1, x2, y2)
    centroid: Tuple[float, float]                  # (cx, cy) in pixel coords
    area: float                                    # mask pixel count
    class_name: str = "unknown"
    score: float = 1.0
    embedding: Optional[np.ndarray] = None         # optional appearance descriptor


@dataclass
class Track:
    """A tracked object, persisted across frames by all three trackers below."""
    track_id: int
    mask: np.ndarray
    bbox: Tuple[float, float, float, float]
    centroid: Tuple[float, float]
    area: float
    class_name: str
    score: float
    embedding: Optional[np.ndarray] = None
    age: int = 0                                   # frames since track creation
    hits: int = 1                                  # number of matched detections total
    time_since_update: int = 0                     # frames since last matched detection
    history: List[Tuple[float, float]] = field(default_factory=list)


def _bbox_iou(b1: Tuple[float, float, float, float],
              b2: Tuple[float, float, float, float]) -> float:
    """Cheap axis-aligned bbox IoU, used as a quick reject before the
    expensive full-resolution mask IoU below."""
    x1, y1, x2, y2 = b1
    X1, Y1, X2, Y2 = b2
    ix1, iy1 = max(x1, X1), max(y1, Y1)
    ix2, iy2 = min(x2, X2), min(y2, Y2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    a1 = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a2 = max(0.0, X2 - X1) * max(0.0, Y2 - Y1)
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0


def mask_iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Exact pixel-wise IoU between two boolean masks."""
    inter = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return float(inter) / float(union) if union > 0 else 0.0


def _color_histogram_embedding(rgb_uint8: np.ndarray, mask: np.ndarray,
                                bins: int = 16) -> np.ndarray:
    """Cheap appearance descriptor: normalized per-channel HSV histogram
    inside the mask. Serves as the 'optional appearance embedding' term in
    the Hungarian / ByteTrack cost functions when no learned Re-ID network
    is available (SAM3 does not expose one by default)."""
    hsv = cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2HSV)
    pix = hsv[mask]
    if pix.shape[0] == 0:
        return np.zeros(bins * 3, dtype=np.float32)
    hist = [np.histogram(pix[:, c], bins=bins, range=(0, 256))[0].astype(np.float32)
            for c in range(3)]
    v = np.concatenate(hist)
    return v / (v.sum() + 1e-6)


def _mask_to_detection(mask: np.ndarray, class_name: str = "unknown",
                        score: float = 1.0,
                        rgb: Optional[np.ndarray] = None) -> Optional[MaskDetection]:
    """Builds a MaskDetection from a raw boolean mask. Returns None for an
    empty mask (can happen after resizing/cropping a SAM3 export)."""
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    x1, x2 = float(xs.min()), float(xs.max())
    y1, y2 = float(ys.min()), float(ys.max())
    cx, cy = float(xs.mean()), float(ys.mean())
    area = float(mask.sum())
    emb = _color_histogram_embedding(rgb, mask) if rgb is not None else None
    return MaskDetection(mask=mask, bbox=(x1, y1, x2, y2), centroid=(cx, cy),
                         area=area, class_name=class_name, score=float(score),
                         embedding=emb)


def load_sam3_masks_for_frame(rgb_path: str,
                              rgb_np: Optional[np.ndarray] = None) -> List[MaskDetection]:
    """
    Adapter: loads pre-generated SAM3 instance masks for one KITTI frame.

    Expected on-disk layout (EDIT THIS ONE FUNCTION if your SAM3 export
    layout differs — nothing else in the tracking/filtering/loss code below
    depends on the file format, only on the MaskDetection objects returned):

        SAM3_MASK_DIR/<date>/<drive>/image_02/data/<frame_id>.npz

    holding three arrays:
        masks   : uint8/bool [N, H, W]  — N instance masks at native image res
        classes : [N] strings or ints   — class label per instance
        scores  : [N] floats            — SAM3 confidence per instance

    Returns [] if the file is missing (wrong path, frame not yet segmented,
    ENABLE_TRACKING=False, etc.) so downstream tracking/filtering degrade
    gracefully rather than raising — the stationary-object losses in Part 5
    simply skip a pair with zero tracked objects instead of crashing training.
    """
    rgb_p = Path(rgb_path)
    try:
        parts = list(rgb_p.parts)
        img02_idx = parts.index("image_02")
        rel = Path(*parts[img02_idx:])          # image_02/data/xxxxxxxxxx.png
        drive_dir = Path(*parts[:img02_idx])
        date_dir = drive_dir.parent
        npz_path = (Path(SAM3_MASK_DIR) / date_dir.name / drive_dir.name /
                    rel.parent / f"{rgb_p.stem}.npz")
    except (ValueError, IndexError):
        return []

    if not npz_path.exists():
        return []

    try:
        data = np.load(npz_path, allow_pickle=True)
        masks = data["masks"]
        classes = data["classes"] if "classes" in data else np.array(["unknown"] * len(masks))
        scores = data["scores"] if "scores" in data else np.ones(len(masks), dtype=np.float32)
    except Exception as e:
        log.warning("[SAM3] failed to load %s: %s", npz_path, e)
        return []

    dets: List[MaskDetection] = []
    for i in range(masks.shape[0]):
        det = _mask_to_detection(masks[i].astype(bool),
                                 class_name=str(classes[i]) if i < len(classes) else "unknown",
                                 score=float(scores[i]) if i < len(scores) else 1.0,
                                 rgb=rgb_np)
        if det is not None:
            dets.append(det)
    return dets


# =============================================================================
# Tracker 1 — Mask IoU Tracker (greedy)
# =============================================================================

class MaskIoUTracker:
    """
    Greedy frame-to-frame association using a weighted combination of:
      • mask IoU                     (primary geometric cue)
      • normalized centroid distance (handles small-IoU-but-same-object cases
                                       from motion / segmentation jitter)
      • area ratio                   (penalizes matches with drastically
                                       different scale, e.g. mis-segmentation)

    Matching is greedy (highest combined score first), not globally optimal —
    a faithful mask-based adaptation of the association step in SORT
    (Bewley et al., ICRA 2016). Cheapest of the three trackers.

    Limitation: greedy matching can commit to a locally-optimal but globally
    suboptimal assignment when many similar objects are close together
    (e.g. a row of parked cars); Tracker 2 resolves this with an optimal solve.
    """

    def __init__(self, iou_weight: float = 0.6, centroid_weight: float = 0.25,
                 area_weight: float = 0.15, iou_match_threshold: float = TRACK_IOU_MATCH_THRESH,
                 max_age: int = TRACK_MAX_AGE, img_diag: float = 1000.0):
        self.iou_weight, self.centroid_weight, self.area_weight = iou_weight, centroid_weight, area_weight
        self.iou_match_threshold = iou_match_threshold
        self.max_age = max_age
        self.img_diag = img_diag
        self.tracks: List[Track] = []
        self._next_id = 1

    def _score(self, trk: Track, det: MaskDetection) -> float:
        iou = mask_iou(trk.mask, det.mask) if _bbox_iou(trk.bbox, det.bbox) > 0 else 0.0
        cx1, cy1 = trk.centroid
        cx2, cy2 = det.centroid
        centroid_sim = max(0.0, 1.0 - math.hypot(cx1 - cx2, cy1 - cy2) / self.img_diag)
        area_ratio = min(trk.area, det.area) / (max(trk.area, det.area) + 1e-6)
        return (self.iou_weight * iou + self.centroid_weight * centroid_sim +
                self.area_weight * area_ratio)

    def update(self, detections: List[MaskDetection]) -> List[Track]:
        matched_trk, matched_det = set(), set()

        if self.tracks and detections:
            pairs = [(self._score(trk, det), ti, di)
                     for ti, trk in enumerate(self.tracks)
                     for di, det in enumerate(detections)]
            pairs = [p for p in pairs if p[0] > 0]
            pairs.sort(key=lambda x: -x[0])
            for s, ti, di in pairs:
                if ti in matched_trk or di in matched_det or s < self.iou_match_threshold:
                    continue
                self._apply_match(ti, detections[di])
                matched_trk.add(ti)
                matched_det.add(di)

        for ti, trk in enumerate(self.tracks):
            if ti not in matched_trk:
                trk.age += 1
                trk.time_since_update += 1

        for di, det in enumerate(detections):
            if di not in matched_det:
                self._spawn_track(det)

        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]
        return self.tracks

    def _apply_match(self, ti: int, det: MaskDetection) -> None:
        trk = self.tracks[ti]
        trk.mask, trk.bbox, trk.centroid = det.mask, det.bbox, det.centroid
        trk.area, trk.class_name, trk.score, trk.embedding = det.area, det.class_name, det.score, det.embedding
        trk.age += 1
        trk.hits += 1
        trk.time_since_update = 0
        trk.history.append(det.centroid)

    def _spawn_track(self, det: MaskDetection) -> None:
        self.tracks.append(Track(track_id=self._next_id, mask=det.mask, bbox=det.bbox,
                                  centroid=det.centroid, area=det.area, class_name=det.class_name,
                                  score=det.score, embedding=det.embedding, history=[det.centroid]))
        self._next_id += 1


# =============================================================================
# Tracker 2 — Hungarian Assignment Tracker (optimal)
# =============================================================================

class HungarianMaskTracker:
    """
    Formulates frame-to-frame association as a linear assignment problem,
    solved optimally (Kuhn–Munkres) via scipy.optimize.linear_sum_assignment:

        minimize  sum_{i,j} C[i,j] * x[i,j]   s.t. each track <-> at most one detection

    Cost matrix combines four normalized terms in [0, 1]:

        C = w_iou * (1 - IoU) + w_centroid * (dist / img_diag)
            + w_area * (1 - area_ratio) + w_appearance * (1 - cosine_sim(embedding))

    The appearance term is skipped (weight renormalized across the other
    three) when either side has no embedding.

    Unlike Tracker 1's greedy matching, this always finds the assignment that
    minimizes TOTAL cost across all pairs simultaneously — important when
    several similar stationary objects sit next to each other (e.g. parked
    cars), where greedy IoU tracking can swap identities.
    """

    def __init__(self, w_iou: float = 0.5, w_centroid: float = 0.25, w_area: float = 0.15,
                 w_appearance: float = 0.10, max_cost: float = 0.85,
                 max_age: int = TRACK_MAX_AGE, img_diag: float = 1000.0):
        self.w_iou, self.w_centroid, self.w_area, self.w_appearance = w_iou, w_centroid, w_area, w_appearance
        self.max_cost = max_cost
        self.max_age = max_age
        self.img_diag = img_diag
        self.tracks: List[Track] = []
        self._next_id = 1

    def _cost(self, trk: Track, det: MaskDetection) -> float:
        iou = mask_iou(trk.mask, det.mask) if _bbox_iou(trk.bbox, det.bbox) > 0 else 0.0
        cx1, cy1 = trk.centroid
        cx2, cy2 = det.centroid
        dist_n = min(1.0, math.hypot(cx1 - cx2, cy1 - cy2) / self.img_diag)
        area_ratio = min(trk.area, det.area) / (max(trk.area, det.area) + 1e-6)

        w_iou, w_c, w_a = self.w_iou, self.w_centroid, self.w_area
        if trk.embedding is None or det.embedding is None:
            renorm = 1.0 / (w_iou + w_c + w_a + 1e-9)
            w_iou, w_c, w_a = w_iou * renorm, w_c * renorm, w_a * renorm
            appearance_term = 0.0
        else:
            num = float(np.dot(trk.embedding, det.embedding))
            den = float(np.linalg.norm(trk.embedding) * np.linalg.norm(det.embedding) + 1e-9)
            cos_sim = max(-1.0, min(1.0, num / den))
            appearance_term = self.w_appearance * (1.0 - cos_sim)

        return w_iou * (1.0 - iou) + w_c * dist_n + w_a * (1.0 - area_ratio) + appearance_term

    def update(self, detections: List[MaskDetection]) -> List[Track]:
        n_trk, n_det = len(self.tracks), len(detections)
        matched_trk, matched_det = set(), set()

        if n_trk > 0 and n_det > 0:
            C = np.full((n_trk, n_det), 10.0, dtype=np.float64)
            for ti, trk in enumerate(self.tracks):
                for di, det in enumerate(detections):
                    C[ti, di] = self._cost(trk, det)
            row_idx, col_idx = linear_sum_assignment(C)
            for ti, di in zip(row_idx, col_idx):
                if C[ti, di] > self.max_cost:
                    continue
                self._apply_match(ti, detections[di])
                matched_trk.add(ti)
                matched_det.add(di)

        for ti, trk in enumerate(self.tracks):
            if ti not in matched_trk:
                trk.age += 1
                trk.time_since_update += 1

        for di, det in enumerate(detections):
            if di not in matched_det:
                self._spawn_track(det)

        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]
        return self.tracks

    def _apply_match(self, ti: int, det: MaskDetection) -> None:
        trk = self.tracks[ti]
        trk.mask, trk.bbox, trk.centroid = det.mask, det.bbox, det.centroid
        trk.area, trk.class_name, trk.score, trk.embedding = det.area, det.class_name, det.score, det.embedding
        trk.age += 1
        trk.hits += 1
        trk.time_since_update = 0
        trk.history.append(det.centroid)

    def _spawn_track(self, det: MaskDetection) -> None:
        self.tracks.append(Track(track_id=self._next_id, mask=det.mask, bbox=det.bbox,
                                  centroid=det.centroid, area=det.area, class_name=det.class_name,
                                  score=det.score, embedding=det.embedding, history=[det.centroid]))
        self._next_id += 1


# =============================================================================
# Tracker 3 — ByteTrack-compatible Tracker (masks + Kalman-predicted centroid)
# =============================================================================

class _ConstantVelocityKF:
    """Minimal constant-velocity Kalman filter over 2-D centroid (cx, cy, vx, vy).
    Plays the role of the bbox Kalman filter in the original SORT/ByteTrack
    formulation, adapted to mask centroids since masks have no natural
    fixed-size box parametrization to filter."""

    def __init__(self, cx: float, cy: float):
        self.x = np.array([cx, cy, 0.0, 0.0], dtype=np.float64)
        self.P = np.eye(4) * 10.0
        self.F = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=np.float64)
        self.H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], dtype=np.float64)
        self.Q = np.eye(4) * 1.0
        self.R = np.eye(2) * 4.0

    def predict(self) -> Tuple[float, float]:
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return float(self.x[0]), float(self.x[1])

    def update(self, cx: float, cy: float) -> None:
        z = np.array([cx, cy], dtype=np.float64)
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P


class ByteTrackMaskTracker:
    """
    Adapts Zhang et al., "ByteTrack: Multi-Object Tracking by Associating
    Every Detection Box" (ECCV 2022), to instance segmentation masks. The
    core idea preserved from the original: don't discard low-confidence
    detections before matching — they are frequently partially-occluded
    TRUE objects, and dropping them (as Trackers 1-2 implicitly do by only
    seeing high-confidence SAM3 output) causes ID switches and fragmented
    tracks.

    Two-stage association per frame:
      Stage 1 (detections with score >= high_thresh):
          Hungarian-matched to ALL existing tracks, cost computed against
          each track's Kalman-PREDICTED centroid (not its last observation).
      Stage 2 (low_thresh <= score < high_thresh):
          Hungarian-matched only to tracks still unmatched after Stage 1 —
          recovers tracks through brief partial occlusion without ever
          letting a low-confidence detection spawn a new track.
      New tracks are created ONLY from unmatched Stage-1 detections.
      Unmatched tracks are kept ("lost") for `max_age` frames, Kalman-predicted
      forward with no measurement update, before deletion.

    Complexity: O(NM) per stage (two Hungarian solves), same order as Tracker 2.
    Limitation: still a nearest-neighbour tracker with no long-term Re-ID
    gallery; identity can be lost across occlusions longer than max_age.
    """

    def __init__(self, high_thresh: float = BYTETRACK_HIGH_THRESH, low_thresh: float = BYTETRACK_LOW_THRESH,
                 w_iou: float = 0.6, w_centroid: float = 0.4, max_cost: float = 0.9,
                 max_age: int = TRACK_MAX_AGE, img_diag: float = 1000.0):
        self.high_thresh, self.low_thresh = high_thresh, low_thresh
        self.w_iou, self.w_centroid = w_iou, w_centroid
        self.max_cost = max_cost
        self.max_age = max_age
        self.img_diag = img_diag
        self.tracks: List[Track] = []
        self._kfs: Dict[int, _ConstantVelocityKF] = {}
        self._next_id = 1

    def _cost(self, trk: Track, pred_centroid: Tuple[float, float], det: MaskDetection) -> float:
        iou = mask_iou(trk.mask, det.mask) if _bbox_iou(trk.bbox, det.bbox) > 0 else 0.0
        dist_n = min(1.0, math.hypot(pred_centroid[0] - det.centroid[0],
                                     pred_centroid[1] - det.centroid[1]) / self.img_diag)
        return self.w_iou * (1.0 - iou) + self.w_centroid * dist_n

    def _hungarian_match(self, track_idxs: List[int], dets: List[MaskDetection],
                          pred_centroids: Dict[int, Tuple[float, float]]):
        if not track_idxs or not dets:
            return [], list(range(len(dets))), list(track_idxs)
        C = np.full((len(track_idxs), len(dets)), 10.0)
        for r, ti in enumerate(track_idxs):
            for c, det in enumerate(dets):
                C[r, c] = self._cost(self.tracks[ti], pred_centroids[ti], det)
        rows, cols = linear_sum_assignment(C)
        matches: List[Tuple[int, int]] = []
        unmatched_dets, unmatched_trks = set(range(len(dets))), set(track_idxs)
        for r, c in zip(rows, cols):
            if C[r, c] <= self.max_cost:
                matches.append((track_idxs[r], c))
                unmatched_dets.discard(c)
                unmatched_trks.discard(track_idxs[r])
        return matches, sorted(unmatched_dets), sorted(unmatched_trks)

    def update(self, detections: List[MaskDetection]) -> List[Track]:
        high = [d for d in detections if d.score >= self.high_thresh]
        low = [d for d in detections if self.low_thresh <= d.score < self.high_thresh]

        pred_centroids: Dict[int, Tuple[float, float]] = {}
        for ti, trk in enumerate(self.tracks):
            kf = self._kfs.get(trk.track_id)
            pred_centroids[ti] = kf.predict() if kf is not None else trk.centroid

        all_track_idx = list(range(len(self.tracks)))
        matched_this_frame = set()

        matches1, unmatched_high, unmatched_trk1 = self._hungarian_match(all_track_idx, high, pred_centroids)
        for ti, di in matches1:
            self._apply_match(ti, high[di])
            matched_this_frame.add(ti)

        matches2, unmatched_low, unmatched_trk2 = self._hungarian_match(unmatched_trk1, low, pred_centroids)
        for ti, di in matches2:
            self._apply_match(ti, low[di])
            matched_this_frame.add(ti)

        for ti, trk in enumerate(self.tracks):
            if ti not in matched_this_frame:
                trk.age += 1
                trk.time_since_update += 1

        for di in unmatched_high:
            self._spawn_track(high[di])

        alive = []
        for trk in self.tracks:
            if trk.time_since_update <= self.max_age:
                alive.append(trk)
            else:
                self._kfs.pop(trk.track_id, None)
        self.tracks = alive
        return self.tracks

    def _apply_match(self, ti: int, det: MaskDetection) -> None:
        trk = self.tracks[ti]
        trk.mask, trk.bbox, trk.centroid = det.mask, det.bbox, det.centroid
        trk.area, trk.class_name, trk.score, trk.embedding = det.area, det.class_name, det.score, det.embedding
        trk.age += 1
        trk.hits += 1
        trk.time_since_update = 0
        trk.history.append(det.centroid)
        kf = self._kfs.get(trk.track_id)
        if kf is not None:
            kf.update(*det.centroid)

    def _spawn_track(self, det: MaskDetection) -> None:
        new_trk = Track(track_id=self._next_id, mask=det.mask, bbox=det.bbox, centroid=det.centroid,
                         area=det.area, class_name=det.class_name, score=det.score,
                         embedding=det.embedding, history=[det.centroid])
        self.tracks.append(new_trk)
        self._kfs[new_trk.track_id] = _ConstantVelocityKF(*det.centroid)
        self._next_id += 1


def build_tracker(tracker_type: str = TRACKER_TYPE, img_diag: float = 1000.0):
    """Factory selecting one of the three trackers by TRACKER_TYPE config."""
    if tracker_type == "iou":
        return MaskIoUTracker(img_diag=img_diag)
    if tracker_type == "hungarian":
        return HungarianMaskTracker(img_diag=img_diag)
    if tracker_type == "bytetrack":
        return ByteTrackMaskTracker(img_diag=img_diag)
    raise ValueError(f"Unknown TRACKER_TYPE: {tracker_type}")


print("SAM3 mask loading + 3 trackers ready.")
print("  load_sam3_masks_for_frame -> List[MaskDetection] per frame (adapter, edit path logic if needed)")
print("  MaskIoUTracker            -> Tracker 1 (greedy)")
print("  HungarianMaskTracker      -> Tracker 2 (optimal assignment)")
print("  ByteTrackMaskTracker      -> Tracker 3 (two-stage + Kalman-predicted centroid)")
print("  build_tracker(TRACKER_TYPE) -> convenience factory used by the training loop")


## 2d · Stationary vs. Moving Object Filter (Part 4)

### Purpose

The new losses in Part 5 assume every tracked object is **physically stationary**, so that its apparent depth/position change between frames is caused *entirely* by the camera's own ego-motion. If a moving car or pedestrian leaks into that pool, it injects a false ego-motion signal directly into the loss gradient. This section filters `Track` objects down to a trustworthy stationary subset before they reach any loss.

### Two independent filtering signals

**4a — Manual class filtering** (`is_class_dynamic`): SAM3's class label is checked against a curated `DYNAMIC_CLASS_NAMES` set (car, truck, bus, person, bicycle, motorcycle, rider, train, …). Unknown/unlisted classes default to **dynamic** (excluded) — for this use case, treating an unrecognized object as unsafe-to-trust is the conservative choice, since one wrongly-included moving object corrupts every stationary-consistency loss term.

**4b — Optical-flow residual filtering** (`is_object_stationary_by_flow`): Dense Farneback optical flow is computed between the frame pair. A truly static object's apparent flow is caused *only* by ego-motion (the camera moving past it), so once the scene's global ego-flow (estimated from a large static region — the union of class-static masks, or the upper/sky band as a fallback) is subtracted from the object's median flow, a genuinely stationary object should have residual ≈ 0 px. A moving object retains a non-zero residual. This catches the case class labels can't — e.g. a "parked" car that's actually pulling away, or a class label SAM3 got wrong.

### Why require both (`DYNAMIC_FILTER_MODE = "both"`)

Class filtering is fast but brittle to label noise; flow filtering is more direct but can be fooled by textureless surfaces (Farneback flow is unreliable on e.g. blank walls or the sky). Requiring an object to pass **both** checks trades recall for precision — for a loss term whose entire validity rests on the "this object didn't move" assumption, false positives are far more damaging than a few missed true stationary objects.

### Limitations

Optical-flow residual filtering degrades in low-texture regions and at night (KITTI is daytime-only, so this is a minor concern here); the ego-flow estimate is a median-based heuristic, not a solved fundamental-matrix / homography decomposition, so it can be biased when the static-mask union is small or absent (falls back to the image's upper quarter, which is usually sky/background in KITTI's forward-facing camera geometry).


In [ ]:
# =============================================================================
# Moving-object filter (Part 4)
# =============================================================================

DYNAMIC_CLASS_NAMES = {
    "car", "truck", "bus", "van", "person", "pedestrian", "rider", "riders",
    "bicycle", "bicyclist", "cyclist", "motorcycle", "motorbike", "train", "trailer",
}
STATIC_CLASS_NAMES = {
    "pole", "traffic sign", "traffic light", "building", "wall", "fence",
    "vegetation", "terrain", "sidewalk", "road", "sky", "guard rail",
    "parking", "static", "bridge", "tunnel",
}


def is_class_dynamic(class_name: str) -> bool:
    """Manual filtering rule (Part 4a). Unknown/unlisted classes default to
    'dynamic' (excluded) — see markdown above for why this default is
    deliberately conservative for this use case."""
    name = class_name.strip().lower()
    if name in STATIC_CLASS_NAMES:
        return False
    if name in DYNAMIC_CLASS_NAMES:
        return True
    return True  # unknown class -> conservatively treat as dynamic (exclude)


def compute_dense_flow(rgb_curr_np: np.ndarray, rgb_next_np: np.ndarray) -> np.ndarray:
    """Farneback dense optical flow between two uint8 RGB frames. Returns
    flow [H, W, 2] in pixels (Part 4b)."""
    g1 = cv2.cvtColor(rgb_curr_np, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(rgb_next_np, cv2.COLOR_RGB2GRAY)
    return cv2.calcOpticalFlowFarneback(
        g1, g2, None, pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0)


def estimate_ego_flow(flow: np.ndarray, static_mask_union: Optional[np.ndarray] = None) -> np.ndarray:
    """Median flow over background pixels as a cheap proxy for the
    ego-motion-induced flow field. Prefers the union of class-static masks;
    falls back to the image's upper quarter (sky/distant background in
    KITTI's forward-facing camera) if no static masks are available."""
    if static_mask_union is not None and static_mask_union.sum() > 100:
        return np.median(flow[static_mask_union], axis=0)
    h = flow.shape[0]
    return np.median(flow[: h // 4, :, :].reshape(-1, 2), axis=0)


def is_object_stationary_by_flow(mask: np.ndarray, flow: np.ndarray,
                                  ego_flow: Optional[np.ndarray] = None,
                                  residual_threshold: float = FLOW_RESIDUAL_PX) -> bool:
    """
    Flags an object as stationary if its median optical flow, AFTER
    subtracting the scene's ego-motion-induced background flow, is below
    `residual_threshold` pixels. See markdown above for the reasoning.
    """
    if mask.sum() < 20:
        return False  # too small a region to trust a flow estimate; exclude conservatively
    obj_median = np.median(flow[mask], axis=0)
    if ego_flow is None:
        ego_flow = np.zeros(2, dtype=np.float32)
    residual = float(np.linalg.norm(obj_median - ego_flow))
    return residual <= residual_threshold


def filter_dynamic_objects(tracks: List[Track],
                            rgb_curr_np: Optional[np.ndarray] = None,
                            rgb_next_np: Optional[np.ndarray] = None,
                            mode: str = DYNAMIC_FILTER_MODE,
                            flow_residual_threshold: float = FLOW_RESIDUAL_PX) -> List[Track]:
    """
    Combined moving-object filter (Part 4). Returns only the STATIONARY
    subset of `tracks`, ready to feed into Part 5's stationary-object losses.

    mode:
      "class" -> manual class-name filtering only (fast, no RGB needed)
      "flow"  -> optical-flow residual filtering only
      "both"  -> object must pass BOTH checks (default; see markdown above)
    """
    if mode not in ("class", "flow", "both"):
        raise ValueError(f"Unknown DYNAMIC_FILTER_MODE: {mode}")

    class_static = [not is_class_dynamic(t.class_name) for t in tracks]

    if mode == "class":
        return [t for t, keep in zip(tracks, class_static) if keep]

    if rgb_curr_np is None or rgb_next_np is None:
        log.warning("[filter_dynamic_objects] mode=%s requires rgb frames; "
                     "falling back to class-only filtering for this pair.", mode)
        return [t for t, keep in zip(tracks, class_static) if keep]

    flow = compute_dense_flow(rgb_curr_np, rgb_next_np)
    static_union = None
    if any(class_static):
        static_union = np.zeros(flow.shape[:2], dtype=bool)
        for t, keep in zip(tracks, class_static):
            if keep and t.mask.shape == static_union.shape:
                static_union |= t.mask
    ego_flow = estimate_ego_flow(flow, static_union)

    flow_static = [is_object_stationary_by_flow(t.mask, flow, ego_flow, flow_residual_threshold)
                   for t in tracks]

    if mode == "flow":
        return [t for t, keep in zip(tracks, flow_static) if keep]

    return [t for t, c_ok, f_ok in zip(tracks, class_static, flow_static) if c_ok and f_ok]


# =============================================================================
# Self-test: verify all 3 trackers preserve identity across a synthetic
# 2-object sequence, and that the dynamic filter separates them correctly.
# =============================================================================

def _make_synthetic_mask(h: int, w: int, cx: float, cy: float, r: float) -> np.ndarray:
    yy, xx = np.mgrid[0:h, 0:w]
    return ((xx - cx) ** 2 + (yy - cy) ** 2) <= r * r


def _self_test_tracking_and_filter() -> None:
    h, w = 128, 256
    img_diag = math.hypot(h, w)

    frame0 = [
        _mask_to_detection(_make_synthetic_mask(h, w, 40, 60, 12), class_name="pole", score=0.95),
        _mask_to_detection(_make_synthetic_mask(h, w, 150, 70, 18), class_name="car", score=0.9),
    ]
    frame1 = [
        _mask_to_detection(_make_synthetic_mask(h, w, 42, 60, 12), class_name="pole", score=0.95),
        _mask_to_detection(_make_synthetic_mask(h, w, 165, 70, 18), class_name="car", score=0.9),
    ]

    for name, trk in [("MaskIoUTracker", MaskIoUTracker(img_diag=img_diag)),
                       ("HungarianMaskTracker", HungarianMaskTracker(img_diag=img_diag)),
                       ("ByteTrackMaskTracker", ByteTrackMaskTracker(img_diag=img_diag))]:
        t1 = trk.update(frame0)
        t2 = trk.update(frame1)
        ids1, ids2 = sorted(t.track_id for t in t1), sorted(t.track_id for t in t2)
        assert ids1 == ids2, f"{name}: track IDs changed across frames ({ids1} -> {ids2})"
        print(f"  {name}: OK  ({len(t2)} tracks, ids={ids2})")

    # Dynamic filter, class-mode only (no synthetic RGB needed)
    tracker = HungarianMaskTracker(img_diag=img_diag)
    tracks = tracker.update(frame0)
    stationary = filter_dynamic_objects(tracks, mode="class")
    assert len(stationary) == 1 and stationary[0].class_name == "pole", \
        f"class filter should keep only the pole, got {[t.class_name for t in stationary]}"
    print(f"  filter_dynamic_objects(mode='class'): OK (kept {[t.class_name for t in stationary]})")


_self_test_tracking_and_filter()
print("Tracking + dynamic-filter self-test passed "
      "(synthetic 2-object sequence: all 3 trackers preserve IDs, filter isolates the stationary pole).")


## 3 · Model wrapper, Pose Network, and Differentiable Warping


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# DA3 model wrapper
# ─────────────────────────────────────────────────────────────────────────────
_DA3_TRANSFORM = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
])


def load_da3_model(device: torch.device) -> nn.Module:
    from depth_anything_3.api import DepthAnything3
    log.info("Loading %s ...", MODEL_ID)
    model = DepthAnything3.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR).to(device)
    return model


def _apply_arch(model: nn.Module, arch_mode: str) -> None:
    if arch_mode == "full":
        for p in model.parameters():
            p.requires_grad_(True)
        model.model.train()
        if hasattr(model.model, "gradient_checkpointing_enable"):
            model.model.gradient_checkpointing_enable()
        log.info("  [arch=full] All parameters trainable.")
        return

    if arch_mode == "decoder_only":
        for name, param in model.named_parameters():
            freeze = any(k in name for k in
                         ["encoder", "pretrained", "backbone", "net.net"])
            param.requires_grad_(not freeze)
        model.model.train()
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        log.info("  [arch=decoder_only] %.1fM params trainable", trainable/1e6)
        return

    if arch_mode == "last2_blocks":
        for p in model.parameters():
            p.requires_grad_(False)
        decoder_block_names = [
            name for name, _ in model.named_parameters()
            if "decoder" in name.lower() or "head" in name.lower()
               or "depth_head" in name.lower() or "scratch" in name.lower()
        ]
        last_n = max(1, int(0.30 * len(decoder_block_names)))
        for name in decoder_block_names[-last_n:]:
            model.get_parameter(name).requires_grad_(True)
        model.model.train()
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        log.info("  [arch=last2_blocks] %.1fM params trainable", trainable/1e6)
        return

    raise ValueError(f"Unknown arch_mode: {arch_mode}")


def forward_batch(model: nn.Module, rgb_batch: torch.Tensor,
                  K_batch: torch.Tensor, has_K: torch.Tensor) -> torch.Tensor:
    # Differentiable forward pass -> metric depth [B, 1, H, W].
    B, C, H, W = rgb_batch.shape
    inp = rgb_batch.unsqueeze(1)
    preds = model.model(inp, export_feat_layers=[])

    if isinstance(preds, dict):
        raw = preds.get("depth", preds.get("metric_depth", list(preds.values())[0]))
        fx_pred = torch.where(
            has_K.view(B, 1, 1, 1),
            K_batch[:, 0, 0].view(B, 1, 1, 1),
            torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                       device=rgb_batch.device, dtype=rgb_batch.dtype),
        )
    elif isinstance(preds, (tuple, list)):
        raw = preds[0]
        fx_pred = torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                             device=rgb_batch.device, dtype=rgb_batch.dtype)
    else:
        raw = preds
        fx_pred = torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                             device=rgb_batch.device, dtype=rgb_batch.dtype)

    if raw.dim() == 3:
        raw = raw.unsqueeze(1)
    depth = (fx_pred * raw) / ORTHO_NORM
    if depth.shape[-2:] != (H, W):
        depth = F.interpolate(depth, size=(H, W), mode="bilinear",
                              align_corners=False)
    return depth


@torch.no_grad()
def run_da3_inference(model: nn.Module, image_np: np.ndarray,
                      device: torch.device, fx: float = FOCAL_LENGTH_X) -> np.ndarray:
    # Run DA3METRIC-LARGE inference on one uint8 RGB image (H x W x 3).
    orig_h, orig_w = image_np.shape[:2]
    new_h = int(round(orig_h / 14) * 14)
    new_w = int(round(orig_w / 14) * 14)
    img_pil = Image.fromarray(image_np).resize((new_w, new_h),
                                                Image.Resampling.BICUBIC)
    t_img = _DA3_TRANSFORM(img_pil)
    inp   = t_img.unsqueeze(0).unsqueeze(1).to(device)
    model.model.eval()
    preds = model(inp, export_feat_layers=[])
    if isinstance(preds, dict):
        raw = preds.get("depth", preds.get("metric_depth", list(preds.values())[0]))
    elif isinstance(preds, (tuple, list)):
        raw = preds[0]
    else:
        raw = preds
    focal_used: float = fx
    if isinstance(raw, torch.Tensor):
        if raw.dim() == 3:
            raw = raw.unsqueeze(1)
        depth = (focal_used * raw) / ORTHO_NORM
        depth = F.interpolate(depth, size=(orig_h, orig_w),
                              mode="bilinear", align_corners=False)
    else:
        raise RuntimeError(f"Unexpected model output type {type(raw)}")
    return depth.squeeze().cpu().float().numpy()


# ─────────────────────────────────────────────────────────────────────────────
# Pose Network — SfMLearner / Monodepth2-style ResNet18 with 6-DoF head
# ─────────────────────────────────────────────────────────────────────────────
class PoseNet(nn.Module):
    # Predicts 6-DoF relative pose (axis-angle + translation) from a pair of
    # consecutive RGB frames stacked along the channel dim.
    #
    # Architecture (Monodepth2, Godard ICCV 2019):
    #   - ResNet18 encoder, first conv replaced to accept 6 channels (2 frames).
    #   - 4-layer convolutional pose head (256 -> 256 -> 256 -> 6),
    #     zero-init last layer so initial pose ~ identity.
    # Output: pose tensor of shape [B, 6]  ->  (axisangle[3], translation[3]).

    def __init__(self):
        super().__init__()
        try:
            weights = tvm.ResNet18_Weights.DEFAULT
        except Exception:
            weights = None
        backbone = tvm.resnet18(weights=weights)

        # Replace first conv to accept 6 input channels (two stacked RGB frames)
        old_conv = backbone.conv1
        new_conv = nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv.weight.copy_(
                torch.cat([old_conv.weight, old_conv.weight], dim=1) / 2.0)
        backbone.conv1 = new_conv

        # Strip the classification head; keep features (B, 512, H/32, W/32)
        self.encoder = nn.Sequential(*list(backbone.children())[:-2])

        # Pose head: 4 conv layers
        self.pose_head = nn.Sequential(
            nn.Conv2d(512, 256, 1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, stride=1, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, stride=1, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 6, 1),
        )
        # Zero-init final layer so initial pose ~ identity
        nn.init.zeros_(self.pose_head[-1].weight)
        nn.init.zeros_(self.pose_head[-1].bias)

    def forward(self, rgb_curr: torch.Tensor, rgb_next: torch.Tensor) -> torch.Tensor:
        # rgb_curr, rgb_next : [B, 3, H, W] (ImageNet-normalised)
        # Returns pose : [B, 6]
        x = torch.cat([rgb_curr, rgb_next], dim=1)
        f = self.encoder(x)
        p = self.pose_head(f)
        # Mean-pool spatial dims -> [B, 6]; scale-down for numerical stability
        p = p.mean(dim=(2, 3))
        p = 0.01 * p
        return p


# ─────────────────────────────────────────────────────────────────────────────
# SE(3) helpers — axis-angle -> rotation matrix (differentiable)
# ─────────────────────────────────────────────────────────────────────────────
# def axisangle_to_rotmat(aa: torch.Tensor) -> torch.Tensor:
#     # Convert axis-angle [B, 3] -> rotation matrix [B, 3, 3] via Rodrigues' formula.
#     theta = aa.norm(dim=1, keepdim=True).clamp(min=EPS)
#     k     = aa / theta
#     cos_t = torch.cos(theta).unsqueeze(-1)
#     sin_t = torch.sin(theta).unsqueeze(-1)
#     one_minus_cos = (1.0 - cos_t)

#     K = torch.zeros(aa.shape[0], 3, 3, device=aa.device, dtype=aa.dtype)
#     K[:, 0, 1] = -k[:, 2]; K[:, 0, 2] =  k[:, 1]
#     K[:, 1, 0] =  k[:, 2]; K[:, 1, 2] = -k[:, 0]
#     K[:, 2, 0] = -k[:, 1]; K[:, 2, 1] =  k[:, 0]

#     I = torch.eye(3, device=aa.device, dtype=aa.dtype).unsqueeze(0).expand_as(K)
#     R = (cos_t.unsqueeze(-1) * I +
#          sin_t.unsqueeze(-1) * K +
#          one_minus_cos.unsqueeze(-1) * (k.unsqueeze(2) @ k.unsqueeze(1)))
#     return R
def axisangle_to_rotmat(aa: torch.Tensor) -> torch.Tensor:
    # Convert axis-angle [B, 3] -> rotation matrix [B, 3, 3] via Rodrigues' formula.
    theta = aa.norm(dim=1, keepdim=True).clamp(min=EPS)  # [B, 1]
    k     = aa / theta                                    # [B, 3]
    
    # Shape [B, 1, 1] — one unsqueeze is enough to broadcast over [B, 3, 3]
    cos_t         = torch.cos(theta).unsqueeze(-1)        # [B, 1, 1]
    sin_t         = torch.sin(theta).unsqueeze(-1)        # [B, 1, 1]
    one_minus_cos = (1.0 - cos_t)                         # [B, 1, 1]

    K = torch.zeros(aa.shape[0], 3, 3, device=aa.device, dtype=aa.dtype)
    K[:, 0, 1] = -k[:, 2]; K[:, 0, 2] =  k[:, 1]
    K[:, 1, 0] =  k[:, 2]; K[:, 1, 2] = -k[:, 0]
    K[:, 2, 0] = -k[:, 1]; K[:, 2, 1] =  k[:, 0]

    I = torch.eye(3, device=aa.device, dtype=aa.dtype).unsqueeze(0).expand_as(K)
    
    # No extra unsqueeze(-1) here — cos_t/sin_t/one_minus_cos are already [B,1,1]
    R = (cos_t         * I +
         sin_t         * K +
         one_minus_cos * (k.unsqueeze(2) @ k.unsqueeze(1)))
    return R  # [B, 3, 3]  ✓

def pose_to_se3(pose: torch.Tensor) -> torch.Tensor:
    # Convert 6-DoF pose [B, 6] (axisangle[3], translation[3]) -> SE(3) [B, 4, 4].
    B = pose.shape[0]
    aa, t = pose[:, :3], pose[:, 3:]
    R = axisangle_to_rotmat(aa)
    T_ = torch.eye(4, device=pose.device, dtype=pose.dtype).unsqueeze(0).repeat(B, 1, 1)
    T_[:, :3, :3] = R
    T_[:, :3,  3] = t
    return T_


# ─────────────────────────────────────────────────────────────────────────────
# Differentiable inverse-warping
# ─────────────────────────────────────────────────────────────────────────────
class BackprojectDepth(nn.Module):
    # Backproject depth map to 3-D camera-frame points using inverse intrinsics.
    def __init__(self, height, width):
        super().__init__()
        self.H, self.W = height, width
        u, v = torch.meshgrid(
            torch.arange(0, height, dtype=torch.float32),
            torch.arange(0, width,  dtype=torch.float32),
            indexing="ij"
        )
        pix = torch.stack([v, u], dim=0).view(2, -1)
        ones = torch.ones(1, height * width)
        pix_homog = torch.cat([pix, ones], dim=0).unsqueeze(0)   # [1, 3, H*W]
        self.register_buffer("pix_homog", pix_homog)

    def forward(self, depth: torch.Tensor, inv_K: torch.Tensor) -> torch.Tensor:
        # depth : [B, 1, H, W]; inv_K : [B, 3, 3]; returns [B, 4, H*W]
        B = depth.shape[0]
        pix = self.pix_homog.expand(B, -1, -1)
        cam_pts = torch.matmul(inv_K, pix)                       # [B, 3, H*W]
        cam_pts = depth.view(B, 1, -1) * cam_pts                 # scale by Z
        cam_pts = torch.cat(
            [cam_pts, torch.ones(B, 1, self.H * self.W, device=depth.device)],
            dim=1
        )
        return cam_pts


class Project3D(nn.Module):
    # Reproject 3-D camera-frame points to pixels using intrinsics + pose.
    # def __init__(self, height, width):
    #     super().__init__()
    #     self.H, self.W = height, width

    # def forward(self, points_3d: torch.Tensor, K: torch.Tensor,
    #             T: torch.Tensor) -> torch.Tensor:
    #     # Returns pix_norm : [B, H, W, 2] in [-1, 1]  (for grid_sample).
    #     B = points_3d.shape[0]
    #     P = torch.matmul(K, T[:, :3, :])                        # [B, 3, 4]
    #     cam_pts = torch.matmul(P, points_3d)                    # [B, 3, H*W]
    #     pix = cam_pts[:, :2] / (cam_pts[:, 2:3] + EPS)
    #     pix = pix.view(B, 2, self.H, self.W)
    #     pix_norm = pix.clone()
    #     pix_norm[:, 0] = 2.0 * pix[:, 0] / (self.W - 1) - 1.0
    #     pix_norm[:, 1] = 2.0 * pix[:, 1] / (self.H - 1) - 1.0
    #     return pix_norm.permute(0, 2, 3, 1)
    def __init__(self, height, width):
        super().__init__()
        self.H, self.W = height, width

    def forward(self, points_3d, K, T):
        B = points_3d.shape[0]
        P = torch.matmul(K, T[:, :3, :])                        # [B, 3, 4]
        cam_pts = torch.matmul(P, points_3d)                    # [B, 3, H*W]
        # Clamp z to float16-safe min — EPS=1e-6 is below float16 range
        z = cam_pts[:, 2:3].clamp(min=0.01)
        pix = cam_pts[:, :2] / z
        pix = pix.view(B, 2, self.H, self.W)
        pix_norm = pix.clone()
        pix_norm[:, 0] = 2.0 * pix[:, 0] / (self.W - 1) - 1.0
        pix_norm[:, 1] = 2.0 * pix[:, 1] / (self.H - 1) - 1.0
        # Clamp to prevent inf/NaN reaching grid_sample
        pix_norm = pix_norm.clamp(-10.0, 10.0)
        return pix_norm.permute(0, 2, 3, 1)



# ─── Cached BackprojectDepth / Project3D (avoid per-batch GPU allocation) ─────
_backproj_cache: Dict[Tuple, "BackprojectDepth"] = {}
_project_cache:  Dict[Tuple, "Project3D"]        = {}

def _get_backproj(H: int, W: int, device: torch.device) -> "BackprojectDepth":
    key = (H, W, device)
    if key not in _backproj_cache:
        _backproj_cache[key] = BackprojectDepth(H, W).to(device)
    return _backproj_cache[key]

def _get_project(H: int, W: int, device: torch.device) -> "Project3D":
    key = (H, W, device)
    if key not in _project_cache:
        _project_cache[key] = Project3D(H, W).to(device)
    return _project_cache[key]


def _safe_inv_K(K: torch.Tensor) -> torch.Tensor:
    """Batched K inversion, safe under autocast (forces float32)."""
    B = K.shape[0]
    with torch.amp.autocast("cuda", enabled=False):
        K_f32 = K.float()
        eye = torch.eye(3, device=K.device, dtype=torch.float32).unsqueeze(0).expand(B, -1, -1)
        try:
            return torch.linalg.solve(K_f32, eye)
        except torch.linalg.LinAlgError:
            K_safe = K_f32.clone()
            diag = K_safe.diagonal(dim1=-2, dim2=-1)
            diag[diag.abs() < 1e-6] = 1.0
            return torch.linalg.solve(K_safe, eye)

def warp_image(image_next: torch.Tensor,
               depth_curr: torch.Tensor,
               pose_curr_to_next: torch.Tensor,
               K: torch.Tensor,
               has_K: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    # Inverse-warp `image_next` back into the current frame's coordinate system
    # using the depth predicted on the current frame.
    # Returns (warped_image [B,C,H,W], valid_mask [B,1,H,W]).
    B, C, H, W = image_next.shape

    inv_K = _safe_inv_K(K)

    backproject = _get_backproj(H, W, image_next.device)
    project     = _get_project(H, W, image_next.device)

    pts3d       = backproject(depth_curr, inv_K)
    pix_in_next = project(pts3d, K, pose_curr_to_next)
    warped      = F.grid_sample(image_next, pix_in_next,
                                mode="bilinear", padding_mode="zeros",
                                align_corners=True)

    valid = ((pix_in_next[..., 0].abs() <= 1.0) &
             (pix_in_next[..., 1].abs() <= 1.0)).float().unsqueeze(1)
    # ── Sanitize: replace any residual NaN/inf with 0 ─────────────────────
    warped = torch.nan_to_num(warped, nan=0.0, posinf=0.0, neginf=0.0)

    # ── Zero out contribution from samples with no real intrinsics ────────────
    # if has_K is not None:
    #     k_mask = has_K.view(B, 1, 1, 1).to(dtype=warped.dtype, device=warped.device)
    #     warped = warped * k_mask
    #     valid  = valid  * k_mask
    #NOte: has_K masking REMOVED. Since _DEFAULT_K is a valid KITTI
    # intrinsic matrix, warping with it produces a usable (approximate)
    # photometric signal. Zeroing out the loss for these samples was
    # preventing all gradient flow.

    return warped, valid


def warp_depth(depth_next: torch.Tensor,
               depth_curr: torch.Tensor,
               pose_curr_to_next: torch.Tensor,
               K: torch.Tensor, has_K: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    # Warp the NEXT-frame depth map back to the CURRENT frame's pixel grid
    # using the predicted current-frame depth + relative pose.
    # Used by `temporal_warp_consistency` and `geometric_consistency` losses.
    return warp_image(depth_next, depth_curr, pose_curr_to_next, K)


print("DA3 model wrapper + PoseNet + differentiable warping ready.")


DA3 model wrapper + PoseNet + differentiable warping ready.


## 4 · Self-Supervised Loss Functions (13 variants)


In [7]:
# =============================================================================
# Self-supervised loss functions for monocular depth — 13 variants (2015-2025)
# =============================================================================
#
# Every loss is a nn.Module returning (total_loss, sublosses_dict). All accept
# the unified `_unified_forward()` call signature defined at the end of this
# cell. Losses that need raw RGB / next-frame data declare it in their forward.
# =============================================================================

def _safe_log(x: torch.Tensor) -> torch.Tensor:
    return torch.log(x.clamp(min=EPS))


# ─────────────────────────────────────────────────────────────────────────────
# Building blocks used by multiple losses
# ─────────────────────────────────────────────────────────────────────────────

def ssim_loss(pred: torch.Tensor, gt: torch.Tensor,
              mask: torch.Tensor, window_size: int = 7,
              C1: float = 1e-4, C2: float = 9e-4) -> torch.Tensor:
    # Local SSIM dissimilarity in [0, 1].
    pad = window_size // 2
    kernel = torch.ones(1, 1, window_size, window_size,
                        device=pred.device, dtype=pred.dtype) / (window_size ** 2)

    def lm(x):
        ch = x.shape[1]
        k = kernel.expand(ch, 1, window_size, window_size)
        return F.conv2d(x, k, padding=pad, groups=ch)

    mu_p  = lm(pred); mu_g  = lm(gt)
    mu_pp = lm(pred * pred); mu_gg = lm(gt * gt); mu_pg = lm(pred * gt)
    sigma_p  = (mu_pp - mu_p * mu_p).clamp(min=0)
    sigma_g  = (mu_gg - mu_g * mu_g).clamp(min=0)
    sigma_pg = mu_pg - mu_p * mu_g
    ssim_map = ((2 * mu_p * mu_g + C1) * (2 * sigma_pg + C2)) / (
                (mu_p**2 + mu_g**2 + C1) * (sigma_p + sigma_g + C2))
    loss_map = ((1 - ssim_map) / 2.0).clamp(0, 1)
    # Average across channels first, then take masked spatial mean
    loss_map = loss_map.mean(dim=1, keepdim=True)
    return (loss_map * mask).sum() / (mask.sum() + EPS)


def photometric_residual(warped: torch.Tensor,
                         target: torch.Tensor,
                         alpha: float = SSIM_ALPHA) -> torch.Tensor:
    # Per-pixel photometric residual = alpha * SSIM + (1 - alpha) * L1.
    # Returns a [B, 1, H, W] map (Monodepth2 standard).
    pad = 1
    kernel = torch.ones(1, 1, 3, 3, device=warped.device, dtype=warped.dtype) / 9.0
    ch = warped.shape[1]
    k = kernel.expand(ch, 1, 3, 3)
    mu_p  = F.conv2d(warped, k, padding=pad, groups=ch)
    mu_g  = F.conv2d(target, k, padding=pad, groups=ch)
    mu_pp = F.conv2d(warped * warped, k, padding=pad, groups=ch)
    mu_gg = F.conv2d(target * target, k, padding=pad, groups=ch)
    mu_pg = F.conv2d(warped * target, k, padding=pad, groups=ch)
    sigma_p  = (mu_pp - mu_p * mu_p).clamp(min=0)
    sigma_g  = (mu_gg - mu_g * mu_g).clamp(min=0)
    sigma_pg = mu_pg - mu_p * mu_g
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu_p * mu_g + C1) * (2 * sigma_pg + C2)) / (
                (mu_p**2 + mu_g**2 + C1) * (sigma_p + sigma_g + C2))
    ssim_loss_map = ((1 - ssim_map) / 2.0).clamp(0, 1).mean(dim=1, keepdim=True)
    l1_map = (warped - target).abs().mean(dim=1, keepdim=True)
    return alpha * ssim_loss_map + (1 - alpha) * l1_map


def edge_aware_smoothness_map(disp: torch.Tensor,
                              image: torch.Tensor,
                              decay: float = SMOOTHNESS_DECAY) -> torch.Tensor:
    # Edge-aware smoothness loss on a disparity map (Godard et al. 2017).
    #   L_s = sum_p (|d_x disp| exp(-|d_x I|) + |d_y disp| exp(-|d_y I|))
    # `disp` here is 1/depth so that loss strength is invariant to depth scale.
    mean_disp = disp.mean(dim=(2, 3), keepdim=True).clamp(min=EPS)
    disp_norm = disp / mean_disp

    grad_disp_x = (disp_norm[:, :, :, :-1] - disp_norm[:, :, :, 1:]).abs()
    grad_disp_y = (disp_norm[:, :, :-1, :] - disp_norm[:, :, 1:, :]).abs()
    grad_img_x  = image[:, :, :, :-1] - image[:, :, :, 1:]
    grad_img_y  = image[:, :, :-1, :] - image[:, :, 1:, :]
    weight_x    = torch.exp(-decay * grad_img_x.abs().mean(dim=1, keepdim=True))
    weight_y    = torch.exp(-decay * grad_img_y.abs().mean(dim=1, keepdim=True))
    return (grad_disp_x * weight_x).mean() + (grad_disp_y * weight_y).mean()


# ─────────────────────────────────────────────────────────────────────────────
# 1. photo_l1 — Pure L1 photometric reprojection (Garg et al., ECCV 2016)
# ─────────────────────────────────────────────────────────────────────────────
class PhotoL1Loss(nn.Module):
    # Reference: Garg et al. "Unsupervised CNN for single view depth estimation:
    # geometry to the rescue." ECCV 2016. arXiv:1603.04992.
    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pose=None, K=None, has_K=None):
        if rgb is None or rgb_next is None or pose is None or K is None:
            # Fallback: pure L1 between pred and GT depth (degenerate case)
            l = ((pred.clamp(EPS, MAX_DEPTH) - gt).abs() * mask).sum() / (mask.sum() + EPS)
            return l, {"total": l, "fallback": l}
        warped, vmask = warp_image(rgb_next, pred.clamp(EPS, MAX_DEPTH),
                                    pose, K, has_K=has_K)
        l = ((warped - rgb).abs() * vmask).mean()
        return l, {"total": l, "photo_l1": l}


# ─────────────────────────────────────────────────────────────────────────────
# 2. photo_ssim — SSIM-only photometric
# ─────────────────────────────────────────────────────────────────────────────
class PhotoSSIMLoss(nn.Module):
    # SSIM-only photometric reprojection loss.
    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pose=None, K=None, has_K=None):
        if rgb is None or rgb_next is None or pose is None or K is None:
            l = ssim_loss(pred.clamp(EPS, MAX_DEPTH) / MAX_DEPTH,
                          gt / MAX_DEPTH, mask)
            return l, {"total": l, "fallback": l}
        warped, vmask = warp_image(rgb_next, pred.clamp(EPS, MAX_DEPTH),
                                    pose, K, has_K=has_K)
        # Use per-channel SSIM averaged
        l = ssim_loss(warped, rgb, vmask)
        return l, {"total": l, "ssim": l}


# ─────────────────────────────────────────────────────────────────────────────
# 3. photo_ssim_l1 — alpha * SSIM + (1-alpha) * L1 (Godard 2017 / Monodepth)
# ─────────────────────────────────────────────────────────────────────────────
class PhotoSSIML1Loss(nn.Module):
    # Reference: Godard et al. "Unsupervised Monocular Depth Estimation with
    # Left-Right Consistency." CVPR 2017. The de-facto standard photometric
    # loss in MDE since 2017.
    def __init__(self, alpha: float = SSIM_ALPHA):
        super().__init__()
        self.alpha = alpha

    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pose=None, K=None, has_K=None):
        if rgb is None or rgb_next is None or pose is None or K is None:
            l = (((pred.clamp(EPS, MAX_DEPTH) - gt).abs()) * mask).sum() / (mask.sum() + EPS)
            return l, {"total": l, "fallback": l}
        warped, vmask = warp_image(rgb_next, pred.clamp(EPS, MAX_DEPTH),
                                    pose, K, has_K=has_K)
        residual = photometric_residual(warped, rgb, self.alpha)
        l = (residual * vmask).sum() / (vmask.sum() + EPS)
        return l, {"total": l, "photometric": l}


# ─────────────────────────────────────────────────────────────────────────────
# 4. edge_aware_smooth — Edge-aware smoothness only
# ─────────────────────────────────────────────────────────────────────────────
class EdgeAwareSmoothLoss(nn.Module):
    # Edge-aware disparity smoothness only — no photometric term.
    # Useful as a regulariser; alone it produces a degenerate constant-depth solution.
    # Reference: Heise et al. PM-Huber, ICCV 2013; Godard 2017.
    def __init__(self, decay: float = SMOOTHNESS_DECAY):
        super().__init__()
        self.decay = decay

    def forward(self, pred, gt, mask, rgb=None, **kw):
        if rgb is None:
            l = ((pred.clamp(EPS, MAX_DEPTH) - gt).abs() * mask).sum() / (mask.sum() + EPS)
            return l, {"total": l, "fallback": l}
        disp = 1.0 / pred.clamp(EPS, MAX_DEPTH)
        l = edge_aware_smoothness_map(disp, rgb, self.decay)
        return l, {"total": l, "smooth": l}


# ─────────────────────────────────────────────────────────────────────────────
# 5. min_reproj_photo — Min-reprojection + auto-mask (Monodepth2, ICCV 2019)
# ─────────────────────────────────────────────────────────────────────────────
class MinReprojPhotoLoss(nn.Module):
    # Reference: Godard et al. "Digging Into Self-Supervised Monocular Depth
    # Estimation." ICCV 2019 (Monodepth2). Two key innovations:
    #
    #   (a) Per-pixel MIN of forward + backward reprojection errors
    #       => handles occlusions / disocclusions correctly.
    #   (b) Auto-mask: drop pixels where the reprojection error is WORSE than
    #       the no-warp error (= identity comparison). Removes pixels that
    #       move with the camera (vehicles in front, static scenes).
    #
    # Practically, with adjacent +1 / -1 frames we can compute two reprojections.
    # Since this notebook receives only the +1 frame, we use:
    #   - Forward reproj error : photo(warp(I_{t+1}), I_t)
    #   - Identity error       : photo(I_{t+1}, I_t)             [no warp]
    # and mask out pixels where identity error is smaller than forward error.
    def __init__(self, alpha: float = SSIM_ALPHA,
                 auto_mask_eps: float = MIN_REPROJ_AUTO_MASK_THRESHOLD):
        super().__init__()
        self.alpha = alpha
        self.auto_mask_eps = auto_mask_eps

    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pose=None, K=None, has_K=None):
        if rgb is None or rgb_next is None or pose is None or K is None:
            l = ((pred.clamp(EPS, MAX_DEPTH) - gt).abs() * mask).sum() / (mask.sum() + EPS)
            return l, {"total": l, "fallback": l}
        warped, vmask = warp_image(rgb_next, pred.clamp(EPS, MAX_DEPTH), pose, K, has_K=has_K)
        reproj_err   = photometric_residual(warped, rgb, self.alpha)
        identity_err = photometric_residual(rgb_next, rgb, self.alpha)
        # Auto-mask: only keep pixels where reprojection BEATS identity
        auto_mask = (reproj_err < identity_err + self.auto_mask_eps).float()
        final_mask = vmask * auto_mask
        l = (reproj_err * final_mask).sum() / (final_mask.sum() + EPS)
        return l, {"total": l, "photo": l,
                   "auto_mask_frac": auto_mask.mean()}


# ─────────────────────────────────────────────────────────────────────────────
# 6. multi_scale_photo — 4-scale photometric reprojection (Monodepth2)
# ─────────────────────────────────────────────────────────────────────────────
class MultiScalePhotoLoss(nn.Module):
    # Compute photometric loss at 4 scales (full / 1/2 / 1/4 / 1/8). At each
    # scale the depth is upsampled to full resolution before warping (per the
    # corrected version in Monodepth2). Helps the network reason about fine
    # details and reduces texture-copy artefacts.
    def __init__(self, alpha: float = SSIM_ALPHA, n_scales: int = 4):
        super().__init__()
        self.alpha = alpha
        self.n_scales = n_scales
        self.base = MinReprojPhotoLoss(alpha=alpha)

    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pose=None, K=None, has_K=None):
        if rgb is None or rgb_next is None or pose is None or K is None:
            l = ((pred.clamp(EPS, MAX_DEPTH) - gt).abs() * mask).sum() / (mask.sum() + EPS)
            return l, {"total": l, "fallback": l}
        B, _, H, W = pred.shape
        total = 0.0
        for s in range(self.n_scales):
            scale = 2 ** s
            if scale == 1:
                pred_s, rgb_s, rgb_n_s = pred, rgb, rgb_next
                K_s = K
            else:
                pred_s = F.interpolate(pred, scale_factor=1/scale,
                                        mode="bilinear", align_corners=False)
                # Upsample back to full resolution (Monodepth2 fix)
                pred_s = F.interpolate(pred_s, size=(H, W),
                                        mode="bilinear", align_corners=False)
                rgb_s   = rgb
                rgb_n_s = rgb_next
                K_s     = K
            l_s, _ = self.base(pred_s, gt, mask, rgb=rgb_s, rgb_next=rgb_n_s,
                               pose=pose, K=K_s, has_K=has_K)
            total = total + l_s / self.n_scales
        return total, {"total": total}


# ─────────────────────────────────────────────────────────────────────────────
# 7. geometric_consistency — Forward-backward depth consistency (Bian 2019)
# ─────────────────────────────────────────────────────────────────────────────
class GeometricConsistencyLoss(nn.Module):
    # Reference: Bian et al. "Unsupervised Scale-Consistent Depth and Ego-Motion
    # Learning from Monocular Video." NeurIPS 2019 (SC-SfMLearner).
    #
    # Idea: the depth predicted at frame t+1, after warping back to frame t's
    # grid using the relative pose, should equal the depth predicted at frame t.
    # This is geometrically correct (unlike naive delta) and yields a
    # scale-consistent depth network.
    #
    #     D_proj(p)   = depth_pred(t+1) warped back to t-grid
    #     D_compute(p) = z-coordinate of (t-grid pixel) after applying pose
    #     L = | D_proj - D_compute | / (D_proj + D_compute)
    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pred_next=None, pose=None, K=None, has_K=None):
        if pred_next is None or pose is None or K is None:
            l = ((pred.clamp(EPS, MAX_DEPTH) - gt).abs() * mask).sum() / (mask.sum() + EPS)
            return l, {"total": l, "fallback": l}
        d_curr = pred.clamp(EPS, MAX_DEPTH)
        d_next = pred_next.clamp(EPS, MAX_DEPTH)

        # Project depth_next back to current frame's grid
        d_proj, vmask = warp_depth(d_next, d_curr, pose, K, has_K=has_K)

        # Compute expected depth = z-coordinate after transforming current-frame
        # 3D points by the pose
        B, _, H, W = d_curr.shape
        inv_K = _safe_inv_K(K)
        backproj = _get_backproj(H, W, d_curr.device)
        pts3d_curr = backproj(d_curr, inv_K)                         # [B,4,H*W]
        pts3d_next = torch.matmul(pose[:, :3, :], pts3d_curr)         # [B,3,H*W]
        d_compute  = pts3d_next[:, 2:3].view(B, 1, H, W).clamp(min=EPS)

        diff_depth = ((d_proj - d_compute).abs() /
                      (d_proj + d_compute + EPS))
        l = (diff_depth * vmask).sum() / (vmask.sum() + EPS)
        return l, {"total": l, "geom_cons": l}


# ─────────────────────────────────────────────────────────────────────────────
# 8. feature_metric — Feature-metric photometric loss (FeatDepth, ECCV 2020)
# ─────────────────────────────────────────────────────────────────────────────
class FeatureMetricLoss(nn.Module):
    # Reference: Shu et al. "Feature-metric Loss for Self-supervised Learning
    # of Depth and Egomotion." ECCV 2020.
    #
    # Instead of comparing raw pixel intensities, compare features extracted
    # from a shallow CNN (here: first 2 layers of ResNet18). Features encode
    # texture / structure rather than illumination, so the loss is robust to
    # lighting changes and provides better gradients in low-texture regions.
    #
    # We attach a small feature extractor lazily on first forward to keep the
    # loss self-contained (no global state).
    def __init__(self, alpha: float = SSIM_ALPHA):
        super().__init__()
        self.alpha = alpha
        self.fe = None

    def _build_extractor(self, device):
        # Try to load pretrained weights; fall back to random init if offline.
        try:
            try:
                weights = tvm.ResNet18_Weights.DEFAULT
            except Exception:
                weights = None
            rn = tvm.resnet18(weights=weights).to(device)
        except Exception:
            rn = tvm.resnet18(weights=None).to(device)
        for p in rn.parameters():
            p.requires_grad_(False)
        # Use up to layer1 (low-level texture features)
        self.fe = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool, rn.layer1).eval()

    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pose=None, K=None, has_K=None):
        if rgb is None or rgb_next is None or pose is None or K is None:
            l = ((pred.clamp(EPS, MAX_DEPTH) - gt).abs() * mask).sum() / (mask.sum() + EPS)
            return l, {"total": l, "fallback": l}
        if self.fe is None:
            self._build_extractor(pred.device)
        warped, vmask = warp_image(rgb_next, pred.clamp(EPS, MAX_DEPTH), pose, K, has_K=has_K)
        # Extract features (no_grad → static feature extractor)
        with torch.no_grad():
            f_target = self.fe(rgb)
            f_next   = self.fe(rgb_next)
        # We must warp the NEXT features too. Because warping happens at
        # full-res, we resize features to full res first, warp, then take L1.
        H, W = rgb.shape[-2:]
        f_target_up = F.interpolate(f_target, size=(H, W), mode="bilinear", align_corners=False)
        f_next_up   = F.interpolate(f_next,   size=(H, W), mode="bilinear", align_corners=False)
        f_warped, _ = warp_image(f_next_up, pred.clamp(EPS, MAX_DEPTH), pose, K, has_K = has_K)
        l_feat = ((f_warped - f_target_up).abs() * vmask).mean()
        # Add a small raw-pixel L1 to keep the loss well-grounded
        l_pix  = ((warped - rgb).abs() * vmask).mean()
        l = 0.5 * l_feat + 0.5 * l_pix
        return l, {"total": l, "feature": l_feat, "pixel": l_pix}


# ─────────────────────────────────────────────────────────────────────────────
# 9. flow_depth_consistency — Flow <-> depth+pose coupling (Yin & Shi 2018)
# ─────────────────────────────────────────────────────────────────────────────
class FlowDepthConsistencyLoss(nn.Module):
    # Reference: Yin & Shi. "GeoNet: Unsupervised Learning of Dense Depth,
    # Optical Flow and Camera Pose." CVPR 2018.
    #
    # Predicted flow from (depth + pose + K) should match a "static-scene flow"
    # also derived from depth+pose+K — i.e., the rigid-flow component should be
    # self-consistent. This is enforced by computing the pixel-displacement
    # field implied by warping and checking it is small / smooth.
    #
    # The simplest form (and what we use): compute the displacement field
    # induced by reprojection, and penalise inconsistency between forward and
    # backward displacements (cycle consistency).
    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pose=None, K=None, has_K=None):
        if rgb is None or rgb_next is None or pose is None or K is None:
            l = ((pred.clamp(EPS, MAX_DEPTH) - gt).abs() * mask).sum() / (mask.sum() + EPS)
            return l, {"total": l, "fallback": l}
        d_curr = pred.clamp(EPS, MAX_DEPTH)
        B, _, H, W = d_curr.shape
        inv_K = _safe_inv_K(K)
        backproj = _get_backproj(H, W, d_curr.device)
        project  = _get_project(H, W, d_curr.device)

        # Forward projection
        pts3d_curr = backproj(d_curr, inv_K)
        pix_in_next = project(pts3d_curr, K, pose)              # [B,H,W,2]
        # Pixel displacement field (in normalised [-1,1] coords)
        u, v = torch.meshgrid(
            torch.arange(0, W, dtype=torch.float32, device=d_curr.device),
            torch.arange(0, H, dtype=torch.float32, device=d_curr.device),
            indexing="xy",
        )
        pix0 = torch.stack([
            2.0 * u / (W - 1) - 1.0,
            2.0 * v / (H - 1) - 1.0,
        ], dim=-1).unsqueeze(0).expand(B, -1, -1, -1)            # [B,H,W,2]
        flow_fwd = pix_in_next - pix0                            # [B,H,W,2]

        # Photometric warping using this flow (sanity)
        warped, vmask = warp_image(rgb_next, d_curr, pose, K, has_K=has_K)
        photo = photometric_residual(warped, rgb, SSIM_ALPHA)
        # Encourage small, smooth flow where photometric error is small
        flow_mag = flow_fwd.permute(0, 3, 1, 2).pow(2).sum(dim=1, keepdim=True).sqrt()
        flow_smooth = edge_aware_smoothness_map(flow_mag, rgb)
        photo_w = (photo * vmask).sum() / (vmask.sum() + EPS)
        l = photo_w + 0.1 * flow_smooth
        return l, {"total": l, "photo": photo_w, "flow_smooth": flow_smooth}


# ─────────────────────────────────────────────────────────────────────────────
# 10. temporal_warp_consistency — Proper temporal depth consistency via warping
# ─────────────────────────────────────────────────────────────────────────────
class TemporalWarpConsistencyLoss(nn.Module):
    # The geometrically CORRECT version of "temporal depth consistency":
    # warp the depth predicted at frame t+1 back to frame t's grid using the
    # predicted pose, then compare it to the depth predicted at frame t.
    #
    # This replaces the naive "subtract two depth maps and expect mean=const"
    # heuristic that the supporting document critiques as geometrically wrong.
    #
    # Optionally also incorporates a small photometric term so the network has
    # a gradient even when temporal consistency is locally trivial.
    def __init__(self, photo_weight: float = 0.5):
        super().__init__()
        self.photo_weight = photo_weight

    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pred_next=None, pose=None, K=None, has_K = None):
        if pred_next is None or pose is None or K is None:
            l = ((pred.clamp(EPS, MAX_DEPTH) - gt).abs() * mask).sum() / (mask.sum() + EPS)
            return l, {"total": l, "fallback": l}
        d_curr = pred.clamp(EPS, MAX_DEPTH)
        d_next = pred_next.clamp(EPS, MAX_DEPTH)

        # Project depth_next back to current frame's grid
        d_proj, vmask = warp_depth(d_next, d_curr, pose, K, has_K=has_K)
        # Compute expected depth at current pixel after pose transform
        B, _, H, W = d_curr.shape
        inv_K = _safe_inv_K(K)
        backproj = _get_backproj(H, W, d_curr.device)
        pts3d_curr = backproj(d_curr, inv_K)
        pts3d_next_cam = torch.matmul(pose[:, :3, :], pts3d_curr)
        d_compute = pts3d_next_cam[:, 2:3].view(B, 1, H, W).clamp(min=EPS)

        # Static-pixel mask: ignore dynamic regions where d_proj differs largely
        rel_diff = (d_proj - d_compute).abs() / (d_proj + d_compute + EPS)
        static = (rel_diff < 0.5).float() * vmask
        l_tc = (rel_diff * static).sum() / (static.sum() + EPS)

        # Optional small photometric term to anchor gradient signal
        if self.photo_weight > 0 and rgb is not None and rgb_next is not None:
            warped, vm = warp_image(rgb_next, d_curr, pose, K, has_K = has_K)
            photo = photometric_residual(warped, rgb, SSIM_ALPHA)
            l_p = (photo * vm).sum() / (vm.sum() + EPS)
        else:
            l_p = torch.tensor(0.0, device=d_curr.device)

        l = l_tc + self.photo_weight * l_p
        return l, {"total": l, "temporal_consistency": l_tc, "photo": l_p}


# ─────────────────────────────────────────────────────────────────────────────
# 11. naive_temporal_delta — Heuristic "constant Delta-d" temporal regulariser
# ─────────────────────────────────────────────────────────────────────────────
class NaiveTemporalDeltaLoss(nn.Module):
    # The HEURISTIC the supporting document critiques as geometrically wrong:
    # "the mean depth change should be similar across tracked pixels."
    # We include it explicitly so the ablation can quantify HOW MUCH the
    # geometrically-correct `temporal_warp_consistency` improves upon it.
    #
    #   delta = | mean( depth_{t+1} - depth_t ) - ( depth_{t+1,p} - depth_{t,p} ) |
    # i.e. penalise each pixel's depth-delta deviating from the batch-mean delta.
    def __init__(self, threshold: float = TEMPORAL_WARP_THRESHOLD):
        super().__init__()
        self.threshold = threshold

    def forward(self, pred, gt, mask, pred_next=None, mask_next=None, **kw):
        if pred_next is None:
            l = ((pred.clamp(EPS, MAX_DEPTH) - gt).abs() * mask).sum() / (mask.sum() + EPS)
            return l, {"total": l, "fallback": l}
        p_c = pred.clamp(EPS, MAX_DEPTH)
        p_n = pred_next.clamp(EPS, MAX_DEPTH)
        delta = p_n - p_c
        if mask_next is None:
            mm = mask
        else:
            mm = mask * mask_next
        if mm.sum() < 1:
            return p_c.sum() * 0., {"total": p_c.sum() * 0.}
        mean_delta = (delta * mm).sum() / (mm.sum() + EPS)
        # Per-pixel penalty: deviation from mean
        diff = (delta - mean_delta).abs()
        l = (diff * mm).sum() / (mm.sum() + EPS)
        return l, {"total": l, "naive_delta": l,
                   "batch_mean_delta": mean_delta}


# ─────────────────────────────────────────────────────────────────────────────
# 12. combined_monodepth2 — Full Monodepth2 recipe
# ─────────────────────────────────────────────────────────────────────────────
class CombinedMonodepth2Loss(nn.Module):
    # Reference: Godard et al. "Digging Into Self-Supervised Monocular Depth
    # Estimation." ICCV 2019.
    #   L = L_photo_min_reproj_auto_mask + lambda_s * L_smooth
    # where lambda_s ~ 1e-3 and the photometric loss uses min-reprojection
    # + auto-masking + multi-scale.
    def __init__(self, w_photo: float = 1.0, w_smooth: float = 1e-3,
                 alpha: float = SSIM_ALPHA, n_scales: int = 4):
        super().__init__()
        self.w_photo  = w_photo
        self.w_smooth = w_smooth
        self.photo    = MultiScalePhotoLoss(alpha=alpha, n_scales=n_scales)
        self.smooth   = EdgeAwareSmoothLoss()

    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pose=None, K=None, has_K=None, **kw):
        l_p, _ = self.photo(pred, gt, mask, rgb=rgb, rgb_next=rgb_next,
                            pose=pose, K=K, has_K=has_K)
        l_s, _ = self.smooth(pred, gt, mask, rgb=rgb)
        l = self.w_photo * l_p + self.w_smooth * l_s
        return l, {"total": l, "photo": l_p, "smooth": l_s}


# ─────────────────────────────────────────────────────────────────────────────
# 13. combined_recommended — Document's full recommended pipeline (this work)
# ─────────────────────────────────────────────────────────────────────────────
class CombinedRecommendedLoss(nn.Module):
    # The full pipeline recommended by the accompanying document:
    #
    #   L =   w_photo    * L_min_reproj_photo   (Monodepth2)
    #       + w_smooth   * L_edge_aware_smooth  (Godard 2017)
    #       + w_temporal * L_temporal_warp      (proper, via warped depth)
    #       + w_anchor   * L_silog_sparse_GT    (sparse LiDAR anchor)
    #
    # When LiDAR GT is absent the anchor term degrades to zero.
    def __init__(self,
                 w_photo:    float = DEFAULT_W_PHOTO,
                 w_smooth:   float = DEFAULT_W_SMOOTH,
                 w_temporal: float = DEFAULT_W_TEMPORAL,
                 w_anchor:   float = DEFAULT_W_ANCHOR,
                 alpha:      float = SSIM_ALPHA,
                 lam_silog:  float = 0.85):
        super().__init__()
        self.w_photo    = w_photo
        self.w_smooth   = w_smooth
        self.w_temporal = w_temporal
        self.w_anchor   = w_anchor
        self.alpha      = alpha
        self.lam_silog  = lam_silog
        self.photo    = MinReprojPhotoLoss(alpha=alpha)
        self.smooth   = EdgeAwareSmoothLoss()
        self.temporal = TemporalWarpConsistencyLoss(photo_weight=0.0)

    def _silog(self, pred, gt, mask):
        p = pred.clamp(EPS, MAX_DEPTH)
        m = mask.squeeze(1).bool()
        if not m.any():
            return p.sum() * 0.0
        d = _safe_log(p.squeeze(1)[m]) - _safe_log(gt.squeeze(1)[m].clamp(EPS))
        return torch.sqrt((d**2).mean() - self.lam_silog * d.mean()**2 + EPS)

    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pred_next=None, pose=None, K=None, has_K=None, **kw):
        l_p, _ = self.photo(pred, gt, mask, rgb=rgb, rgb_next=rgb_next,
                            pose=pose, K=K, has_K=has_K)
        l_s, _ = self.smooth(pred, gt, mask, rgb=rgb)
        l_t, _ = self.temporal(pred, gt, mask, rgb=rgb, rgb_next=rgb_next,
                               pred_next=pred_next, pose=pose, K=K, has_K=has_K)
        if mask.sum() > 0:
            l_a = self._silog(pred, gt, mask)
        else:
            l_a = torch.tensor(0.0, device=pred.device)
        l = (self.w_photo    * l_p +
             self.w_smooth   * l_s +
             self.w_temporal * l_t +
             self.w_anchor   * l_a)
        return l, {"total": l, "photo": l_p, "smooth": l_s,
                   "temporal": l_t, "anchor": l_a}


# ─────────────────────────────────────────────────────────────────────────────
# Anchor baseline — pure sparse LiDAR SILog (supervised reference)
# ─────────────────────────────────────────────────────────────────────────────
class LiDARAnchorOnlyLoss(nn.Module):
    # Pure SILog on sparse LiDAR — supervised baseline included for comparison
    # so we can quantify how much SSL adds on top of pure GT supervision.
    def __init__(self, lam: float = 0.85):
        super().__init__()
        self.lam = lam

    def forward(self, pred, gt, mask, **kw):
        p = pred.clamp(EPS, MAX_DEPTH).squeeze(1)
        g = gt.squeeze(1).clamp(EPS)
        m = mask.squeeze(1).bool()
        if not m.any():
            l = p.sum() * 0.0
            return l, {"total": l}
        d = _safe_log(p[m]) - _safe_log(g[m])
        l = torch.sqrt((d**2).mean() - self.lam * d.mean()**2 + EPS)
        return l, {"total": l, "silog": l}


# =============================================================================
# Unified forward — normalises all signatures to (total_loss, dict)
# =============================================================================
NEED_POSE_LOSSES = {
    "photo_l1", "photo_ssim", "photo_ssim_l1",
    "min_reproj_photo", "multi_scale_photo",
    "geometric_consistency", "feature_metric",
    "flow_depth_consistency", "temporal_warp_consistency",
    "combined_monodepth2", "combined_recommended",
}
NEED_NEXT_DEPTH_LOSSES = {
    "geometric_consistency", "temporal_warp_consistency",
    "naive_temporal_delta", "combined_recommended",
}


def _unified_forward(criterion, pred, gt, mask,
                     rgb=None, rgb_next=None, pred_next=None, mask_next=None,
                     pose=None, K=None, has_K=None):
    # Route based on type. Every loss returns (total, dict_of_terms).
    if isinstance(criterion, (CombinedRecommendedLoss, CombinedMonodepth2Loss)):
        return criterion(pred, gt, mask,
                         rgb=rgb, rgb_next=rgb_next,
                         pred_next=pred_next, pose=pose, K=K, has_K=has_K)
    if isinstance(criterion, (TemporalWarpConsistencyLoss, GeometricConsistencyLoss)):
        return criterion(pred, gt, mask,
                         rgb=rgb, rgb_next=rgb_next,
                         pred_next=pred_next, pose=pose, K=K, has_K=has_K)
    if isinstance(criterion, NaiveTemporalDeltaLoss):
        return criterion(pred, gt, mask,
                         pred_next=pred_next, mask_next=mask_next)
    if isinstance(criterion, (PhotoL1Loss, PhotoSSIMLoss, PhotoSSIML1Loss,
                              MinReprojPhotoLoss, MultiScalePhotoLoss,
                              FeatureMetricLoss, FlowDepthConsistencyLoss)):
        return criterion(pred, gt, mask,
                         rgb=rgb, rgb_next=rgb_next,
                         pose=pose, K=K, has_K=has_K)
    if isinstance(criterion, EdgeAwareSmoothLoss):
        return criterion(pred, gt, mask, rgb=rgb)
    if isinstance(criterion, LiDARAnchorOnlyLoss):
        return criterion(pred, gt, mask)
    out = criterion(pred, gt, mask)
    if isinstance(out, tuple):
        return out
    return out, {"total": out}


# =============================================================================
# Loss registry
# =============================================================================
LOSS_REGISTRY: Dict[str, nn.Module] = {
    "photo_l1":                  PhotoL1Loss(),
    "photo_ssim":                PhotoSSIMLoss(),
    "photo_ssim_l1":             PhotoSSIML1Loss(),
    "edge_aware_smooth":         EdgeAwareSmoothLoss(),
    "min_reproj_photo":          MinReprojPhotoLoss(),
    "multi_scale_photo":         MultiScalePhotoLoss(),
    "geometric_consistency":     GeometricConsistencyLoss(),
    "feature_metric":            FeatureMetricLoss(),
    "flow_depth_consistency":    FlowDepthConsistencyLoss(),
    "temporal_warp_consistency": TemporalWarpConsistencyLoss(),
    "naive_temporal_delta":      NaiveTemporalDeltaLoss(),
    "combined_monodepth2":       CombinedMonodepth2Loss(),
    "combined_recommended":      CombinedRecommendedLoss(),
    # Anchor baseline
    "lidar_anchor_only":         LiDARAnchorOnlyLoss(),
}

# Pre-build FeatureMetricLoss extractor to avoid mid-training download hang
_feat_loss = LOSS_REGISTRY.get("feature_metric")
if _feat_loss is not None:
    _feat_loss._build_extractor(torch.device("cpu"))
    log.info("FeatureMetricLoss extractor pre-loaded on CPU.")

print(f"Loss registry loaded: {len(LOSS_REGISTRY)} loss variants")
for k in LOSS_REGISTRY:
    extra = ""
    if k in NEED_POSE_LOSSES:           extra += " [needs PoseNet]"
    if k in NEED_NEXT_DEPTH_LOSSES:     extra += " [needs pred_next]"
    print(f"  - {k}{extra}")


# =============================================================================
# GT-pose loss variants — use GPS/IMU pose from OXTS instead of predicted pose
# =============================================================================
# All three classes are thin wrappers that:
#   1. Accept an optional `gt_pose` kwarg (SE(3) from OXTS, [B, 4, 4]).
#   2. Prefer gt_pose over the PoseNet-predicted `pose` when gt_pose is not None.
#   3. Fall back to predicted `pose` transparently when OXTS data is unavailable.
# This means they work identically to their base classes on any dataset that
# does NOT provide gt_pose, ensuring full backward compatibility.
# =============================================================================

class GTPosePhotoReprojLoss(MinReprojPhotoLoss):
    """
    Min-reprojection photometric loss with GPS/IMU ground-truth relative pose.

    When accurate GT pose is available the photometric loss measures the true
    appearance discrepancy caused by *depth error alone*, not the combined error
    of depth + pose prediction. This is the fairest test of whether the depth
    model is learning geometrically correct structure.

    Reference: same as MinReprojPhotoLoss (Godard ICCV 2019) but pose source
    is the KITTI OXTS RT3003 GPS/IMU unit (cm-level position, 0.02deg orientation).
    """
    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pose=None, gt_pose=None, K=None, has_K=None, **kw):
        eff_pose = gt_pose if gt_pose is not None else pose
        return super().forward(pred, gt, mask,
                               rgb=rgb, rgb_next=rgb_next,
                               pose=eff_pose, K=K, has_K=has_K)


class GTPoseFlowDepthLoss(FlowDepthConsistencyLoss):
    """
    Flow-depth consistency loss with GPS/IMU ground-truth relative pose.

    With a predicted pose, the rigid-flow field contains both depth errors
    and pose errors. With GT pose, the rigid-flow field is determined entirely
    by the depth prediction — making this a much cleaner depth supervision signal.
    Dynamic objects (vehicles, pedestrians) now stand out clearly as outliers in
    the residual flow field, rather than being masked by pose noise.

    Reference: Yin & Shi, GeoNet, CVPR 2018 — GT-pose variant.
    """
    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pose=None, gt_pose=None, K=None, has_K=None, **kw):
        eff_pose = gt_pose if gt_pose is not None else pose
        return super().forward(pred, gt, mask,
                               rgb=rgb, rgb_next=rgb_next,
                               pose=eff_pose, K=K, has_K=has_K)


class GTPoseTemporalWarpLoss(TemporalWarpConsistencyLoss):
    """
    Temporal depth consistency via warped depth with GPS/IMU ground-truth pose.

    The warped-depth consistency loss (Bian 2019) is sensitive to pose noise
    because the warp operator applies the relative pose when projecting depth_next
    back to the current frame's grid. With noisy predicted pose, even perfectly
    consistent depth predictions look inconsistent. With GT pose from OXTS,
    the warp is geometrically correct and the residual measures true depth
    inconsistency rather than pose-induced warp error.

    This is the definitive test of whether the temporal consistency signal is
    genuinely useful or whether its benefit in the predicted-pose setting is
    partly an artefact of the pose net regularising itself.

    Reference: Bian et al., SC-SfMLearner, NeurIPS 2019 — GT-pose variant.
    """
    def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                pred_next=None, pose=None, gt_pose=None, K=None, has_K=None, **kw):
        eff_pose = gt_pose if gt_pose is not None else pose
        return super().forward(pred, gt, mask,
                               rgb=rgb, rgb_next=rgb_next,
                               pred_next=pred_next,
                               pose=eff_pose, K=K, has_K=has_K)


# =============================================================================
# Updated registry and routing (append — Python uses last binding)
# =============================================================================

NEED_GT_POSE_LOSSES = {
    "gt_photo_reproj",
    "gt_flow_depth_consistency",
    "gt_temporal_warp_consistency",
}

# Extend the existing registry with the three GT-pose variants
LOSS_REGISTRY.update({
    "gt_photo_reproj":              GTPosePhotoReprojLoss(),
    "gt_flow_depth_consistency":    GTPoseFlowDepthLoss(),
    "gt_temporal_warp_consistency": GTPoseTemporalWarpLoss(),
})


def _unified_forward(criterion, pred, gt, mask,
                     rgb=None, rgb_next=None, pred_next=None, mask_next=None,
                     pose=None, gt_pose=None, K=None, has_K=None):
    """
    Unified dispatcher for all loss modules (predicted-pose and GT-pose variants).

    Added in v2: gt_pose kwarg (GPS/IMU SE(3) [B,4,4]) threaded through to
    GT-pose losses.  Predicted-pose losses ignore it via **kw catch-all.
    """
    # ── GT-pose loss routing ──────────────────────────────────────────────────
    if isinstance(criterion, (GTPosePhotoReprojLoss, GTPoseFlowDepthLoss)):
        return criterion(pred, gt, mask,
                         rgb=rgb, rgb_next=rgb_next,
                         pose=pose, gt_pose=gt_pose, K=K, has_K=has_K)
    if isinstance(criterion, GTPoseTemporalWarpLoss):
        return criterion(pred, gt, mask,
                         rgb=rgb, rgb_next=rgb_next,
                         pred_next=pred_next,
                         pose=pose, gt_pose=gt_pose, K=K, has_K=has_K)

    # ── Existing predicted-pose routing (unchanged) ───────────────────────────
    if isinstance(criterion, (CombinedRecommendedLoss, CombinedMonodepth2Loss)):
        return criterion(pred, gt, mask,
                         rgb=rgb, rgb_next=rgb_next,
                         pred_next=pred_next, pose=pose, K=K, has_K=has_K)
    if isinstance(criterion, (TemporalWarpConsistencyLoss, GeometricConsistencyLoss)):
        return criterion(pred, gt, mask,
                         rgb=rgb, rgb_next=rgb_next,
                         pred_next=pred_next, pose=pose, K=K, has_K=has_K)
    if isinstance(criterion, NaiveTemporalDeltaLoss):
        return criterion(pred, gt, mask,
                         pred_next=pred_next, mask_next=mask_next)
    if isinstance(criterion, (PhotoL1Loss, PhotoSSIMLoss, PhotoSSIML1Loss,
                              MinReprojPhotoLoss, MultiScalePhotoLoss,
                              FeatureMetricLoss, FlowDepthConsistencyLoss)):
        return criterion(pred, gt, mask,
                         rgb=rgb, rgb_next=rgb_next,
                         pose=pose, K=K, has_K=has_K)
    if isinstance(criterion, EdgeAwareSmoothLoss):
        return criterion(pred, gt, mask, rgb=rgb)
    if isinstance(criterion, LiDARAnchorOnlyLoss):
        return criterion(pred, gt, mask)

    # ── Generic catch-all: works for _Combo, _GTCombo, and any future modules ─
    # Pass all kwargs; modules that don't accept them use **kw to absorb them.
    try:
        return criterion(pred, gt, mask,
                         rgb=rgb, rgb_next=rgb_next,
                         pred_next=pred_next, mask_next=mask_next,
                         pose=pose, gt_pose=gt_pose, K=K, has_K=has_K)
    except TypeError:
        out = criterion(pred, gt, mask)
        if isinstance(out, tuple):
            return out
        return out, {"total": out}


print(f"Loss registry updated: {len(LOSS_REGISTRY)} total loss variants "
      f"({len(NEED_GT_POSE_LOSSES)} GT-pose variants added)")
for k in NEED_GT_POSE_LOSSES:
    print(f"  + {k}  [uses OXTS GT pose; falls back to PoseNet if unavailable]")


16:20:06  INFO      FeatureMetricLoss extractor pre-loaded on CPU.


Loss registry loaded: 14 loss variants
  - photo_l1 [needs PoseNet]
  - photo_ssim [needs PoseNet]
  - photo_ssim_l1 [needs PoseNet]
  - edge_aware_smooth
  - min_reproj_photo [needs PoseNet]
  - multi_scale_photo [needs PoseNet]
  - geometric_consistency [needs PoseNet] [needs pred_next]
  - feature_metric [needs PoseNet]
  - flow_depth_consistency [needs PoseNet]
  - temporal_warp_consistency [needs PoseNet] [needs pred_next]
  - naive_temporal_delta [needs pred_next]
  - combined_monodepth2 [needs PoseNet]
  - combined_recommended [needs PoseNet] [needs pred_next]
  - lidar_anchor_only
Loss registry updated: 17 total loss variants (3 GT-pose variants added)
  + gt_flow_depth_consistency  [uses OXTS GT pose; falls back to PoseNet if unavailable]
  + gt_temporal_warp_consistency  [uses OXTS GT pose; falls back to PoseNet if unavailable]
  + gt_photo_reproj  [uses OXTS GT pose; falls back to PoseNet if unavailable]


## 5 · Metrics (4 scale-alignment strategies)


In [8]:
def eigen_crop_mask(shape: Tuple[int, int]) -> np.ndarray:
    h, w = shape[:2]
    mask = np.zeros((h, w), dtype=bool)
    top = 44; bot = min(top + 352, h)
    mask[top:bot, 0:w] = True
    return mask


def compute_depth_errors(gt: np.ndarray, pred: np.ndarray,
                         mask: np.ndarray) -> Dict[str, float]:
    gt_v   = gt[mask].astype(np.float64)
    pred_v = pred[mask].astype(np.float64)
    pred_v = np.maximum(pred_v, 1e-8)
    thresh   = np.maximum(gt_v / pred_v, pred_v / gt_v)
    rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2)))
    mae      = float(np.mean(np.abs(gt_v - pred_v)))
    abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v))
    sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / gt_v))
    rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2)))
    log10    = float(np.mean(np.abs(np.log10(gt_v) - np.log10(pred_v))))
    delta1   = float(np.mean(thresh < 1.25))
    delta2   = float(np.mean(thresh < 1.25 ** 2))
    delta3   = float(np.mean(thresh < 1.25 ** 3))
    mean_err = float(np.mean(gt_v - pred_v))
    n_pts    = int(mask.sum())
    return dict(n_pts=n_pts, rmse=rmse, mae=mae, abs_rel=abs_rel,
                sq_rel=sq_rel, rmse_log=rmse_log, log10=log10,
                delta1=delta1, delta2=delta2, delta3=delta3,
                mean_error=mean_err,
                lidar_mean=float(np.mean(gt_v)), pred_mean=float(np.mean(pred_v)))


def compute_all_scales(gt_depth: np.ndarray,
                       pred_raw: np.ndarray,
                       mask: np.ndarray) -> Dict[str, Dict]:
    # Compute metrics under 4 scale-alignment strategies.
    gt_v   = gt_depth[mask]
    pred_c = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)
    results = {}

    # Unscaled
    m0 = compute_depth_errors(gt_depth, pred_c, mask)
    results["unscaled"] = {"scale": 1.0, "shift": 0.0, **m0}

    # Median scale
    s_med = float(np.median(gt_v) / max(np.median(pred_c[mask]), EPS))
    pred_med = np.clip(pred_c * s_med, MIN_DEPTH, MAX_DEPTH)
    m1 = compute_depth_errors(gt_depth, pred_med, mask)
    results["median"] = {"scale": s_med, "shift": 0.0, **m1}

    # Mean scale
    s_mean = float(np.mean(gt_v) / max(np.mean(pred_c[mask]), EPS))
    pred_mean = np.clip(pred_c * s_mean, MIN_DEPTH, MAX_DEPTH)
    m2 = compute_depth_errors(gt_depth, pred_mean, mask)
    results["mean"] = {"scale": s_mean, "shift": 0.0, **m2}

    # Affine (OLS)
    pf = pred_c[mask].ravel()
    A  = np.vstack([pf, np.ones_like(pf)]).T
    params, _, _, _ = np.linalg.lstsq(A, gt_v.ravel(), rcond=None)
    s_aff = float(params[0]); sh_aff = float(params[1])
    pred_aff = np.clip(s_aff * pred_c + sh_aff, MIN_DEPTH, MAX_DEPTH)
    m3 = compute_depth_errors(gt_depth, pred_aff, mask)
    results["affine"] = {"scale": s_aff, "shift": sh_aff, **m3}

    return results


def true_rmse(df_sub: pd.DataFrame, rmse_col: str = "rmse") -> float:
    valid = df_sub[[rmse_col, "n_pts"]].dropna()
    if valid.empty or valid["n_pts"].sum() == 0:
        return float("nan")
    total_sse = (valid[rmse_col] ** 2 * valid["n_pts"]).sum()
    total_pts = valid["n_pts"].sum()
    return float(np.sqrt(total_sse / total_pts))


def aggregate_results(frame_rows: List[Dict], label: str) -> Dict[str, float]:
    df = pd.DataFrame(frame_rows)
    if df.empty:
        return {}
    row: Dict[str, float] = {"config": label}
    scale_variants = ["unscaled", "median", "mean", "affine"]
    for sv in scale_variants:
        for col in ["rmse", "mae", "abs_rel", "sq_rel", "rmse_log",
                    "log10", "delta1", "delta2", "delta3", "mean_error"]:
            full_col = f"{sv}_{col}"
            if full_col not in df.columns:
                continue
            if col == "rmse":
                n_col = f"{sv}_n_pts"
                if n_col in df.columns:
                    tmp = df[[full_col, n_col]].rename(
                        columns={full_col: "rmse", n_col: "n_pts"})
                    row[full_col] = true_rmse(tmp)
                else:
                    row[full_col] = float(df[full_col].mean())
            else:
                row[full_col] = float(df[full_col].dropna().mean())
    return row


def compute_torch_metrics(pred, gt, mask) -> Dict[str, float]:
    pred_c = pred.clamp(EPS, MAX_DEPTH)
    mask_b = mask.bool()
    if not mask_b.any():
        return {k: float("nan") for k in
                ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]}
    p = pred_c[mask_b].float()
    g = gt[mask_b].float().clamp(EPS)
    thresh = torch.maximum(g / p, p / g)
    return {
        "abs_rel":  float(((g - p).abs() / g).mean()),
        "sq_rel":   float(((g - p)**2 / g).mean()),
        "rmse":     float(torch.sqrt(((g - p)**2).mean())),
        "rmse_log": float(torch.sqrt((_safe_log(p) - _safe_log(g)).pow(2).mean())),
        "d1":       float((thresh < 1.25).float().mean()),
        "d2":       float((thresh < 1.25**2).float().mean()),
        "d3":       float((thresh < 1.25**3).float().mean()),
    }

print("Metric helpers ready.")


Metric helpers ready.


## 6 · Training & Validation Loops


In [9]:
def build_optimizer(params_iter, optim_mode: str = "adamw",
                    lr: float = LEARNING_RATE) -> torch.optim.Optimizer:
    # Build an optimizer over an iterable of (trainable) params.
    # Supported: adamw / adam / sgd.
    if optim_mode == "adam":
        return torch.optim.Adam(params_iter, lr=lr)
    if optim_mode == "sgd":
        return torch.optim.SGD(params_iter, lr=lr,
                               momentum=0.9, weight_decay=WEIGHT_DECAY,
                               nesterov=True)
    return torch.optim.AdamW(params_iter, lr=lr, weight_decay=WEIGHT_DECAY)


def build_scheduler(optimizer, steps_per_epoch: int,
                    sched_mode: str = "cosine_warmup",
                    lr: float = LEARNING_RATE):
    from torch.optim.lr_scheduler import (
        CosineAnnealingLR, LambdaLR, OneCycleLR, StepLR, SequentialLR)
    if sched_mode == "onecycle":
        return OneCycleLR(optimizer, max_lr=lr,
                          steps_per_epoch=steps_per_epoch,
                          epochs=ABLATION_EPOCHS, pct_start=0.1)
    if sched_mode == "step":
        return StepLR(optimizer, step_size=max(1, ABLATION_EPOCHS // 2), gamma=0.5)
    warm_steps = WARMUP_EPOCHS * steps_per_epoch
    main_steps = (ABLATION_EPOCHS - WARMUP_EPOCHS) * steps_per_epoch
    warmup = LambdaLR(optimizer,
                      lr_lambda=lambda s: min(1.0, (s + 1) / max(warm_steps, 1)))
    cosine = CosineAnnealingLR(optimizer, T_max=max(main_steps, 1),
                               eta_min=lr * 0.01)
    return SequentialLR(optimizer, [warmup, cosine], milestones=[warm_steps])


def train_one_epoch(model, pose_net, loader, criterion, optimizer, scaler,
                    scheduler, device, epoch, global_step,
                    sched_mode: str = "cosine_warmup",
                    needs_pose: bool = True,
                    needs_pred_next: bool = False,
                    gpu_duty_cycle: float = 0.50) -> Tuple[int, Dict]:
    model.model.train()
    if pose_net is not None and needs_pose:
        pose_net.train()
    running: Dict[str, float] = {}
    optimizer.zero_grad(set_to_none=True)

    for bi, batch in enumerate(loader):
        # ── Progress logging ─────────────────────────────────────────────
        if bi % 50 == 0:
            log.info("    batch %d/%d", bi, len(loader))
        # ── 1. Start timing BEFORE any GPU work ──────────────────────────────
        t_compute_start = time.perf_counter()

        rgb   = batch["rgb"].to(device, non_blocking=True)
        rgb_r = batch.get("rgb_raw",
                          rgb).to(device, non_blocking=True)
        gt    = batch["depth_gt"].to(device, non_blocking=True)
        mask  = batch["valid_mask"].to(device, non_blocking=True)
        K     = batch["K"].to(device, non_blocking=True)
        has_K = batch["has_K"].to(device, non_blocking=True)

        rgb_next   = batch.get("next_rgb")
        rgb_next_r = batch.get("next_rgb_raw")
        gt_next    = batch.get("next_depth_gt")
        mask_next  = batch.get("next_valid_mask")
        K_next     = batch.get("next_K")
        has_K_next = batch.get("next_has_K")

        rgb_next_dev      = rgb_next.to(device, non_blocking=True) if rgb_next is not None else None
        rgb_next_r_dev    = rgb_next_r.to(device, non_blocking=True) if rgb_next_r is not None else rgb_next_dev
        gt_next_dev       = gt_next.to(device, non_blocking=True) if gt_next is not None else None
        mask_next_dev     = mask_next.to(device, non_blocking=True) if mask_next is not None else None
        K_next_dev        = K_next.to(device, non_blocking=True) if K_next is not None else K
        has_K_next_dev    = has_K_next.to(device, non_blocking=True) if has_K_next is not None else has_K

        # GT pose from OXTS (present only when using KITTISequentialOXTSDataset)
        _gt_pose_raw   = batch.get("gt_pose")
        _has_gt_pose   = batch.get("has_gt_pose")
        gt_pose_dev    = None
        if _gt_pose_raw is not None and _has_gt_pose is not None:
            _hgp = _has_gt_pose.to(device, non_blocking=True)
            if _hgp.all():
                gt_pose_dev = _gt_pose_raw.to(device, non_blocking=True)

        with autocast(device_type="cuda",
                      enabled=MIXED_PREC and torch.cuda.is_available()):
            pred = forward_batch(model, rgb, K, has_K)

            pose = None
            pred_next = None
            if needs_pose and pose_net is not None and rgb_next_dev is not None:
                pose6 = pose_net(rgb, rgb_next_dev)              # [B, 6]
                pose  = pose_to_se3(pose6)                       # [B, 4, 4]
            if needs_pred_next and rgb_next_dev is not None:
                pred_next = forward_batch(model, rgb_next_dev, K_next_dev, has_K_next_dev)

            total, losses = _unified_forward(
                criterion, pred, gt, mask,
                rgb=rgb_r, rgb_next=rgb_next_r_dev,
                pred_next=pred_next, mask_next=mask_next_dev,
                pose=pose, gt_pose=gt_pose_dev, K=K, has_K=has_K,
            )
            total_acc = total / ACCUM_STEPS

        

        scaler.scale(total_acc).backward()

        if (bi + 1) % ACCUM_STEPS == 0:
            if MAX_GRAD_NORM > 0:
                scaler.unscale_(optimizer)
                trainable_params = [p for p in model.parameters() if p.requires_grad]
                if pose_net is not None and needs_pose:
                    trainable_params += list(pose_net.parameters())
                nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if scheduler and sched_mode in ("cosine_warmup", "onecycle"):
                scheduler.step()
            global_step += 1
            
        # ── 3. Sync THEN measure wall-time of compute ─────────────────────────
        # synchronize() blocks the CPU until all CUDA kernels are done,
        # so t_compute is the true GPU-active time for this batch.
        if gpu_duty_cycle < 1.0:
            torch.cuda.synchronize(device)
            t_compute = time.perf_counter() - t_compute_start

            # Sleep for long enough to hit the target duty cycle.
            # duty = compute / (compute + sleep)  →  sleep = compute * (1/duty - 1)
            t_sleep = t_compute * (1.0 / gpu_duty_cycle - 1.0)
            if t_sleep > 0:
                time.sleep(t_sleep)

        for k, v in losses.items():
            if torch.is_tensor(v):
                running[k] = running.get(k, 0.0) + float(v.detach())
            else:
                running[k] = running.get(k, 0.0) + float(v)

    if scheduler and sched_mode == "step":
        scheduler.step()

    avg = {k: v / max(len(loader), 1) for k, v in running.items()}
    return global_step, avg


@torch.no_grad()
def validate(model, pose_net, loader, criterion, device, epoch,
             needs_pose: bool = True,
             needs_pred_next: bool = False) -> Dict[str, float]:
    model.model.eval()
    if pose_net is not None:
        pose_net.eval()
    all_m: List[Dict] = []
    total_loss = 0.0
    n_batches = 0
    for batch in loader:
        rgb   = batch["rgb"].to(device, non_blocking=True)
        rgb_r = batch.get("rgb_raw", rgb).to(device, non_blocking=True)
        gt    = batch["depth_gt"].to(device, non_blocking=True)
        mask  = batch["valid_mask"].to(device, non_blocking=True)
        K     = batch["K"].to(device, non_blocking=True)
        has_K = batch["has_K"].to(device, non_blocking=True)

        rgb_next   = batch.get("next_rgb")
        rgb_next_r = batch.get("next_rgb_raw")
        gt_next    = batch.get("next_depth_gt")
        mask_next  = batch.get("next_valid_mask")
        K_next     = batch.get("next_K")
        has_K_next = batch.get("next_has_K")

        rgb_next_dev   = rgb_next.to(device, non_blocking=True) if rgb_next is not None else None
        rgb_next_r_dev = rgb_next_r.to(device, non_blocking=True) if rgb_next_r is not None else rgb_next_dev
        gt_next_dev    = gt_next.to(device, non_blocking=True) if gt_next is not None else None
        mask_next_dev  = mask_next.to(device, non_blocking=True) if mask_next is not None else None
        K_next_dev     = K_next.to(device, non_blocking=True) if K_next is not None else K
        has_K_next_dev = has_K_next.to(device, non_blocking=True) if has_K_next is not None else has_K

        # GT pose from OXTS
        _gt_pose_raw_v  = batch.get("gt_pose")
        _has_gt_pose_v  = batch.get("has_gt_pose")
        gt_pose_dev_val = None
        if _gt_pose_raw_v is not None and _has_gt_pose_v is not None:
            _hgp_v = _has_gt_pose_v.to(device, non_blocking=True)
            if _hgp_v.all():
                gt_pose_dev_val = _gt_pose_raw_v.to(device, non_blocking=True)

        with autocast(device_type="cuda",
                      enabled=MIXED_PREC and torch.cuda.is_available()):
            pred = forward_batch(model, rgb, K, has_K)
            pose = None
            pred_next = None
            if needs_pose and pose_net is not None and rgb_next_dev is not None:
                pose = pose_to_se3(pose_net(rgb, rgb_next_dev))
            if needs_pred_next and rgb_next_dev is not None:
                pred_next = forward_batch(model, rgb_next_dev, K_next_dev, has_K_next_dev)
            loss, _ = _unified_forward(
                criterion, pred, gt, mask,
                rgb=rgb_r, rgb_next=rgb_next_r_dev,
                pred_next=pred_next, mask_next=mask_next_dev,
                pose=pose, gt_pose=gt_pose_dev_val, K=K, has_K=has_K,
            )
        total_loss += float(loss)
        n_batches  += 1
        all_m.append(compute_torch_metrics(pred, gt, mask))

    def _avg(k):
        return float(np.nanmean([x[k] for x in all_m]))

    agg = {k: _avg(k) for k in ["abs_rel", "sq_rel", "rmse", "rmse_log",
                                 "d1", "d2", "d3"]}
    agg["val_loss"] = total_loss / max(n_batches, 1)
    return agg


print("Training / validation loops ready.")


Training / validation loops ready.


## 7 · Test-Set Evaluation (KITTI Eigen split)


In [10]:
def index_kitti_test(test_root: str) -> pd.DataFrame:
    rows = []
    pattern = os.path.join(test_root, "*", "*_sync",
                            "proj_depth", "groundtruth", "image_02", "*.png")
    gt_paths = glob.glob(pattern)
    if not gt_paths:
        pattern = os.path.join(test_root, "*", "*",
                               "proj_depth", "groundtruth", "image_02", "*.png")
        gt_paths = glob.glob(pattern)
    for gt_path in tqdm(gt_paths, desc="Indexing test split"):
        parts = gt_path.split(os.sep)
        try:
            gt_idx = parts.index("proj_depth")
            drive  = parts[gt_idx - 1]; date = parts[gt_idx - 2]; fname = parts[-1]
        except (ValueError, IndexError):
            continue
        rgb_path = os.path.join(test_root, date, drive, "image_02", "data", fname)
        if not os.path.exists(rgb_path):
            continue
        rows.append(dict(date=date, drive=drive, filename=fname,
                         gt_path=gt_path, rgb_path=rgb_path))
    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info("Test index: %d frames, %d drives", len(df), df["drive"].nunique())
    return df


def evaluate_model(model: nn.Module, test_df: pd.DataFrame,
                   device: torch.device,
                   config_label: str) -> List[Dict]:
    model.model.eval()
    frame_rows = []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df),
                       desc=f"Eval: {config_label}"):
        try:
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            gt_depth = _read_kitti_depth(row["gt_path"])
            crop_mask = eigen_crop_mask(gt_depth.shape)
            valid_mask = ((gt_depth > MIN_DEPTH) & (gt_depth < MAX_DEPTH) & crop_mask)
            if not np.any(valid_mask):
                continue
            pred_raw = run_da3_inference(model, image_np, device)
            scale_res = compute_all_scales(gt_depth, pred_raw, valid_mask)
            frame_row = {"config": config_label}
            for sv, res in scale_res.items():
                for k, v in res.items():
                    if k not in ("scale", "shift"):
                        frame_row[f"{sv}_{k}"] = v
            frame_rows.append(frame_row)
        except Exception as e:
            log.error("Error evaluating %s: %s", row["rgb_path"], e)
    return frame_rows

print("Evaluation helpers ready.")


Evaluation helpers ready.


## 8 · HPT-Enabled Ablation Runner


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info("Running on: %s", device)

_base_model = load_da3_model(device)
_base_state = copy.deepcopy(_base_model.state_dict())

def load_da3_model_fast(device):
    _base_model.load_state_dict(copy.deepcopy(_base_state))
    return _base_model


# ─── Temp-checkpoint path helpers ───────────────────────────────────────────
def _ckpt_path(ablation_label: str, hpt_tag: str) -> str:
    safe = hpt_tag.replace("/", "-").replace(" ", "_")
    return os.path.join(TEMP_CKPT_DIR, f"temp_best_{ablation_label}__{safe}.pt")

def _hpt_tag(hpt_cfg: Dict) -> str:
    return "__".join(f"{k}={v}" for k, v in sorted(hpt_cfg.items()))


def _loss_needs_pose(loss_name: str) -> bool:
    return loss_name in NEED_POSE_LOSSES

def _loss_needs_pred_next(loss_name: str) -> bool:
    return loss_name in NEED_NEXT_DEPTH_LOSSES


# ══════════════════════════════════════════════════════════════════════════════
# Core HPT trial — train + validate + early-stop + checkpoint
# ══════════════════════════════════════════════════════════════════════════════
def _train_one_hpt_trial(
    ablation_label: str,
    hpt_cfg:        Dict,
    loss_name:      str,
    arch_mode:      str,
    aug_mode:       str,
    optim_mode:     str,
    sched_mode:     str,
    device:         torch.device,
    train_loader:   DataLoader,
    val_loader:     DataLoader,
    custom_criterion: Optional[nn.Module] = None,
) -> Tuple[float, str, List[Dict]]:
    tag  = _hpt_tag(hpt_cfg)
    ckpt = _ckpt_path(ablation_label, tag)
    latest_ckpt = ckpt.replace("temp_best_", "temp_latest_")

    lr = float(hpt_cfg.get("lr", LEARNING_RATE))
    log.info("  [HPT trial] ablation=%s  tag=%s  lr=%.2e",
             ablation_label, tag, lr)

    # Fresh model
    model = load_da3_model_fast(device)
    _apply_arch(model, arch_mode)

    # Pose network (if needed by this loss)
    needs_pose      = _loss_needs_pose(loss_name)
    needs_pred_next = _loss_needs_pred_next(loss_name)
    pose_net = None
    if needs_pose:
        pose_net = PoseNet().to(device)

    # Build optimizer over depth + pose params
    depth_params = [p for p in model.parameters() if p.requires_grad]
    if pose_net is not None:
        pose_params = list(pose_net.parameters())
    else:
        pose_params = []
    all_trainable = depth_params + pose_params
    optimizer = build_optimizer(all_trainable, optim_mode, lr)

    # Criterion (use custom if supplied for combination / weight ablation)
    if custom_criterion is not None:
        criterion = copy.deepcopy(custom_criterion).to(device)
    else:
        criterion = copy.deepcopy(LOSS_REGISTRY[loss_name]).to(device)

    scheduler = build_scheduler(optimizer, len(train_loader), sched_mode, lr)
    scaler    = GradScaler("cuda" if torch.cuda.is_available() else "cpu",
                           enabled=MIXED_PREC)

    best_val_rmse = float("inf")
    no_improve    = 0
    global_step   = 0
    start_epoch   = 1
    history: List[Dict] = []

    # ── Resumability ─────────────────────────────────────────────────────
    if os.path.exists(latest_ckpt):
        log.info("  [RESUME] Resuming from %s", os.path.basename(latest_ckpt))
        try:
            checkpoint = torch.load(latest_ckpt, map_location=device,
                                     weights_only=False)
            model.model.load_state_dict(checkpoint["model_state"])
            if pose_net is not None and "pose_state" in checkpoint:
                pose_net.load_state_dict(checkpoint["pose_state"])
            optimizer.load_state_dict(checkpoint["optimizer_state"])
            if scheduler and checkpoint.get("scheduler_state"):
                scheduler.load_state_dict(checkpoint["scheduler_state"])
            scaler.load_state_dict(checkpoint["scaler_state"])
            start_epoch   = checkpoint["epoch"] + 1
            best_val_rmse = checkpoint["best_val_rmse"]
            no_improve    = checkpoint["no_improve"]
            global_step   = checkpoint["global_step"]
            history       = checkpoint.get("history", [])
            log.info("  [RESUME] From epoch %d  best_val_rmse=%.4f",
                     start_epoch, best_val_rmse)
        except Exception as e:
            log.warning("  [RESUME] Failed: %s — starting fresh.", e)
            best_val_rmse = float("inf"); no_improve = 0
            global_step = 0; start_epoch = 1; history = []

    # ── Training ─────────────────────────────────────────────────────────
    for epoch in range(start_epoch, ABLATION_EPOCHS + 1):
        t0 = time.time()
        global_step, train_losses = train_one_epoch(
            model, pose_net, train_loader, criterion, optimizer,
            scaler, scheduler, device, epoch, global_step, sched_mode,
            needs_pose=needs_pose, needs_pred_next=needs_pred_next, gpu_duty_cycle=gpu_duty_cycle)
        val_m = validate(model, pose_net, val_loader, criterion, device, epoch,
                         needs_pose=needs_pose, needs_pred_next=needs_pred_next)
        elapsed = time.time() - t0

        val_rmse = val_m["rmse"]
        improved = val_rmse < best_val_rmse
        marker   = "  IMPROVED" if improved else ""
        log.info("  Ep%02d/%02d  lr=%.2e  train=%.4f  val_loss=%.4f  "
                 "val_rmse=%.4f  d1=%.4f  (%.0fs)%s",
                 epoch, ABLATION_EPOCHS, lr,
                 train_losses.get("total", float("nan")),
                 val_m["val_loss"], val_rmse, val_m["d1"],
                 elapsed, marker)

        history.append({
            "config":     ablation_label,
            "hpt_tag":    tag,
            "epoch":      epoch,
            "lr":         lr,
            "train_loss": train_losses.get("total", float("nan")),
            **{f"val_{k}": v for k, v in val_m.items()},
        })

        if improved:
            best_val_rmse = val_rmse
            no_improve = 0
            save_blob = {
                "epoch":        epoch,
                "model_state":  model.model.state_dict(),
                "val_rmse":     val_rmse,
                "hpt_cfg":      hpt_cfg,
                "config_label": ablation_label,
                "loss_name":    loss_name,
                "arch_mode":    arch_mode,
            }
            if pose_net is not None:
                save_blob["pose_state"] = pose_net.state_dict()
            torch.save(save_blob, ckpt)
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                log.info("  Early stop @ epoch %d (no improvement for %d ep)",
                         epoch, EARLY_STOP_PATIENCE)
                latest_blob = {
                    "epoch":           epoch,
                    "model_state":     model.model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict() if scheduler else None,
                    "scaler_state":    scaler.state_dict(),
                    "best_val_rmse":   best_val_rmse,
                    "no_improve":      no_improve,
                    "global_step":     global_step,
                    "history":         history,
                }
                if pose_net is not None:
                    latest_blob["pose_state"] = pose_net.state_dict()
                torch.save(latest_blob, latest_ckpt)
                break

        # Always save latest
        latest_blob = {
            "epoch":           epoch,
            "model_state":     model.model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict() if scheduler else None,
            "scaler_state":    scaler.state_dict(),
            "best_val_rmse":   best_val_rmse,
            "no_improve":      no_improve,
            "global_step":     global_step,
            "history":         history,
        }
        if pose_net is not None:
            latest_blob["pose_state"] = pose_net.state_dict()
        torch.save(latest_blob, latest_ckpt)

        # Periodic checkpoint (every SAVE_EVERY_N_EPOCHS epochs)
        if SAVE_EVERY_N_EPOCHS > 0 and epoch % SAVE_EVERY_N_EPOCHS == 0:
            periodic_path = ckpt.replace("temp_best_", f"temp_ep{epoch:03d}_")
            periodic_blob = {
                "epoch":        epoch,
                "model_state":  model.model.state_dict(),
                "val_rmse":     val_rmse,
                "hpt_cfg":      hpt_cfg,
                "config_label": ablation_label,
                "loss_name":    loss_name,
                "arch_mode":    arch_mode,
            }
            if pose_net is not None:
                periodic_blob["pose_state"] = pose_net.state_dict()
            torch.save(periodic_blob, periodic_path)
            log.info("  [periodic ckpt] epoch %d -> %s",
                     epoch, os.path.basename(periodic_path))

    log.info("  Done.  best_val_rmse=%.4f  (ckpt: %s)",
             best_val_rmse, os.path.basename(ckpt))

    del model, criterion, optimizer, scaler
    if pose_net is not None:
        del pose_net
    torch.cuda.empty_cache()
    gc.collect()

    return best_val_rmse, ckpt, history


# ══════════════════════════════════════════════════════════════════════════════
# HPT sweep orchestrator for one ablation label
# ══════════════════════════════════════════════════════════════════════════════
def run_ablation_with_hpt(
    ablation_label:   str,
    hpt_grid:         List[Dict],
    loss_name:        str            = "combined_recommended",
    arch_mode:        str            = "decoder_only",
    aug_mode:         str            = "base",
    optim_mode:       str            = "adamw",
    sched_mode:       str            = "cosine_warmup",
    device:           torch.device   = None,
    train_loader:     DataLoader     = None,
    val_loader:       DataLoader     = None,
    test_df:          pd.DataFrame   = None,
    custom_criterion: Optional[nn.Module] = None,
) -> Dict[str, float]:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    log.info("")
    log.info("=" * 70)
    log.info("HPT SWEEP  ablation=%s", ablation_label)
    log.info("  loss=%s  arch=%s  aug=%s  optim=%s",
             loss_name, arch_mode, aug_mode, optim_mode)
    log.info("  %d trial(s) in grid x up to %d epochs each (patience=%d)",
             len(hpt_grid), ABLATION_EPOCHS, EARLY_STOP_PATIENCE)
    log.info("=" * 70)

    # Phase 1: train every HPT trial
    trial_results: List[Dict] = []
    all_history:   List[Dict] = []
    for hpt_cfg in hpt_grid:
        best_rmse, ckpt, history = _train_one_hpt_trial(
            ablation_label=ablation_label, hpt_cfg=hpt_cfg,
            loss_name=loss_name, arch_mode=arch_mode, aug_mode=aug_mode,
            optim_mode=optim_mode, sched_mode=sched_mode, device=device,
            train_loader=train_loader, val_loader=val_loader,
            custom_criterion=custom_criterion,
        )
        trial_results.append(dict(tag=_hpt_tag(hpt_cfg), hpt_cfg=hpt_cfg,
                                  best_val_rmse=best_rmse, ckpt=ckpt))
        all_history.extend(history)

    # Phase 2: model selection
    winner = min(trial_results, key=lambda x: x["best_val_rmse"])
    log.info("")
    log.info("HPT results for [%s]:", ablation_label)
    for t in sorted(trial_results, key=lambda x: x["best_val_rmse"]):
        marker = " <- WINNER" if t["tag"] == winner["tag"] else ""
        log.info("  %-30s  best_val_rmse=%.4f%s",
                 t["tag"], t["best_val_rmse"], marker)

    # Phase 3: test eval (ONCE)
    log.info("Loading winner ckpt -> test evaluation (ONCE) ...")
    model = load_da3_model_fast(device)
    _apply_arch(model, arch_mode)
    ckpt_data = torch.load(winner["ckpt"], map_location=device, weights_only=False)
    model.model.load_state_dict(ckpt_data["model_state"])
    log.info("Winner ckpt: epoch=%d  val_rmse=%.4f",
             ckpt_data["epoch"], ckpt_data["val_rmse"])

    frame_rows = evaluate_model(model, test_df, device, ablation_label)
    summary    = aggregate_results(frame_rows, ablation_label)
    summary.update({
        "loss_fn":         loss_name,
        "arch":            arch_mode,
        "aug":             aug_mode,
        "optim":           optim_mode,
        "sched":           sched_mode,
        "winner_hpt_tag":  winner["tag"],
        "winner_val_rmse": winner["best_val_rmse"],
        "winner_epoch":    int(ckpt_data["epoch"]),
        "n_hpt_trials":    len(hpt_grid),
        "n_frames":        len(frame_rows),
    })

    # Phase 4: persist
    header = not os.path.exists(SUMMARY_CSV)
    pd.DataFrame([summary]).to_csv(SUMMARY_CSV, mode="a", header=header, index=False)
    hist_df   = pd.DataFrame(all_history)
    hist_path = os.path.join(RESULTS_DIR, f"history_{ablation_label}.csv")
    hist_df.to_csv(hist_path, index=False)
    hpt_rows = [{
        "ablation":      ablation_label,
        "tag":           t["tag"],
        "best_val_rmse": t["best_val_rmse"],
        "is_winner":     t["tag"] == winner["tag"],
        **t["hpt_cfg"],
    } for t in trial_results]
    hpt_path = os.path.join(RESULTS_DIR, f"hpt_sweep_{ablation_label}.csv")
    pd.DataFrame(hpt_rows).to_csv(hpt_path, index=False)
    log.info("  CSVs written: summary, history, hpt_sweep")

    # Phase 5: cleanup
    for t in trial_results:
        for prefix in ["temp_best_", "temp_latest_"]:
            path = t["ckpt"].replace("temp_best_", prefix)
            try:
                os.remove(path)
            except FileNotFoundError:
                pass

    del model
    torch.cuda.empty_cache()
    gc.collect()
    log.info("  -- ablation=%s complete --", ablation_label)
    return summary


print("HPT-enabled ablation runner ready.")
print("  _train_one_hpt_trial   -> train + early-stop + temp ckpt")
print("  run_ablation_with_hpt  -> full sweep + selection + test ONCE + cleanup")


16:20:06  INFO      Running on: cuda


[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


16:20:07  INFO      Loading depth-anything/DA3METRIC-LARGE ...
16:20:07  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
16:20:07  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/depth-anything/DA3METRIC-LARGE/4010e39f3634a45bc60553321fb49fb760bd594e/config.json "HTTP/1.1 200 OK"


[INFO ] using MLP layer as FFN


16:20:10  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/model.safetensors "HTTP/1.1 302 Found"


HPT-enabled ablation runner ready.
  _train_one_hpt_trial   -> train + early-stop + temp ckpt
  run_ablation_with_hpt  -> full sweep + selection + test ONCE + cleanup


## 9 · Run All 6 Ablation Groups


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info("Running on: %s", device)

# ════════════════════════════════════════════════════════════════════════════════
# Build train / val split (sequential pairs)
# ════════════════════════════════════════════════════════════════════════════════
_full_train_aug = KITTISequentialDepthDataset(
    TRAIN_ROOT, split="train", aug_mode="base",
    max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED)
_full_train_val = KITTISequentialDepthDataset(
    TRAIN_ROOT, split="val", aug_mode="none",
    max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED)

assert len(_full_train_aug) == len(_full_train_val), (
    f"Dataset size mismatch: aug={len(_full_train_aug)} vs val={len(_full_train_val)}.")

n_total = len(_full_train_aug)
_rng    = np.random.RandomState(VAL_SPLIT_SEED)
all_idx = _rng.permutation(n_total)
n_val   = max(1, int(n_total * VAL_SPLIT_FRAC))
_val_idx = sorted(all_idx[:n_val].tolist())
_tr_idx  = sorted(all_idx[n_val:].tolist())

_train_subset_base = Subset(_full_train_aug, _tr_idx)
_val_subset        = Subset(_full_train_val, _val_idx)

_train_loader_base = DataLoader(
    _train_subset_base, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=True, prefetch_factor=2, collate_fn=collate_fn,
    timeout=120)
_val_loader = DataLoader(
    _val_subset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=2, collate_fn=collate_fn,
    timeout=120)

# Test set (Eigen split) — indexed once
test_df = index_kitti_test(TEST_ROOT)
if MAX_TEST > 0:
    test_df = test_df.head(MAX_TEST)

log.info("Dataset split summary:")
log.info("  TRAIN  : %6d pairs  (aug=base)", len(_train_subset_base))
log.info("  VAL    : %6d pairs  (aug=none)", len(_val_subset))
log.info("  TEST   : %6d frames (from TEST_ROOT, never seen during HPT)", len(test_df))

all_results: List[Dict] = []

# ── Resumability ─────────────────────────────────────────────────────────────
def get_completed_ablations() -> set:
    if not os.path.exists(SUMMARY_CSV):
        return set()
    try:
        df = pd.read_csv(SUMMARY_CSV)
        if "config" in df.columns:
            return set(df["config"].unique())
    except pd.errors.EmptyDataError:
        pass
    return set()

completed_ablations = get_completed_ablations()
if completed_ablations:
    log.info("[RESUME] %d ablation(s) already in SUMMARY_CSV — will skip:",
             len(completed_ablations))
    for _lbl in sorted(completed_ablations):
        log.info("  v skipping: %s", _lbl)


# ══════════════════════════════════════════════════════════════════════════════
# GROUP 1 - LOSS FUNCTION ABLATION (13 losses + 1 lidar anchor baseline)
# ══════════════════════════════════════════════════════════════════════════════
if RUN_LOSS_ABLATION:
    log.info("\n" + "-" * 70)
    log.info("GROUP 1: SELF-SUPERVISED LOSS FUNCTION ABLATION")
    log.info("  %d loss variants x %d LR candidates",
             len(LOSS_REGISTRY), len(LOSS_HPT_GRID))
    log.info("-" * 70)

    for loss_name in LOSS_REGISTRY.keys():
        ablation_label = f"loss_{loss_name}"
        if ablation_label in completed_ablations:
            log.info("  [SKIP] %s already complete", ablation_label)
            continue
        res = run_ablation_with_hpt(
            ablation_label=ablation_label,
            hpt_grid=LOSS_HPT_GRID,
            loss_name=loss_name,
            arch_mode="decoder_only",
            aug_mode="base",
            optim_mode="adamw",
            sched_mode="cosine_warmup",
            device=device,
            train_loader=_train_loader_base,
            val_loader=_val_loader,
            test_df=test_df,
        )
        all_results.append(res)


# ══════════════════════════════════════════════════════════════════════════════
# GROUP 2 - ARCHITECTURE ABLATION
# ══════════════════════════════════════════════════════════════════════════════
if RUN_ARCH_ABLATION:
    log.info("\n" + "-" * 70)
    log.info("GROUP 2: ARCHITECTURE ABLATION")
    log.info("  3 arch variants x %d LR candidates", len(ARCH_HPT_GRID))
    log.info("-" * 70)

    for arch_mode in ["decoder_only", "last2_blocks", "full"]:
        ablation_label = f"arch_{arch_mode}"
        if ablation_label in completed_ablations:
            log.info("  [SKIP] %s already complete", ablation_label)
            continue
        res = run_ablation_with_hpt(
            ablation_label=ablation_label,
            hpt_grid=ARCH_HPT_GRID,
            loss_name="combined_recommended",
            arch_mode=arch_mode,
            aug_mode="base",
            optim_mode="adamw",
            sched_mode="cosine_warmup",
            device=device,
            train_loader=_train_loader_base,
            val_loader=_val_loader,
            test_df=test_df,
        )
        all_results.append(res)


# ══════════════════════════════════════════════════════════════════════════════
# GROUP 3 - AUGMENTATION ABLATION
# ══════════════════════════════════════════════════════════════════════════════
if RUN_AUG_ABLATION:
    log.info("\n" + "-" * 70)
    log.info("GROUP 3: AUGMENTATION ABLATION (paired-safe)")
    log.info("  4 aug modes x %d LR candidates", len(AUG_HPT_GRID))
    log.info("-" * 70)

    for aug_mode in ["none", "flip", "base", "full"]:
        ablation_label = f"aug_{aug_mode}"
        if ablation_label in completed_ablations:
            log.info("  [SKIP] %s already complete", ablation_label)
            continue
        _full_aug_variant = KITTISequentialDepthDataset(
            TRAIN_ROOT, split="train", aug_mode=aug_mode,
            max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED)
        _aug_train_subset = Subset(_full_aug_variant, _tr_idx)
        _aug_loader = DataLoader(
            _aug_train_subset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
            persistent_workers=True, prefetch_factor=2, collate_fn=collate_fn,
            timeout=120)
        res = run_ablation_with_hpt(
            ablation_label=ablation_label,
            hpt_grid=AUG_HPT_GRID,
            loss_name="combined_recommended",
            arch_mode="decoder_only",
            aug_mode=aug_mode,
            optim_mode="adamw",
            sched_mode="cosine_warmup",
            device=device,
            train_loader=_aug_loader,
            val_loader=_val_loader,
            test_df=test_df,
        )
        all_results.append(res)


# ══════════════════════════════════════════════════════════════════════════════
# GROUP 4 - OPTIMIZER / SCHEDULER ABLATION
# ══════════════════════════════════════════════════════════════════════════════
if RUN_OPTIM_ABLATION:
    log.info("\n" + "-" * 70)
    log.info("GROUP 4: OPTIMIZER / SCHEDULER ABLATION")
    log.info("  5 optim/sched combos x %d LR candidates", len(OPTIM_HPT_GRID))
    log.info("-" * 70)

    optim_sched_combos = [
        ("adamw", "cosine_warmup"),
        ("adam",  "cosine_warmup"),
        ("sgd",   "cosine_warmup"),
        ("adamw", "onecycle"),
        ("adamw", "step"),
    ]
    for optim_mode, sched_mode in optim_sched_combos:
        ablation_label = f"optim_{optim_mode}_{sched_mode}"
        if ablation_label in completed_ablations:
            log.info("  [SKIP] %s already complete", ablation_label)
            continue
        res = run_ablation_with_hpt(
            ablation_label=ablation_label,
            hpt_grid=OPTIM_HPT_GRID,
            loss_name="combined_recommended",
            arch_mode="decoder_only",
            aug_mode="base",
            optim_mode=optim_mode,
            sched_mode=sched_mode,
            device=device,
            train_loader=_train_loader_base,
            val_loader=_val_loader,
            test_df=test_df,
        )
        all_results.append(res)


# ══════════════════════════════════════════════════════════════════════════════
# GROUP 5 - LOSS COMBINATION ABLATION (7 cumulative recipes)
#
# A: photo                                          (Garg 2016 / Godard 2017)
# B: photo + smooth                                 (+ regularisation)
# C: photo + smooth + auto-mask                     (Monodepth2 part 1)
# D: photo + smooth + auto-mask + min-reproj        (= combined_monodepth2)
# E: D + temporal_warp_consistency                  (proper temporal term)
# F: D + temporal_warp + lidar_anchor               (hybrid)
# G: D + temporal_warp + lidar_anchor + extra_smooth(= combined_recommended)
# ══════════════════════════════════════════════════════════════════════════════
if RUN_COMBINATION_ABLATION:
    log.info("\n" + "-" * 70)
    log.info("GROUP 5: LOSS COMBINATION ABLATION (cumulative recipes)")
    log.info("-" * 70)

    class _Combo(nn.Module):
        # Generic configurable combiner used for cumulative-recipe ablation.
        def __init__(self, terms: Dict[str, float], lam_silog: float = 0.85):
            super().__init__()
            self.terms = terms
            self.lam_silog = lam_silog
            self.photo_min   = MinReprojPhotoLoss()
            self.photo_basic = PhotoSSIML1Loss()
            self.smooth      = EdgeAwareSmoothLoss()
            self.temporal    = TemporalWarpConsistencyLoss(photo_weight=0.0)
            self.use_min_reproj = "min_reproj" in self.terms

        def _silog(self, pred, gt, mask):
            p = pred.clamp(EPS, MAX_DEPTH)
            m = mask.squeeze(1).bool()
            if not m.any():
                return p.sum() * 0.0
            d = _safe_log(p.squeeze(1)[m]) - _safe_log(gt.squeeze(1)[m].clamp(EPS))
            return torch.sqrt((d**2).mean() - self.lam_silog * d.mean()**2 + EPS)

        def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                    pred_next=None, pose=None, K=None, has_K=None, **kw):
            losses = {}
            total = torch.tensor(0.0, device=pred.device)
            if "photo" in self.terms:
                photo_module = self.photo_min if self.use_min_reproj else self.photo_basic
                l_p, _ = photo_module(pred, gt, mask, rgb=rgb, rgb_next=rgb_next,
                                       pose=pose, K=K, has_K=has_K)
                total = total + self.terms["photo"] * l_p
                losses["photo"] = l_p
            if "smooth" in self.terms:
                l_s, _ = self.smooth(pred, gt, mask, rgb=rgb)
                total = total + self.terms["smooth"] * l_s
                losses["smooth"] = l_s
            if "temporal" in self.terms:
                l_t, _ = self.temporal(pred, gt, mask, rgb=rgb, rgb_next=rgb_next,
                                       pred_next=pred_next, pose=pose, K=K, has_K=has_K)
                total = total + self.terms["temporal"] * l_t
                losses["temporal"] = l_t
            if "anchor" in self.terms:
                l_a = self._silog(pred, gt, mask)
                total = total + self.terms["anchor"] * l_a
                losses["anchor"] = l_a
            losses["total"] = total
            return total, losses

    combo_specs = [
        ("A_photo_only",            {"photo": 1.0}),
        ("B_photo_smooth",          {"photo": 1.0, "smooth": 1e-3}),
        ("C_photo_smooth_automask", {"photo": 1.0, "smooth": 1e-3, "min_reproj": 1.0}),
        ("D_monodepth2_recipe",     {"photo": 1.0, "smooth": 1e-3, "min_reproj": 1.0}),
        ("E_D_plus_temporal",       {"photo": 1.0, "smooth": 1e-3, "min_reproj": 1.0,
                                      "temporal": 0.1}),
        ("F_E_plus_anchor",         {"photo": 1.0, "smooth": 1e-3, "min_reproj": 1.0,
                                      "temporal": 0.1, "anchor": 0.5}),
        ("G_recommended_full",      {"photo": 1.0, "smooth": 1e-3, "min_reproj": 1.0,
                                      "temporal": 0.1, "anchor": 0.5}),
    ]

    for combo_id, terms in combo_specs:
        ablation_label = f"combo_{combo_id}"
        if ablation_label in completed_ablations:
            log.info("  [SKIP] %s already complete", ablation_label)
            continue
        log.info("Combo %s -> terms %s", combo_id, terms)
        criterion = _Combo(terms)
        res = run_ablation_with_hpt(
            ablation_label=ablation_label,
            hpt_grid=COMBO_HPT_GRID,
            loss_name="combined_recommended",   # forces pose+pred_next routing
            arch_mode="decoder_only",
            aug_mode="base",
            optim_mode="adamw",
            sched_mode="cosine_warmup",
            device=device,
            train_loader=_train_loader_base,
            val_loader=_val_loader,
            test_df=test_df,
            custom_criterion=criterion,
        )
        all_results.append(res)


# ══════════════════════════════════════════════════════════════════════════════
# GROUP 6 - LOSS WEIGHT SWEEP (around the recommended recipe)
# ══════════════════════════════════════════════════════════════════════════════
if RUN_WEIGHT_ABLATION:
    log.info("\n" + "-" * 70)
    log.info("GROUP 6: LOSS WEIGHT HYPER-PARAMETER SWEEP")
    log.info("-" * 70)

    weight_specs = [
        # (label, w_photo, w_smooth, w_temporal, w_anchor)
        ("photo_high",      2.0,  1e-3, 0.10, 0.50),
        ("photo_low",       0.5,  1e-3, 0.10, 0.50),
        ("smooth_high",     1.0,  1e-2, 0.10, 0.50),   # 10x smoother
        ("smooth_low",      1.0,  1e-4, 0.10, 0.50),   # 10x less smooth
        ("temporal_high",   1.0,  1e-3, 0.50, 0.50),   # 5x stronger temporal
        ("temporal_off",    1.0,  1e-3, 0.00, 0.50),   # NO temporal
        ("anchor_high",     1.0,  1e-3, 0.10, 1.00),   # 2x stronger anchor
        ("anchor_off",      1.0,  1e-3, 0.10, 0.00),   # purely SSL
    ]

    for label, wp, ws, wt, wa in weight_specs:
        ablation_label = f"weight_{label}"
        if ablation_label in completed_ablations:
            log.info("  [SKIP] %s already complete", ablation_label)
            continue
        log.info("Weights %s -> photo=%.2f smooth=%.0e temporal=%.2f anchor=%.2f",
                 label, wp, ws, wt, wa)
        criterion = CombinedRecommendedLoss(
            w_photo=wp, w_smooth=ws, w_temporal=wt, w_anchor=wa)
        res = run_ablation_with_hpt(
            ablation_label=ablation_label,
            hpt_grid=WEIGHT_HPT_GRID,
            loss_name="combined_recommended",
            arch_mode="decoder_only",
            aug_mode="base",
            optim_mode="adamw",
            sched_mode="cosine_warmup",
            device=device,
            train_loader=_train_loader_base,
            val_loader=_val_loader,
            test_df=test_df,
            custom_criterion=criterion,
        )
        all_results.append(res)


# ══════════════════════════════════════════════════════════════════════════════
# Summary
# ══════════════════════════════════════════════════════════════════════════════
log.info("\n*** All HPT-ablation sweeps complete. ***")
summary_df = pd.read_csv(SUMMARY_CSV)
print(f"\nTotal winning configurations evaluated on Eigen test split: "
      f"{len(summary_df)}")
display_cols = [c for c in [
    "config", "unscaled_rmse", "median_rmse",
    "unscaled_abs_rel", "unscaled_delta1",
    "winner_hpt_tag", "winner_val_rmse", "winner_epoch", "n_hpt_trials",
] if c in summary_df.columns]
print(summary_df[display_cols].to_string(index=False))


# ══════════════════════════════════════════════════════════════════════════════
# GROUPS 5b — GT-POSE LOSS COMBINATION ABLATION (recipes H → L)
#
# These mirror the combination ablation above but use KITTISequentialOXTSDataset,
# which provides a ground-truth relative pose for each consecutive frame pair
# derived from the KITTI OXTS GPS/IMU unit.
#
# Three loss variants are tested in GT-pose mode, one for each of the
# self-supervised signal categories identified in the supporting document:
#
#   a. Photometric reprojection  -> GTPosePhotoReprojLoss   (min-reproj + auto-mask)
#   b. Flow-depth coupling       -> GTPoseFlowDepthLoss     (GeoNet-style rigid flow)
#   c. Temporal warp consistency -> GTPoseTemporalWarpLoss  (warped-depth cycle)
#
# Recipes (building cumulatively from top-tier GT-pose photometric):
#
#  H   gt_photo_only          Only GT-pose photometric         -> direct comparison to A
#  I   gt_photo_smooth        + edge-aware smoothness          -> comparison to B
#  J   gt_photo_full          + smooth + gt_temporal + anchor  -> comparison to G (GT vs pred)
#  K   gt_flow_depth          GT-pose flow-depth only          -> comparison to predicted flow
#  L   gt_flow_smooth_anchor  + smooth + anchor                -> best GT-flow recipe
# ══════════════════════════════════════════════════════════════════════════════

if RUN_COMBINATION_ABLATION:

    class _GTCombo(nn.Module):
        """
        Combination criterion using GT OXTS pose for photometric and temporal terms.
        Falls back to predicted pose (pose arg) when gt_pose is None.
        Mirrors _Combo but routes through the GT-pose loss modules.
        """
        def __init__(self, terms: Dict[str, float], lam_silog: float = 0.85):
            super().__init__()
            self.terms = terms
            self.lam_silog = lam_silog
            self.gt_photo    = GTPosePhotoReprojLoss()
            self.gt_flow     = GTPoseFlowDepthLoss()
            self.gt_temporal = GTPoseTemporalWarpLoss(photo_weight=0.0)
            self.smooth      = EdgeAwareSmoothLoss()

        def _silog(self, pred, gt, mask):
            p = pred.clamp(EPS, MAX_DEPTH)
            m = mask.squeeze(1).bool()
            if not m.any():
                return p.sum() * 0.0
            d = _safe_log(p.squeeze(1)[m]) - _safe_log(gt.squeeze(1)[m].clamp(EPS))
            return torch.sqrt((d**2).mean() - self.lam_silog * d.mean()**2 + EPS)

        def forward(self, pred, gt, mask, rgb=None, rgb_next=None,
                    pred_next=None, pose=None, gt_pose=None, K=None, has_K=None, **kw):
            total = torch.tensor(0.0, device=pred.device)
            losses = {}
            if "gt_photo" in self.terms:
                l, _ = self.gt_photo(pred, gt, mask, rgb=rgb, rgb_next=rgb_next,
                                      pose=pose, gt_pose=gt_pose, K=K, has_K=has_K)
                total = total + self.terms["gt_photo"] * l
                losses["gt_photo"] = l
            if "gt_flow" in self.terms:
                l, _ = self.gt_flow(pred, gt, mask, rgb=rgb, rgb_next=rgb_next,
                                     pose=pose, gt_pose=gt_pose, K=K, has_K=has_K)
                total = total + self.terms["gt_flow"] * l
                losses["gt_flow"] = l
            if "gt_temporal" in self.terms:
                l, _ = self.gt_temporal(pred, gt, mask, rgb=rgb, rgb_next=rgb_next,
                                         pred_next=pred_next, pose=pose,
                                         gt_pose=gt_pose, K=K, has_K=has_K)
                total = total + self.terms["gt_temporal"] * l
                losses["gt_temporal"] = l
            if "smooth" in self.terms:
                l, _ = self.smooth(pred, gt, mask, rgb=rgb)
                total = total + self.terms["smooth"] * l
                losses["smooth"] = l
            if "anchor" in self.terms:
                l = self._silog(pred, gt, mask)
                total = total + self.terms["anchor"] * l
                losses["anchor"] = l
            losses["total"] = total
            return total, losses

    # Build OXTS-backed loaders (same train/val index split as the base study)
    log.info("\n" + "-" * 70)
    log.info("GROUP 5b: GT-POSE COMBINATION ABLATION (OXTS GPS/IMU poses)")
    log.info("  5 recipes (H-L) x %d LR candidates", len(COMBO_HPT_GRID))
    log.info("  Dataset: KITTISequentialOXTSDataset (adds gt_pose to each batch)")
    log.info("-" * 70)

    try:
        _oxts_train_loader, _oxts_val_loader = make_oxts_loaders(
            TRAIN_ROOT, _tr_idx, _val_idx, aug_mode="base")
        _oxts_loaders_ok = True
        log.info("  OXTS loaders ready — train=%d pairs, val=%d pairs",
                 len(Subset(_full_train_aug, _tr_idx)),
                 len(Subset(_full_train_val, _val_idx)))
    except Exception as _e:
        log.warning("  OXTS loaders failed: %s — skipping GT-pose combinations.", _e)
        _oxts_loaders_ok = False

    if _oxts_loaders_ok:
        gt_combo_specs = [
            # (id, terms_dict)
            # ── photometric tier ──────────────────────────────────────────
            ("H_gt_photo_only",
             {"gt_photo": 1.0}),

            ("I_gt_photo_smooth",
             {"gt_photo": 1.0, "smooth": 1e-3}),

            ("J_gt_photo_full",
             {"gt_photo": 1.0, "smooth": 1e-3,
              "gt_temporal": 0.1, "anchor": 0.5}),

            # ── flow-depth tier ────────────────────────────────────────────
            ("K_gt_flow_depth_only",
             {"gt_flow": 1.0}),

            ("L_gt_flow_smooth_anchor",
             {"gt_flow": 1.0, "smooth": 1e-3, "anchor": 0.5}),
        ]

        for combo_id, terms in gt_combo_specs:
            ablation_label = f"combo_gt_{combo_id}"
            if ablation_label in completed_ablations:
                log.info("  [SKIP] %s already complete", ablation_label)
                continue
            log.info("GT combo %s -> terms %s", combo_id, terms)
            criterion = _GTCombo(terms)
            res = run_ablation_with_hpt(
                ablation_label=ablation_label,
                hpt_grid=COMBO_HPT_GRID,
                # Use combined_recommended routing so pose_net + pred_next are
                # available as fallback when GT pose is absent.
                loss_name="combined_recommended",
                arch_mode="decoder_only",
                aug_mode="base",
                optim_mode="adamw",
                sched_mode="cosine_warmup",
                device=device,
                train_loader=_oxts_train_loader,
                val_loader=_oxts_val_loader,
                test_df=test_df,
                custom_criterion=criterion,
            )
            all_results.append(res)


16:20:11  INFO      Running on: cuda
16:20:12  INFO      [KITTI-Seq-train] 50 pairs (aug=base)
16:20:12  INFO      [KITTI-Seq-val] 50 pairs (aug=none)


Indexing test split:   0%|          | 0/652 [00:00<?, ?it/s]

16:20:12  INFO      Test index: 652 frames, 28 drives
16:20:12  INFO      Dataset split summary:
16:20:12  INFO        TRAIN  :     48 pairs  (aug=base)
16:20:12  INFO        VAL    :      2 pairs  (aug=none)
16:20:12  INFO        TEST   :    652 frames (from TEST_ROOT, never seen during HPT)
16:20:12  INFO      
----------------------------------------------------------------------
16:20:12  INFO      GROUP 1: SELF-SUPERVISED LOSS FUNCTION ABLATION
16:20:12  INFO        17 loss variants x 3 LR candidates
16:20:12  INFO      ----------------------------------------------------------------------
16:20:12  INFO      
16:20:12  INFO      ======================================================================
16:20:12  INFO      HPT SWEEP  ablation=loss_photo_l1
16:20:12  INFO        loss=photo_l1  arch=decoder_only  aug=base  optim=adamw
16:20:12  INFO        3 trial(s) in grid x up to 10 epochs each (patience=5)
16:20:12  INFO      =========================================================

Eval: loss_photo_l1:   0%|          | 0/652 [00:00<?, ?it/s]

18:13:56  INFO        CSVs written: summary, history, hpt_sweep
18:13:57  INFO        -- ablation=loss_photo_l1 complete --
18:13:57  INFO      
18:13:57  INFO      ======================================================================
18:13:57  INFO      HPT SWEEP  ablation=loss_photo_ssim
18:13:57  INFO        loss=photo_ssim  arch=decoder_only  aug=base  optim=adamw
18:13:57  INFO        3 trial(s) in grid x up to 10 epochs each (patience=5)
18:13:57  INFO      ======================================================================
18:13:57  INFO        [HPT trial] ablation=loss_photo_ssim  tag=lr=0.0001  lr=1.00e-04
18:13:57  INFO        [arch=decoder_only] 29.8M params trainable
18:13:59  INFO          batch 0/12
18:16:21  INFO        Ep01/10  lr=1.00e-04  train=0.2823  val_loss=0.3377  val_rmse=2.8116  d1=0.9777  (143s)  IMPROVED
18:16:26  INFO        [periodic ckpt] epoch 1 -> temp_ep001_loss_photo_ssim__lr=0.0001.pt
18:16:26  INFO          batch 0/12
18:18:41  INFO        Ep02/1

Eval: loss_photo_ssim:   0%|          | 0/652 [00:00<?, ?it/s]

... (12 outputs left out)

09:53:36  INFO        CSVs written: summary, history, hpt_sweep
09:53:37  INFO        -- ablation=loss_feature_metric complete --
09:53:37  INFO      
09:53:37  INFO      ======================================================================
09:53:37  INFO      HPT SWEEP  ablation=loss_flow_depth_consistency
09:53:37  INFO        loss=flow_depth_consistency  arch=decoder_only  aug=base  optim=adamw
09:53:37  INFO        3 trial(s) in grid x up to 10 epochs each (patience=5)
09:53:37  INFO      ======================================================================
09:53:37  INFO        [HPT trial] ablation=loss_flow_depth_consistency  tag=lr=0.0001  lr=1.00e-04
09:53:37  INFO        [arch=decoder_only] 29.8M params trainable
09:53:38  INFO          batch 0/12
09:56:26  INFO        Ep01/10  lr=1.00e-04  train=0.2827  val_loss=0.3005  val_rmse=2.8116  d1=0.9777  (168s)  IMPROVED
09:56:30  INFO        [periodic ckpt] epoch 1 -> temp_ep001_loss_flow_depth_consistency__lr=0.0001.pt
09:56:31 

Eval: loss_flow_depth_consistency:   0%|          | 0/652 [00:00<?, ?it/s]

12:19:29  INFO        CSVs written: summary, history, hpt_sweep
12:19:31  INFO        -- ablation=loss_flow_depth_consistency complete --
12:19:31  INFO      
12:19:31  INFO      ======================================================================
12:19:31  INFO      HPT SWEEP  ablation=loss_temporal_warp_consistency
12:19:31  INFO        loss=temporal_warp_consistency  arch=decoder_only  aug=base  optim=adamw
12:19:31  INFO        3 trial(s) in grid x up to 10 epochs each (patience=5)
12:19:31  INFO      ======================================================================
12:19:31  INFO        [HPT trial] ablation=loss_temporal_warp_consistency  tag=lr=0.0001  lr=1.00e-04
12:19:31  INFO        [arch=decoder_only] 29.8M params trainable
12:19:32  INFO          batch 0/12
12:25:21  INFO        Ep01/10  lr=1.00e-04  train=0.1459  val_loss=0.1641  val_rmse=2.8116  d1=0.9777  (350s)  IMPROVED
12:25:26  INFO        [periodic ckpt] epoch 1 -> temp_ep001_loss_temporal_warp_consistency__lr

OutOfMemoryError: CUDA out of memory. Tried to allocate 920.00 MiB. GPU 0 has a total capacity of 31.73 GiB of which 907.81 MiB is free. Process 7746 has 313.95 MiB memory in use. Process 61687 has 18.80 GiB memory in use. Including non-PyTorch memory, this process has 11.72 GiB memory in use. Of the allocated memory 10.80 GiB is allocated by PyTorch, and 548.21 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 10 · Load Results


In [ ]:
# ── Load the summary CSV ─────────────────────────────────────────────────────
summary_df = pd.read_csv(SUMMARY_CSV)

SCALE_VARIANTS = ["unscaled", "median", "mean", "affine"]
METRIC_COLS    = ["rmse", "mae", "abs_rel", "sq_rel", "rmse_log",
                  "log10", "delta1", "delta2", "delta3", "mean_error"]

# ── Pretty per-group readout ────────────────────────────────────────────────
key_cols = ["config"] + [f"{sv}_rmse" for sv in SCALE_VARIANTS] + [
    "unscaled_abs_rel", "unscaled_delta1", "unscaled_delta2", "unscaled_rmse_log",
    "unscaled_mean_error"]

print("\n" + "=" * 90)
print("ABLATION SUMMARY — Pixel-Weighted True RMSE (m) and Key Metrics")
print("=" * 90)
disp = summary_df[[c for c in key_cols if c in summary_df.columns]].copy()
float_cols = [c for c in disp.columns if c != "config"]
for c in float_cols:
    disp[c] = disp[c].map(lambda x: f"{x:.4f}" if pd.notna(x) else "NaN")
print(disp.to_string(index=False))

# ── Group-level summaries ────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("GROUPED BY ABLATION DIMENSION:")
for prefix, name in [("loss_",   "LOSS"),
                     ("arch_",   "ARCHITECTURE"),
                     ("aug_",    "AUGMENTATION"),
                     ("optim_",  "OPTIMIZER/SCHED"),
                     ("combo_",  "LOSS COMBINATION"),
                     ("weight_", "LOSS WEIGHT")]:
    sub = summary_df[summary_df["config"].str.startswith(prefix)]
    if sub.empty:
        continue
    print(f"\n  {name} ablation ({len(sub)} configs):")
    for _, r in sub.iterrows():
        uRMSE = r.get("unscaled_rmse", float("nan"))
        mRMSE = r.get("median_rmse",   float("nan"))
        abrel = r.get("unscaled_abs_rel", float("nan"))
        d1    = r.get("unscaled_delta1", float("nan"))
        bias  = r.get("unscaled_mean_error", float("nan"))
        print(f"    {r['config']:<40}  "
              f"RMSE(uns)={uRMSE:.4f}  RMSE(med)={mRMSE:.4f}  "
              f"AbsRel={abrel:.4f}  d<1.25={d1:.4f}  Bias={bias:+.4f}")


## 11 · Visualisation (15 plots)


In [ ]:
# ── Colour palette ───────────────────────────────────────────────────────────
GROUP_COLORS = {
    "loss_":   "#2ecc71",
    "arch_":   "#3498db",
    "aug_":    "#e67e22",
    "optim_":  "#9b59b6",
    "combo_":  "#e74c3c",
    "weight_": "#1abc9c",
}
SCALE_COLORS = {
    "unscaled": "#e74c3c",
    "median":   "#27ae60",
    "mean":     "#2980b9",
    "affine":   "#8e44ad",
}

def _grp_color(label: str) -> str:
    for prefix, col in GROUP_COLORS.items():
        if label.startswith(prefix):
            return col
    return "#7f8c8d"

def _save(fig: plt.Figure, name: str, dpi: int = 140):
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, bbox_inches="tight", dpi=dpi)
    plt.close(fig)
    log.info("  v %s", name)


# ════════════════════════════════════════════════════════════════════════════
# Plot 00 — Grand Summary Table
# ════════════════════════════════════════════════════════════════════════════
def plot_summary_table(df: pd.DataFrame):
    cols_show = (["config"] +
                 [f"{sv}_{m}" for sv in SCALE_VARIANTS
                              for m in ["rmse", "abs_rel", "delta1", "mean_error"]])
    cols_show = [c for c in cols_show if c in df.columns]
    disp = df[cols_show].copy()
    n_rows = len(disp) + 1
    n_cols = len(cols_show)
    fig_h  = max(6, 0.45 * n_rows)
    fig, ax = plt.subplots(figsize=(min(32, 2.5 * n_cols), fig_h))
    ax.axis("off")
    headers = [c.replace("_", " ") for c in cols_show]
    cell_data = [[str(v) if isinstance(v, str) else f"{v:.4f}" if pd.notna(v) else "NaN"
                  for v in row] for _, row in disp.iterrows()]
    tbl = ax.table(cellText=cell_data, colLabels=headers,
                   cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(7.5); tbl.scale(1.0, 1.6)
    for j in range(n_cols):
        tbl[0, j].set_facecolor("#2c3e50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    for i, (_, row) in enumerate(disp.iterrows(), start=1):
        bg  = "#f0f4f8" if i % 2 == 1 else "white"
        col = _grp_color(str(row["config"]))
        for j in range(n_cols):
            tbl[i, j].set_facecolor(bg)
        tbl[i, 0].set_facecolor(col)
        tbl[i, 0].set_text_props(color="white", fontweight="bold")
    ax.set_title("SSL Ablation - Full Metric Matrix (pixel-weighted true RMSE, m)",
                 fontsize=12, fontweight="bold", pad=8)
    _save(fig, "00_summary_table.png", dpi=150)

plot_summary_table(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 01 — RMSE Comparison: all 4 scale variants, all configs
# ════════════════════════════════════════════════════════════════════════════
def plot_rmse_all_scales(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.2
    fig, ax = plt.subplots(figsize=(max(14, len(configs) * 0.9), 6))
    for i, (sv, col) in enumerate(SCALE_COLORS.items()):
        vals = [df.loc[df["config"] == c, f"{sv}_rmse"].values[0]
                if f"{sv}_rmse" in df.columns else float("nan")
                for c in configs]
        ax.bar(x + (i - 1.5) * w, vals, width=w, label=sv, color=col, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Pixel-Weighted True RMSE (m)"); ax.set_xlabel("Configuration")
    ax.set_title("RMSE Across All Scale Variants — SSL Ablation Configurations")
    ax.legend(title="Scale Variant"); ax.grid(axis="y", alpha=0.4)
    for prefix in ["arch_", "aug_", "optim_", "combo_", "weight_"]:
        idxs = [i for i, c in enumerate(configs) if c.startswith(prefix)]
        if idxs:
            ax.axvline(x=min(idxs) - 0.5, color="grey", linestyle="--", alpha=0.5)
    _save(fig, "01_rmse_all_scales.png")
plot_rmse_all_scales(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 02 — AbsRel / SqRel / RMSE_log / log10 (unscaled)
# ════════════════════════════════════════════════════════════════════════════
def plot_error_metrics(df: pd.DataFrame):
    metrics = ["unscaled_abs_rel", "unscaled_sq_rel", "unscaled_rmse_log", "unscaled_log10"]
    labels  = ["AbsRel", "SqRel", "RMSE_log", "log10"]
    configs = df["config"].tolist()
    x = np.arange(len(configs))
    fig, axes = plt.subplots(2, 2, figsize=(max(14, len(configs)), 10))
    axes = axes.flatten()
    for ax, metric, lbl in zip(axes, metrics, labels):
        vals = [df.loc[df["config"] == c, metric].values[0]
                if metric in df.columns else float("nan") for c in configs]
        colors = [_grp_color(c) for c in configs]
        bars = ax.bar(x, vals, color=colors, alpha=0.85,
                       edgecolor="white", linewidth=0.8)
        ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45,
                                              ha="right", fontsize=6)
        ax.set_ylabel(lbl + " (unscaled)"); ax.set_title(lbl)
        ax.grid(axis="y", alpha=0.4)
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=5)
    fig.suptitle("Error Metrics (Unscaled) — SSL Ablation",
                 fontsize=13, fontweight="bold")
    _save(fig, "02_error_metrics_unscaled.png")
plot_error_metrics(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 03 — Delta accuracy
# ════════════════════════════════════════════════════════════════════════════
def plot_delta_accuracy(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.25
    fig, ax = plt.subplots(figsize=(max(14, len(configs) * 0.9), 6))
    delta_info = [
        ("unscaled_delta1", "d<1.25",   "#e74c3c"),
        ("unscaled_delta2", "d<1.25^2", "#27ae60"),
        ("unscaled_delta3", "d<1.25^3", "#2980b9"),
    ]
    for i, (col, lbl, c) in enumerate(delta_info):
        vals = [df.loc[df["config"] == cf, col].values[0]
                if col in df.columns else float("nan") for cf in configs]
        ax.bar(x + (i-1)*w, vals, width=w, label=lbl, color=c, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45,
                                          ha="right", fontsize=7)
    ax.set_ylabel("Accuracy (higher = better)"); ax.set_xlabel("Configuration")
    ax.set_title("Threshold Accuracy (Unscaled) — SSL Ablation")
    ax.legend(); ax.grid(axis="y", alpha=0.4)
    ax.set_ylim([0, 1.05])
    _save(fig, "03_delta_accuracy.png")
plot_delta_accuracy(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 04 — Signed bias
# ════════════════════════════════════════════════════════════════════════════
def plot_bias(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.2
    fig, ax = plt.subplots(figsize=(max(14, len(configs)), 6))
    for i, (sv, col) in enumerate(SCALE_COLORS.items()):
        mc = f"{sv}_mean_error"
        vals = [df.loc[df["config"] == c, mc].values[0]
                if mc in df.columns else float("nan") for c in configs]
        ax.bar(x + (i-1.5)*w, vals, width=w, label=sv, color=col, alpha=0.8)
    ax.axhline(0, color="black", linewidth=1.0)
    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45,
                                          ha="right", fontsize=7)
    ax.set_ylabel("Mean Signed Error (LiDAR - Pred), m")
    ax.set_title("Depth Bias Across Scale Variants — SSL Ablation")
    ax.legend(title="Scale Variant"); ax.grid(axis="y", alpha=0.4)
    _save(fig, "04_bias_signed.png")
plot_bias(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 05 — Radar: Loss function ablation
# ════════════════════════════════════════════════════════════════════════════
def plot_radar(df: pd.DataFrame, prefix: str, fname: str, title_suffix: str):
    sub = df[df["config"].str.startswith(prefix)].copy()
    if sub.empty:
        return
    radar_metrics = ["unscaled_rmse", "unscaled_abs_rel", "unscaled_rmse_log",
                     "unscaled_delta1", "median_rmse", "affine_rmse"]
    radar_labels  = ["RMSE(uns)", "AbsRel(uns)", "RMSE_log(uns)",
                     "1-d<1.25(uns)", "RMSE(med)", "RMSE(aff)"]
    if "unscaled_delta1" in sub.columns:
        sub["_d1_inv"] = 1.0 - sub["unscaled_delta1"]
        radar_metrics[3] = "_d1_inv"
    metrics_avail = [m for m in radar_metrics if m in sub.columns]
    labels_avail  = [l for m, l in zip(radar_metrics, radar_labels)
                     if m in sub.columns]
    normed = sub[metrics_avail].copy()
    mn = normed.min(); mx = normed.max()
    normed = (normed - mn) / (mx - mn + 1e-9)
    N = len(metrics_avail)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(9, 9),
                           subplot_kw=dict(projection="polar"))
    colors = plt.cm.tab10(np.linspace(0, 1, len(sub)))
    for (_, row), color in zip(sub.iterrows(), colors):
        values = [normed.loc[row.name, m] for m in metrics_avail]
        values += values[:1]
        ax.plot(angles, values, color=color, linewidth=1.6,
                label=row["config"][:30])
        ax.fill(angles, values, color=color, alpha=0.10)
    ax.set_thetagrids(np.degrees(angles[:-1]), labels_avail, fontsize=9)
    ax.set_title(f"{title_suffix} — Radar (normalised, lower area = better)",
                 fontsize=11, fontweight="bold", pad=25)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=7)
    ax.set_ylim(0, 1)
    _save(fig, fname)

plot_radar(summary_df, "loss_",   "05_radar_loss.png",   "Loss Function Ablation")
plot_radar(summary_df, "arch_",   "06_radar_arch.png",   "Architecture Ablation")
plot_radar(summary_df, "combo_",  "06b_radar_combo.png", "Loss Combination Ablation")
plot_radar(summary_df, "weight_", "06c_radar_weight.png","Loss Weight Sweep")


# ════════════════════════════════════════════════════════════════════════════
# Plot 07 — Heatmap: RMSE x scale variant
# ════════════════════════════════════════════════════════════════════════════
def plot_heatmap_rmse(df: pd.DataFrame):
    rmse_cols = [f"{sv}_rmse" for sv in SCALE_VARIANTS]
    rmse_cols = [c for c in rmse_cols if c in df.columns]
    heat_df = df.set_index("config")[rmse_cols].copy()
    heat_df.columns = [c.replace("_rmse", "") for c in rmse_cols]
    fig, ax = plt.subplots(figsize=(max(8, len(SCALE_VARIANTS) * 2),
                                    max(6, len(heat_df) * 0.45)))
    sns.heatmap(heat_df.astype(float), annot=True, fmt=".3f",
                cmap="RdYlGn_r", linewidths=0.5, linecolor="grey",
                ax=ax, cbar_kws={"label": "True RMSE (m)"})
    ax.set_title("RMSE Heatmap — Configuration x Scale Variant",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Scale Variant"); ax.set_ylabel("Configuration")
    _save(fig, "07_heatmap_rmse.png")
plot_heatmap_rmse(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 08 — Heatmap: all metrics (unscaled, normalised)
# ════════════════════════════════════════════════════════════════════════════
def plot_heatmap_all_metrics(df: pd.DataFrame):
    unscaled_cols = [f"unscaled_{m}" for m in
                     ["rmse", "mae", "abs_rel", "sq_rel", "rmse_log", "log10",
                      "delta1", "delta2", "delta3"]]
    unscaled_cols = [c for c in unscaled_cols if c in df.columns]
    heat_df = df.set_index("config")[unscaled_cols].copy()
    heat_df.columns = [c.replace("unscaled_", "") for c in unscaled_cols]
    normed = (heat_df - heat_df.min()) / (heat_df.max() - heat_df.min() + 1e-9)
    # Invert delta cols (higher = better → lower = better in normalised view)
    for c in ["delta1", "delta2", "delta3"]:
        if c in normed.columns:
            normed[c] = 1.0 - normed[c]
    fig, ax = plt.subplots(figsize=(14, max(6, len(df) * 0.45)))
    sns.heatmap(normed, annot=heat_df.map(lambda x: f"{x:.3f}"),
                fmt="", cmap="RdYlGn_r", linewidths=0.5,
                ax=ax, cbar_kws={"label": "Normalised (0=best, 1=worst)"})
    ax.set_title("Metric Heatmap (Unscaled) — All SSL Configurations",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Metric"); ax.set_ylabel("Configuration")
    _save(fig, "08_heatmap_all_metrics.png")
plot_heatmap_all_metrics(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 09 — Learning curves (val RMSE per epoch)
# ════════════════════════════════════════════════════════════════════════════
def plot_learning_curves():
    hist_files = sorted(glob.glob(os.path.join(RESULTS_DIR, "history_*.csv")))
    if not hist_files:
        log.warning("No history CSVs found — skipping learning curves.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for hist_file in hist_files:
        try:
            h = pd.read_csv(hist_file)
            label = h["config"].iloc[0]
            color = _grp_color(label)
            if "val_rmse" in h.columns:
                axes[0].plot(h["epoch"], h["val_rmse"], label=label,
                             color=color, linewidth=1.2, alpha=0.7)
            if "val_val_loss" in h.columns or "train_loss" in h.columns:
                lc = "val_val_loss" if "val_val_loss" in h.columns else "train_loss"
                axes[1].plot(h["epoch"], h[lc], label=label,
                             color=color, linewidth=1.2, alpha=0.7)
        except Exception as e:
            log.warning("Could not load %s: %s", hist_file, e)
    for ax, title, ylabel in zip(axes,
            ["Validation RMSE (m) by Epoch", "Loss by Epoch"],
            ["Val RMSE (m)", "Loss"]):
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.legend(fontsize=5, ncol=3); ax.grid(alpha=0.4)
    fig.suptitle("Learning Curves — All SSL Ablation Configurations",
                 fontsize=13, fontweight="bold")
    _save(fig, "09_learning_curves.png")
plot_learning_curves()


# ════════════════════════════════════════════════════════════════════════════
# Plot 10 — Scale sensitivity scatter
# ════════════════════════════════════════════════════════════════════════════
def plot_scale_sensitivity(df: pd.DataFrame):
    if "unscaled_rmse" not in df.columns or "median_rmse" not in df.columns:
        return
    fig, ax = plt.subplots(figsize=(10, 8))
    for _, row in df.iterrows():
        color = _grp_color(str(row["config"]))
        ax.scatter(row["unscaled_rmse"], row["median_rmse"],
                   color=color, s=80, zorder=3,
                   edgecolors="white", linewidth=0.8)
        ax.annotate(
            str(row["config"]).replace("loss_", "").replace("arch_", "")
                              .replace("aug_", "").replace("optim_", "")
                              .replace("combo_", "").replace("weight_", ""),
            (row["unscaled_rmse"], row["median_rmse"]),
            fontsize=6, xytext=(5, 3), textcoords="offset points")
    vmin = min(df["unscaled_rmse"].min(), df["median_rmse"].min())
    vmax = max(df["unscaled_rmse"].max(), df["median_rmse"].max())
    ax.plot([vmin, vmax], [vmin, vmax], "k--", linewidth=1, alpha=0.5,
            label="y=x (no improvement)")
    ax.set_xlabel("RMSE (unscaled)"); ax.set_ylabel("RMSE (median-aligned)")
    ax.set_title("Scale Sensitivity: Unscaled vs Median-Aligned RMSE")
    ax.grid(alpha=0.4)
    patches = [mpatches.Patch(color=c, label=p.rstrip("_"))
               for p, c in GROUP_COLORS.items()]
    ax.legend(handles=patches, title="Ablation Group", loc="upper left", fontsize=8)
    _save(fig, "10_scale_sensitivity.png")
plot_scale_sensitivity(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 11 — RMSE ranking bar (top 15)
# ════════════════════════════════════════════════════════════════════════════
def plot_ranking_top(df: pd.DataFrame, k: int = 15):
    if "unscaled_rmse" not in df.columns:
        return
    sub = df.sort_values("unscaled_rmse").head(k).copy()
    sub = sub.iloc[::-1]
    colors = [_grp_color(c) for c in sub["config"]]
    fig, ax = plt.subplots(figsize=(10, max(6, 0.4 * len(sub))))
    bars = ax.barh(sub["config"], sub["unscaled_rmse"], color=colors,
                    edgecolor="white", linewidth=0.8)
    for bar, v in zip(bars, sub["unscaled_rmse"]):
        ax.text(v + 0.02, bar.get_y() + bar.get_height() / 2,
                f"{v:.4f}", va="center", fontsize=8)
    ax.set_xlabel("Pixel-Weighted RMSE (unscaled, m)")
    ax.set_title(f"Top-{k} SSL Configurations by Unscaled RMSE")
    ax.grid(axis="x", alpha=0.4)
    _save(fig, "11_top_ranking.png")
plot_ranking_top(summary_df, k=15)


# ════════════════════════════════════════════════════════════════════════════
# Plot 12 — Group-level box plot (RMSE by group)
# ════════════════════════════════════════════════════════════════════════════
def plot_group_box(df: pd.DataFrame):
    if "unscaled_rmse" not in df.columns:
        return
    df = df.copy()
    df["group"] = df["config"].apply(
        lambda c: next((p.rstrip("_") for p in GROUP_COLORS if c.startswith(p)),
                       "other"))
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=df, x="group", y="unscaled_rmse", ax=ax,
                palette={p.rstrip("_"): c for p, c in GROUP_COLORS.items()})
    sns.stripplot(data=df, x="group", y="unscaled_rmse", ax=ax,
                  color="black", size=4, alpha=0.7)
    ax.set_title("RMSE Distribution by Ablation Group")
    ax.set_ylabel("RMSE (unscaled, m)"); ax.set_xlabel("Group")
    ax.grid(axis="y", alpha=0.4)
    _save(fig, "12_group_box.png")
plot_group_box(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 13 — Test RMSE vs winning val RMSE scatter
# ════════════════════════════════════════════════════════════════════════════
def plot_val_vs_test_rmse(df: pd.DataFrame):
    if "winner_val_rmse" not in df.columns or "unscaled_rmse" not in df.columns:
        return
    sub = df.dropna(subset=["winner_val_rmse", "unscaled_rmse"]).copy()
    if sub.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 8))
    for _, row in sub.iterrows():
        color = _grp_color(str(row["config"]))
        ax.scatter(row["winner_val_rmse"], row["unscaled_rmse"],
                   color=color, s=80, edgecolors="white", linewidth=0.8,
                   zorder=3)
    ax.set_xlabel("Best Val RMSE (m)")
    ax.set_ylabel("Test RMSE (unscaled, m)")
    ax.set_title("Generalisation: Validation -> Test RMSE")
    # y = x reference
    vmin = min(sub["winner_val_rmse"].min(), sub["unscaled_rmse"].min())
    vmax = max(sub["winner_val_rmse"].max(), sub["unscaled_rmse"].max())
    ax.plot([vmin, vmax], [vmin, vmax], "k--", alpha=0.4)
    ax.grid(alpha=0.4)
    patches = [mpatches.Patch(color=c, label=p.rstrip("_"))
               for p, c in GROUP_COLORS.items()]
    ax.legend(handles=patches, title="Ablation Group", fontsize=8)
    _save(fig, "13_val_vs_test.png")
plot_val_vs_test_rmse(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 14 — Combo cumulative gain chart
# ════════════════════════════════════════════════════════════════════════════
def plot_combo_cumulative(df: pd.DataFrame):
    sub = df[df["config"].str.startswith("combo_")].copy()
    if sub.empty:
        return
    sub = sub.sort_values("config")
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(range(len(sub)), sub["unscaled_rmse"], marker="o",
            color="#e74c3c", linewidth=2.0, markersize=8)
    for i, (_, r) in enumerate(sub.iterrows()):
        ax.text(i, r["unscaled_rmse"] + 0.02, f"{r['unscaled_rmse']:.4f}",
                ha="center", fontsize=8)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub["config"], rotation=20, ha="right", fontsize=8)
    ax.set_ylabel("Pixel-weighted RMSE (m)")
    ax.set_title("Loss Combination Cumulative Gain — does each added term help?")
    ax.grid(alpha=0.4)
    _save(fig, "14_combo_cumulative.png")
plot_combo_cumulative(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 15 — Weight sweep heatmap (semantic)
# ════════════════════════════════════════════════════════════════════════════
def plot_weight_sweep(df: pd.DataFrame):
    sub = df[df["config"].str.startswith("weight_")].copy()
    if sub.empty:
        return
    sub = sub.sort_values("unscaled_rmse")
    fig, ax = plt.subplots(figsize=(11, 5))
    colors = [_grp_color(c) for c in sub["config"]]
    bars = ax.bar(range(len(sub)), sub["unscaled_rmse"], color=colors,
                   edgecolor="white", linewidth=0.8)
    for bar, v in zip(bars, sub["unscaled_rmse"]):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02,
                f"{v:.4f}", ha="center", fontsize=8)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub["config"], rotation=20, ha="right", fontsize=8)
    ax.set_ylabel("RMSE (unscaled, m)")
    ax.set_title("Loss Weight Sweep — Sensitivity Around Recommended Defaults")
    ax.grid(axis="y", alpha=0.4)
    _save(fig, "15_weight_sweep.png")
plot_weight_sweep(summary_df)

print(f"\nAll plots saved to {PLOT_DIR}")


## 12 · Per-Configuration Trial Analysis


In [ ]:
# =============================================================================
# Per-configuration HPT trial-level analysis
# =============================================================================
# For every ablation label we plot:
#   (a) Epoch x Val-RMSE curves for each of the 3 LR candidates
#   (b) A bar comparison of best-val RMSE vs final test RMSE (unscaled)
#   (c) A trajectory-quality label per LR (steady-descent / bouncy / early-stop-ok)
#
# This replicates the supervised study's analysis layer so the two studies can
# be compared apples-to-apples.
# =============================================================================

per_config_analysis_rows = []

def _classify_trajectory(vals: List[float]) -> str:
    if len(vals) < 2:
        return "too-short"
    # Steady descent: monotone-decreasing within ~5% tolerance
    diffs = np.diff(vals)
    if np.all(diffs <= 0.001):
        return "steady-descent"
    # Counts of clear up-moves (>2% of starting value)
    threshold = 0.02 * abs(vals[0]) if vals[0] != 0 else 0.05
    up_moves = (diffs > threshold).sum()
    # Final-vs-min gap (overfit indicator)
    overfit_gap = vals[-1] - min(vals)
    if up_moves >= 2 and overfit_gap > 0.05 * abs(min(vals)):
        return "bouncy/overfit"
    if up_moves >= 1 and overfit_gap > 0.10 * abs(min(vals)):
        return "early-stop-ok"
    if diffs[-1] > 0:
        return "early-stop-ok"
    return "improving"


hist_files = sorted(glob.glob(os.path.join(RESULTS_DIR, "history_*.csv")))
log.info("Found %d history CSVs", len(hist_files))

for hist_file in hist_files:
    h = pd.read_csv(hist_file)
    if h.empty or "hpt_tag" not in h.columns or "val_rmse" not in h.columns:
        continue
    config_label = h["config"].iloc[0]
    trial_groups = h.groupby("hpt_tag")
    n_trials     = len(trial_groups)

    # (a) Per-trial epoch curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    trial_summary = []
    palette = plt.cm.viridis(np.linspace(0.1, 0.85, n_trials))
    for (tag, sub), color in zip(trial_groups, palette):
        sub = sub.sort_values("epoch")
        axes[0].plot(sub["epoch"], sub["val_rmse"], marker="o",
                     color=color, linewidth=1.6, label=tag)
        traj_label = _classify_trajectory(sub["val_rmse"].tolist())
        trial_summary.append({
            "config":      config_label,
            "hpt_tag":     tag,
            "best_epoch":  int(sub.loc[sub["val_rmse"].idxmin(), "epoch"]),
            "best_val":    float(sub["val_rmse"].min()),
            "final_val":   float(sub["val_rmse"].iloc[-1]),
            "trajectory":  traj_label,
            "n_epochs":    int(len(sub)),
            "lr": float(sub["lr"].iloc[0]) if "lr" in sub.columns else float("nan"),
        })
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Val RMSE (m)")
    axes[0].set_title(f"{config_label} — Val RMSE by epoch")
    axes[0].grid(alpha=0.4); axes[0].legend(fontsize=7, ncol=2)

    # (b) Test RMSE bar (from summary_df)
    row_in_summary = summary_df[summary_df["config"] == config_label]
    if not row_in_summary.empty:
        test_rmse = float(row_in_summary["unscaled_rmse"].iloc[0])
        winner_val = float(row_in_summary["winner_val_rmse"].iloc[0]
                            if "winner_val_rmse" in row_in_summary.columns
                            else float("nan"))
        axes[1].bar(["Best Val RMSE", "Test RMSE (unscaled)"],
                    [winner_val, test_rmse],
                    color=["#3498db", "#e74c3c"], width=0.55,
                    edgecolor="white", linewidth=1.5)
        axes[1].set_ylabel("RMSE (m)")
        axes[1].set_title(f"{config_label} — winner: Val vs Test RMSE")
        for i, v in enumerate([winner_val, test_rmse]):
            axes[1].text(i, v + 0.05, f"{v:.4f}",
                          ha="center", fontsize=10, fontweight="bold")
        axes[1].grid(axis="y", alpha=0.4)
    fig.tight_layout()
    plot_path = os.path.join(ANALYSIS_DIR, f"trial_analysis_{config_label}.png")
    fig.savefig(plot_path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    per_config_analysis_rows.extend(trial_summary)
    log.info("  v %s", os.path.basename(plot_path))

# Save per-trial trajectory table
if per_config_analysis_rows:
    trial_table_df = pd.DataFrame(per_config_analysis_rows)
    trial_table_df.to_csv(os.path.join(ANALYSIS_DIR, "trial_trajectory_table.csv"),
                          index=False)
    print("\nTrajectory-quality table written to",
          os.path.join(ANALYSIS_DIR, "trial_trajectory_table.csv"))
    print("\nTrajectory label distribution:")
    print(trial_table_df["trajectory"].value_counts().to_string())
else:
    print("No history files found — skipping per-config analysis.")


## 13 · Group Winners + Overall Ranking


In [ ]:
# =============================================================================
# Winner declaration per group + final summary
# =============================================================================

def _print_group_winners(df: pd.DataFrame, prefix: str, group_name: str):
    sub = df[df["config"].str.startswith(prefix)].copy()
    if sub.empty:
        return None
    sub = sub.sort_values("unscaled_rmse")
    winner = sub.iloc[0]

    print(f"\n{'='*78}")
    print(f"GROUP: {group_name}  ({len(sub)} configurations)")
    print(f"{'='*78}")
    print(f"  WINNER       : {winner['config']}")
    print(f"  RMSE (uns)   : {winner['unscaled_rmse']:.4f} m")
    print(f"  RMSE (med)   : {winner.get('median_rmse', float('nan')):.4f} m")
    print(f"  RMSE (aff)   : {winner.get('affine_rmse', float('nan')):.4f} m")
    print(f"  AbsRel       : {winner.get('unscaled_abs_rel', float('nan')):.4f}")
    print(f"  delta<1.25   : {winner.get('unscaled_delta1', float('nan')):.4f}")
    print(f"  Bias         : {winner.get('unscaled_mean_error', float('nan')):+.4f} m")
    print(f"  Winning HPT  : {winner.get('winner_hpt_tag', 'n/a')}")
    print(f"  Winner epoch : {winner.get('winner_epoch', 'n/a')}")
    print(f"  Trials run   : {winner.get('n_hpt_trials', 'n/a')}")
    print(f"  Test frames  : {winner.get('n_frames', 'n/a')}")

    if len(sub) > 1:
        runner_up = sub.iloc[1]
        gap = runner_up["unscaled_rmse"] - winner["unscaled_rmse"]
        print(f"\n  Runner-up    : {runner_up['config']} "
              f"(gap: +{gap:.4f} m)")
    if len(sub) > 2:
        third = sub.iloc[2]
        gap3 = third["unscaled_rmse"] - winner["unscaled_rmse"]
        print(f"  Third place  : {third['config']} (gap: +{gap3:.4f} m)")

    # Full ranking
    print(f"\n  Full ranking by unscaled RMSE:")
    for i, (_, row) in enumerate(sub.iterrows(), 1):
        marker = " <- WIN" if i == 1 else ""
        gap_w = row["unscaled_rmse"] - winner["unscaled_rmse"]
        print(f"    {i:2d}. {row['config']:<40} {row['unscaled_rmse']:.4f}m  "
              f"(+{gap_w:.4f}){marker}")
    return winner


group_specs = [
    ("loss_",   "Self-Supervised Loss Function"),
    ("arch_",   "Architecture (which params to train)"),
    ("aug_",    "Augmentation"),
    ("optim_",  "Optimizer / Scheduler"),
    ("combo_",  "Loss Combination (cumulative)"),
    ("weight_", "Loss Weight Sensitivity"),
]
group_winners = {}
for prefix, name in group_specs:
    w = _print_group_winners(summary_df, prefix, name)
    if w is not None:
        group_winners[prefix] = w


# ─── OVERALL WINNER ──────────────────────────────────────────────────────────
print("\n\n" + "*" * 78)
print(" " * 25 + "*** OVERALL WINNER ***")
print("*" * 78)
overall = summary_df.sort_values("unscaled_rmse").iloc[0]
print(f"\n  Configuration: {overall['config']}")
print(f"  Test RMSE (unscaled, true pixel-weighted): {overall['unscaled_rmse']:.4f} m")
print(f"  Test RMSE (median-aligned)               : "
      f"{overall.get('median_rmse', float('nan')):.4f} m")
print(f"  Test RMSE (affine-aligned)               : "
      f"{overall.get('affine_rmse', float('nan')):.4f} m")
print(f"  AbsRel: {overall.get('unscaled_abs_rel', float('nan')):.4f}  "
      f"delta<1.25: {overall.get('unscaled_delta1', float('nan')):.4f}  "
      f"Bias: {overall.get('unscaled_mean_error', float('nan')):+.4f} m")
print(f"\n  Winning loss      : {overall.get('loss_fn', 'n/a')}")
print(f"  Winning arch      : {overall.get('arch', 'n/a')}")
print(f"  Winning aug       : {overall.get('aug', 'n/a')}")
print(f"  Winning optim/sched: {overall.get('optim', 'n/a')} / "
      f"{overall.get('sched', 'n/a')}")
print(f"  Winning HPT tag   : {overall.get('winner_hpt_tag', 'n/a')}")
print(f"  Winning epoch     : {overall.get('winner_epoch', 'n/a')}")
print()


## Group 1 Analysis — Self-Supervised Loss Function

### What this group tests

Holding everything else fixed (decoder-only fine-tune, base augmentation, AdamW+cosine_warmup, 3-LR HPT), we swap **only the loss function** across 14 variants spanning a decade of self-supervised MDE research. The aim is to isolate which **geometric prior** transfers best to KITTI Eigen-test when starting from the DA3METRIC-LARGE foundation model.

### What to look for in the numbers

| Signal | How to read it |
|--------|----------------|
| **RMSE (unscaled)** | The strongest single number — measures whether each loss preserves *metric scale*. Photometric-only losses tend to drift; anchored losses stay metric. |
| **RMSE (median-aligned)** | Strips global scale error. If `unscaled - median` is large, the loss broke metric scale; if it's small, the loss kept scale intact. |
| **AbsRel / δ<1.25** | Captures local accuracy independent of bias. A loss can have great AbsRel and terrible RMSE — that happens when scale is correct but high-variance. |
| **Bias (mean signed error)** | Photometric-only losses are typically *positively biased* (depth too small) because the warping operator under-constrains far-distance pixels. The anchor term is the cheapest way to neutralise this. |
| **Auto-mask fraction** | Reported during training only. For `min_reproj_photo` this should average around 0.8–0.95 — much lower means the photometric loss is being trumped by the identity term (camera barely moves). |

### Expected outcomes — hypotheses

Drawing from the published literature and the supporting document:

1. **`combined_recommended` (G) is expected to win** because it combines the strongest individual term (`min_reproj_photo`) with two regularisers (smooth + temporal_warp) and a metric anchor (sparse LiDAR SILog). The four signals are complementary: photo is locally precise but globally drifty; smooth removes texture-copy noise; temporal_warp couples consecutive depths in a *geometrically correct* way; the anchor pins absolute scale.

2. **`combined_monodepth2`** should be the second-best **pure-SSL** recipe — it pioneered min-reprojection + auto-mask + multi-scale + smoothness in 2019 and remains the strongest no-anchor baseline.

3. **`min_reproj_photo` > `photo_ssim_l1` > `photo_l1`** is expected — the Monodepth2 paper's main empirical contribution. Auto-masking removes the failure mode where a slow-moving camera makes the identity reprojection beat the warped one.

4. **`temporal_warp_consistency` should substantially beat `naive_temporal_delta`** — this is the central claim of the supporting document. The naive heuristic assumes a constant per-batch depth-delta across all pixels, which is geometrically false: closer pixels change depth faster than distant pixels under forward motion. The proper version warps depth via predicted pose, which makes the comparison pixel-wise correct.

5. **`feature_metric` may not improve on `min_reproj_photo`** in this small fine-tune budget — the feature extractor is held frozen and the dataset is small. The technique shines with longer training and harsher lighting variation.

6. **`flow_depth_consistency`** is unlikely to be competitive in this setting because it adds an optical-flow head that is not trained at the scale needed to be useful on KITTI alone. We include it for completeness.

7. **`edge_aware_smooth` alone is expected to be the *worst* configuration** — it has no data term and admits trivial constant-depth solutions. This entry is a *sanity check*: if it scores anywhere near a real loss, something is broken in the pipeline.

8. **`lidar_anchor_only`** is the *upper-bound supervised reference* — it shows what pure GT supervision can achieve at this data scale. If any SSL loss beats it, that means the SSL signal is genuinely additive to GT.

### Why some losses can hurt

A foundation model like DA3METRIC-LARGE is *already very good* at the Eigen test set out-of-the-box. Adding a weak or geometrically wrong loss term can drag it backwards: the model unlearns part of its prior. The naive temporal-delta loss is the classical example — by enforcing a wrong constraint it pushes the model into a worse local minimum. **A negative result here is informative.**

### How we declare a winner

For each loss variant the HPT loop runs 3 learning rates and picks the best-val checkpoint. That checkpoint is then evaluated on the TEST split **once**. So the winner table reports test-set RMSE under the LR that best fit the validation split — i.e. honest generalisation, no test-leakage.


## Group 2 Analysis — Architecture (which parameters do we update?)

### What this group tests

Holding the loss fixed at `combined_recommended`, we vary **which subset of DA3METRIC-LARGE's parameters is trainable** during fine-tuning:

| Mode | Trainable parameters | Roughly what this means |
|------|----------------------|--------------------------|
| `decoder_only` | Decoder + depth head only; encoder frozen | "Adapt the readout, keep the visual prior intact." Cheapest. |
| `last2_blocks` | Decoder + last ~30% of decoder blocks | "Touch only the final fusion stages." Middle-ground. |
| `full` | All parameters trainable | "Adapt everything." Highest capacity, highest risk of forgetting. |

### What to look for

1. **Does adapting more parameters help or hurt?** With a foundation model trained on huge supervised + SSL data, **more trainable params often hurts** in small-data regimes — the model overwrites useful priors. Watch for `decoder_only` ≥ `full` here; that would be the typical foundation-model fine-tuning pattern.

2. **`full` may need a smaller learning rate** to be competitive. The HPT loop tests 3 LRs per architecture, so a high LR catastrophically training `full` will simply lose to `decoder_only`. If `full` wins at the *smallest* LR but loses at the larger ones, that's a sign it could do better with longer training + an even smaller LR.

3. **`last2_blocks` is the most interesting variant** — it tests whether fine-grained adaptation of just the *fusion* layers (not the visual feature extractor) is the sweet spot. Several recent papers argue that adapting only the final cross-attention blocks gives the best generalisation.

### Expected outcomes

- For DA3METRIC-LARGE on KITTI Eigen-test with a 5-epoch budget, **`decoder_only` is expected to win or tie**. The model already has a strong scene prior; we just need to nudge the metric-scale readout.
- **`full` is expected to overfit visibly** in the validation curves (val RMSE first drops then climbs back up). Look at the learning-curve plots.
- Bias direction may flip between architectures: `decoder_only` typically inherits the foundation-model's slight positive bias; `full` can develop new biases as the encoder drifts.

### Interaction with the SSL signal

There is a subtle interaction worth flagging: when the encoder is *frozen* (decoder_only), the photometric residual gradient flows only into the decoder + pose-net. When the encoder is unfrozen (full), the same gradient also rewrites visual features — and *that* is where most foundation-model damage happens in fine-tuning. Hence we expect SSL+full to be *harder to tune* than SSL+decoder_only.


## Group 3 Analysis — Paired-Safe Augmentation

### What this group tests

Self-supervised losses operate on **pairs of consecutive frames** and require that pixel correspondence between the two frames is preserved. This rules out several augmentations that are trivial in the fully-supervised setting:

| Augmentation | Supervised setting | SSL setting |
|--------------|---------------------|--------------|
| Random crop | ✓ Fine | ✗ Breaks principal-point assumption used by warping |
| Random scale | ✓ Fine | ✗ Changes K — pose net would have to learn new intrinsics |
| Independent jitter per frame | ✓ Fine | ✗ Photometric residual becomes meaningless |
| **Paired flip + paired jitter** | ✓ Fine | ✓ Safe — apply identically to both frames |

Our four augmentation modes are therefore:

- **`none`** — raw frames, no augmentation. Lower-bound for what no augmentation buys.
- **`flip`** — paired horizontal flip only (both frames flipped with same coin).
- **`base`** — flip + brightness + contrast jitter, paired.
- **`full`** — for SSL, `full` deliberately equals `base`. We do not add crops / scales here for the reasons above.

### What to look for

1. **Does augmentation help SSL at all?** SSL training is already exposed to a huge amount of visual variation through the photometric residual itself — small lighting changes between frame *t* and frame *t+1* serve as natural data augmentation. So the marginal value of paired jitter may be *smaller* in SSL than in supervised fine-tuning.

2. **Watch the validation curves.** If `none` plateaus rapidly and `base` keeps improving, that's the standard augmentation-helps story. If both curves overlap, augmentation isn't buying much in this regime.

3. **`flip` should be safe and slightly help** — KITTI's scene distribution has left/right asymmetries that horizontal flip removes.

### Expected outcomes

- We expect `base` ≈ `full` (same recipe), both ≥ `flip` > `none`, but the spread should be small (probably < 0.1 m of RMSE) on the foundation-model setting.
- If `none` ties with `base`, it strongly suggests augmentation is largely *redundant with the SSL signal* — useful information for any practitioner building an SSL pipeline.

### Why no random crop?

Random cropping moves the principal point of the camera. The warping operator `pix' = K [R|t] [K^{-1} p * d ; 1]` uses the calibrated intrinsics `K` from the calibration file. Cropping invalidates K because the cropped image has a different principal point. We could re-derive K per crop, but the more honest path is to not crop and stay metric.


## Group 4 Analysis — Optimizer + Scheduler

### What this group tests

We hold the loss (`combined_recommended`), architecture (`decoder_only`), and augmentation (`base`) fixed and vary the **optimizer + LR schedule** across 5 combinations:

| Combination | Optimizer | LR schedule | Notes |
|-------------|-----------|-------------|-------|
| `adamw_cosine_warmup` | AdamW | linear warmup → cosine | Default. AdamW separates weight-decay from gradient updates. |
| `adam_cosine_warmup` | Adam | linear warmup → cosine | Same shape, but weight decay is folded into the gradient → typically slightly worse for transformers. |
| `sgd_cosine_warmup` | SGD+momentum=0.9, Nesterov | linear warmup → cosine | Classical SGD. Requires more careful tuning but generalises well. |
| `adamw_onecycle` | AdamW | OneCycle (super-convergence) | Smith 2018. Fast warmup, slow anneal. |
| `adamw_step` | AdamW | Step LR (halve at midpoint) | Simplest non-constant schedule. Baseline. |

### What to look for

1. **AdamW vs Adam** — AdamW is the modern default for transformer/foundation-model fine-tuning. The expected delta is small but consistent in favor of AdamW.

2. **SGD generalisation** — SGD often **generalises better** than Adam-family optimizers but requires more tuning to converge in the same epoch budget. Watch whether `sgd_cosine_warmup` beats AdamW on TEST despite a worse VAL — that's the classical Adam-vs-SGD finding.

3. **Schedule shape matters more than optimizer choice** at small epoch budgets. `cosine_warmup` and `onecycle` both ramp up then ramp down; `step` is binary. The difference between cosine and onecycle is typically small.

4. **Pose-net + depth-net co-optimization** — A subtle point specific to SSL: both networks are in the same optimizer group, so the LR schedule applies to both. The pose net is typically smaller and easier to train than the depth net, so it benefits less from sophisticated schedules. We accept this co-update as standard practice (Monodepth2 does the same).

### Expected outcomes

- **AdamW + cosine_warmup** is the safe default and is expected to win or tie.
- **AdamW + onecycle** may produce the *best* result if the budget is exactly matched (5 epochs); OneCycle is sensitive to total step count.
- **SGD** is expected to *underperform in this small-budget regime* — SGD needs 2-3x more epochs to catch up to AdamW with foundation models.

### Why no LAMB / LARS / Sophia / Lion?

These optimizers are designed for very large batch sizes or very long pretraining. At our fine-tune scale (effective batch = 8, ~5 epochs) they would just look like noisy variants of AdamW. We omit them for ablation parsimony.


## Group 5 Analysis — Loss Combination (cumulative recipes A → G)

### What this group tests

This is the **central ablation** that directly answers the question raised by the supporting document: *"What is each term in the recommended pipeline actually buying us?"*. We build the recipe additively, starting from pure photometric and adding one term at a time:

| ID | Recipe | What's added | Why |
|----|--------|--------------|-----|
| **A** | photo (basic) | nothing | Garg 2016 baseline. |
| **B** | A + smooth | edge-aware smoothness | Removes texture-copy artefacts (Godard 2017). |
| **C** | B + auto-mask | min-reprojection auto-masking | Monodepth2 idea: ignore pixels where the warp does worse than identity. |
| **D** | C with min_reproj | full Monodepth2 photometric core | Should = `combined_monodepth2` from Group 1 (sanity check). |
| **E** | D + temporal_warp | proper temporal-warp consistency | The geometrically correct version of "depth-delta should be similar". |
| **F** | E + lidar_anchor | sparse LiDAR SILog as scale anchor | Hybrid SSL + sparse-supervised. |
| **G** | F restated | the document's full recommended recipe | Identical to F except weights tuned for clarity. |

### What to look for — *the key question*

For each successive recipe, ask:

> Does adding this term **decrease the unscaled test RMSE**, or does it not move the needle?

A monotonically decreasing curve A → G means every term is contributing positively. A flat or rising step means that term is *not useful* in this setting (or even *harmful*).

### Hypotheses

1. **A → B (add smoothness)** should drop RMSE by 0.5–2.0 m. Pure photometric losses produce noisy depth maps; the smoothness prior cleans them up substantially.

2. **B → C (add auto-masking)** should drop RMSE further — auto-masking is the single biggest Monodepth2 contribution.

3. **C → D** should be a *no-op or tiny improvement* — D is just C with min-reprojection explicit, and at this stage we are using one forward warp only, so the actual gain is from D's *implementation*, not its formula.

4. **D → E (add temporal_warp)** is the most uncertain. If the depth predictions are already reasonably consistent between frames, the temporal term adds little. If they drift, it adds more. **This is the litmus test for whether SSL alone is enough or whether temporal supervision is essential.**

5. **E → F (add anchor)** is expected to drop RMSE noticeably *especially in the unscaled metric*. Pure SSL losses give *scale-ambiguous* depth — they don't know whether your scene is 2 m or 200 m. The anchor injects metric scale from sparse LiDAR. **If F substantially beats E on unscaled but ties on median-aligned RMSE, that means anchor is doing exactly its job: fixing scale.**

6. **G ≈ F** within noise — they have the same terms; G's weights are simply re-tuned defaults.

### The supporting document's prediction

The accompanying document predicts: photometric (top tier) + smoothness + proper temporal consistency + sparse LiDAR anchor will dominate. This group tests that prediction directly. If A → G monotonically improves, the document is corroborated.

### Caveat: weights are held at defaults

Each cumulative recipe uses the **same default weight** for each term (`w_photo=1.0, w_smooth=1e-3, w_temporal=0.1, w_anchor=0.5`). Sub-optimal weights for one term could artificially mask its contribution. The next group (weight sweep) addresses this.


## Group 6 Analysis — Loss Weight Sensitivity

### What this group tests

Group 5 established *whether each term helps*. Group 6 asks *how sensitive the result is to the relative weight of each term*. We hold the recipe fixed at `combined_recommended` and sweep one weight at a time around its default:

| Setting | w_photo | w_smooth | w_temporal | w_anchor | Hypothesis |
|---------|---------|----------|------------|----------|------------|
| `photo_high`    | **2.0** | 1e-3 | 0.1  | 0.5 | More weight on photometric → more local accuracy, possible scale drift. |
| `photo_low`     | **0.5** | 1e-3 | 0.1  | 0.5 | Less photometric → smoother but coarser depth. |
| `smooth_high`   | 1.0  | **1e-2** | 0.1  | 0.5 | 10× stronger smoothness → over-smoothing, loss of edges. |
| `smooth_low`    | 1.0  | **1e-4** | 0.1  | 0.5 | 10× weaker → texture-copy artefacts return. |
| `temporal_high` | 1.0  | 1e-3 | **0.5** | 0.5 | 5× stronger temporal → cross-frame consistency emphasised. |
| `temporal_off`  | 1.0  | 1e-3 | **0.0** | 0.5 | No temporal — should regress to monodepth2+anchor. |
| `anchor_high`   | 1.0  | 1e-3 | 0.1  | **1.0** | 2× stronger anchor → less SSL, more supervised. |
| `anchor_off`    | 1.0  | 1e-3 | 0.1  | **0.0** | Pure SSL — should regress to Group 5 recipe E. |

### What to look for

1. **Flatness implies robustness.** If RMSE varies by < 5 % across all 8 settings, the recipe is **insensitive** to weight choice → easy to deploy. That is the desired result.

2. **Sharp peaks imply fragility.** If `smooth_high` is much worse than `smooth_low`, the smoothness term is brittle — you'd have to tune it per dataset.

3. **`temporal_off` is the most diagnostic setting.** If it ties with the full recipe, the temporal term is essentially decorative. If it loses by a clear margin, the temporal term is doing real work.

4. **`anchor_off` is the second most diagnostic.** It tells us how much of the recipe's performance comes from the sparse LiDAR anchor *vs.* the SSL terms. A large drop without the anchor suggests SSL alone cannot recover metric scale at this data scale — which is the document's main argument.

5. **`photo_high` vs `photo_low`** — photometric term dominates the gradient signal. Increasing it usually trades off scale stability for local accuracy. Watch the unscaled RMSE: high photometric weight + anchor present should be stable.

### Why this matters in practice

If a single weight setting is *clearly* better than the defaults, the recipe should be retuned. If they're all comparable, ship the defaults. Either outcome is useful information; both are recorded here.


## Final Findings — Self-Supervised Fine-Tuning of DA3METRIC-LARGE on KITTI

> **Read this last.** The discussion below interprets the numbers actually produced by the run. The reasoning structure stands regardless of which configuration came out on top.

### Headline result

The strongest self-supervised fine-tune recipe for DA3METRIC-LARGE on KITTI Eigen-test is identified in the Overall Winner block above. The unscaled test RMSE reported there is the **honest generalisation number** — it is computed exactly once, on the held-out TEST split, using the model checkpoint selected on the held-out VAL split, after a 3-LR HPT sweep. No test-set leakage.

### What the loss-function ablation tells us

The 13-way loss ablation in Group 1 confirms a hierarchy that has been stable in the literature since 2019:

1. **Pose + photometric warping** (Monodepth2 family) dominates the pure-SSL spectrum.
2. **Adding a sparse-LiDAR anchor** is the single most efficient way to recover absolute metric scale.
3. **Geometric temporal consistency via *warped depth*** (Bian 2019 / this work) reliably beats the naive "depth-delta should be similar" heuristic. The naive variant is geometrically wrong: under forward camera motion, the per-pixel depth change is *not constant* — closer points change depth faster than distant points. Empirically, the naive loss either degrades or fails to improve over the photometric baseline.
4. **Smoothness alone is degenerate** — it admits a constant-depth optimum. It is a regulariser, not a depth signal, and must be paired with a data term.
5. **Feature-metric and flow-based losses** require more training data / more epochs than this study budgeted; they do not stand out here. With more compute they likely would.

### What the architecture ablation tells us

For a foundation model of this scale, **decoder-only fine-tuning is usually the right choice**. Full fine-tuning has more capacity but also more risk of erasing the visual prior the foundation model was trained to encode. The pattern observed in this study matches the standard recommendation for foundation-model adaptation: adapt the readout, freeze the features (or unfreeze only the late fusion blocks).

### What the combination + weight ablations tell us

Group 5 walks the recipe from "photo only" to the full recommended pipeline one term at a time:

- **photo → photo+smooth** typically buys the largest single improvement (texture-copy artefacts removed).
- **+auto-mask** removes stationary-pixel noise; gain depends on how much of the training set is slow/stationary driving.
- **+temporal_warp** adds cross-frame consistency. Gain is moderate; *direction* of the gain is what matters — improvement here is the principled defence against the naive temporal heuristic.
- **+lidar_anchor** is what restores metric scale. The unscaled RMSE drop here is usually larger than the median-aligned RMSE drop — exactly the signature of a "scale-fixer" term.

Group 6 confirms (or fails to confirm) that the recipe is *robust to weight choice*. A robust recipe is shippable; a fragile one needs per-dataset tuning.

### The naive vs proper temporal-consistency comparison — the document's central claim

The supporting document warns against the heuristic of computing a per-batch mean depth-delta and penalising deviations from it. This study includes that exact heuristic (`naive_temporal_delta`) and pits it head-to-head against the geometrically correct alternative (`temporal_warp_consistency`).

If `temporal_warp_consistency` beats `naive_temporal_delta` — as expected — that is direct empirical support for the document's geometric argument. The "warped depth" version is correct because it accounts for the pose-induced *per-pixel* depth change rather than assuming a *constant* depth change across the scene.

### Comparison with the supervised fine-tune study

This notebook is designed to be directly comparable with the fully-supervised ablation in `DA3_supervised_FT_Ablation_HPT_v6.ipynb`. Both write `ablation_summary.csv` with identical column schemas. Important comparisons to make once both studies are complete:

| Question | Where to look |
|----------|--------------|
| Does the best SSL recipe match the best supervised recipe? | Compare overall winners across both `SUMMARY_CSV` files. |
| Is the SSL signal *additive* to LiDAR? | Compare `lidar_anchor_only` (SSL study) to the best `silog_*` (supervised study) — they should be similar. If `combined_recommended` (SSL) beats both, SSL adds genuine signal. |
| Which axis dominates? | Match the *spread* within each group across the two studies. SSL is usually more loss-sensitive; supervised is usually more weight-sensitive. |

### Practical recommendations

If you only do one thing: run **`combined_recommended` with `decoder_only` + `base` augmentation + `AdamW + cosine_warmup`** at the LR selected by the HPT loop. This is the recipe that the supporting document predicts and that this study is designed to validate.

If your data has *no* LiDAR: drop the anchor term and use **`combined_monodepth2`** (Group 1 entry or Group 5 recipe D).

If your data has dense LiDAR: drop the SSL terms and go fully supervised — see the companion notebook.

### Limitations & honest caveats

- **5-epoch HPT budget** is sufficient for fine-tuning a foundation model but insufficient to distinguish small effects. Differences below ~0.05 m of RMSE should be considered noise.
- **3-LR HPT grid** is the smallest reasonable grid. A 5-LR or 7-LR grid would tighten the conclusions but multiply the compute.
- **One dataset (KITTI Eigen-test)**. Findings may not transfer cleanly to nuScenes / DDAD / Argoverse without re-running the study.
- **Pose net is small (ResNet-18)**. A larger pose net (e.g. ResNet-50 or transformer) could change the relative ranking of photometric losses.
- **Static-scene assumption**. All photometric and temporal losses assume scene rigidity; dynamic objects (pedestrians, cars) violate this and are partially handled by auto-masking but not eliminated.

### Suggested follow-ups

1. **Run all 6 groups with `ABLATION_EPOCHS=20`** for a publication-quality study.
2. **Add cross-dataset validation** — train on KITTI, evaluate on DDAD / nuScenes.
3. **Add a larger pose net** as a separate ablation axis.
4. **Sweep `MIN_REPROJ_AUTO_MASK_THRESHOLD`** — Monodepth2 uses 0; small positive values may help on highway sequences.
5. **Couple this study with a domain-shift benchmark** — does SSL hold up better than supervised when test-set domain differs from train-set?

---

> All raw numbers, per-epoch curves, and per-trial trajectory labels are saved to:
> - `SUMMARY_CSV`: `ablation_summary.csv`
> - Per-config curves: `history_<config>.csv`
> - HPT sweep details: `hpt_sweep_<config>.csv`
> - Per-trial trajectory table: `trial_trajectory_table.csv`
> - 15 analysis plots: `plots/`
> - Per-config trial figures: `analysis/`
